# **C The Signs**

---
Youssef Drissi

---

## Overview

### Objective

Cancer Prediction using Longitudinal Patient Data.

### Approach


Since the patient dataset contains longitudinal data, the proposed approach is to use an LSTM model because it can efficiently exploit the temporal component of the longitunal data (e.g. sequence of snomed IDs recorded for a patient), and also incorporate other features like gender and ethnicity, to provide more context and improve the prediction performance.

This involves the following steps:

1. **Load and Inspect Data:** Load the csv file into a Pandas DataFrame and display its head, info, and basic statistics to understand the data structure and types.

2. **Define Target Variable and Initial Preprocessing:** Create the target variable ('has_cancer_diagnosis') based on the 'date_of_diagnosis' column. Convert date columns to datetime objects and calculate patient age at each observation. Drop the 'date_of_diagnosis' column as it's the target.

3. **Feature Engineering for Sequences:** Process categorical features ('snomed_id', 'gender', 'ethnicity') using appropriate encoding (e.g., one-hot encoding or label encoding/embedding preparatory steps) and prepare the data into sequences per patient for LSTM input, ensuring consistent sequence lengths through padding.

4. **Split Data into Training, Validation, and Testing Sets:** Split the preprocessed patient sequences and their corresponding target labels into training, validation (20% of the data), and testing (20% of the data) datasets, ensuring patient data is not mixed across sets.

5. **Build Keras LSTM Model:** Design and compile a Keras LSTM model suitable for binary classification, considering the shape of the input sequences and the type of features used.

6. **Train LSTM Model:** Train the built LSTM model using the prepared training data and use the validation set for monitoring performance during training.

7. **Evaluate Model Performance:** Evaluate the trained LSTM model's performance on the test set using appropriate metrics such as sensitivity, specificity, accuracy, precision, recall, and F1-score.


## Code

### Configuration

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled")
    except RuntimeError as e:
        print("Error:", e)

In [ ]:
# initial imports
import os
import random
import json
import numpy as np
import pandas as pd
from IPython.display import display

import keras
from keras import layers
from keras import initializers

In [ ]:
def check_gpus(config):
    print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Explicit device placement (for keras models)
# with tf.device('/GPU:0'):
#     model.fit(X_train_scaled, y_train_encoded, epochs=1)

In [ ]:
# enable_better_reproducibility(config)
def enable_better_reproducibility(config):
    # Set a random seeds for better Reproducibility
    SEED = config['SEED']
    os.environ['TF_DETERMINISTIC_OPS'] = '1'  # v147_k7: GPU determinism
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'  # v147_k7: cuDNN determinism
    os.environ['PYTHONHASHSEED'] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
    keras.utils.set_random_seed(SEED)

    # Enable deterministic behavior for TensorFlow operations
    tf.config.experimental.enable_op_determinism()

In [ ]:
def get_config():
    config = dict()
    
    return config

### Data Processing

#### Helper Functions

In [ ]:
# cancer vs no cancer in the data
def print_df_info(df_name, df, patient_id_column_name='patient_id'):
    cancer = (df.has_cancer_diagnosis == 1)
    no_cancer = (df.has_cancer_diagnosis == 0)
    
    cancer_patients_count = len(df[cancer][patient_id_column_name].unique().tolist())
    no_cancer_patients_count = len(df[no_cancer][patient_id_column_name].unique().tolist())
    cancer_records_count = len(df[cancer])
    no_cancer_records_count = len(df[no_cancer])
    
    print(f'\n\n{df_name}')
    
    print(f'\n\tPatients:')
    print(f'\t\tTotal: \t\t{len(df[patient_id_column_name].unique().tolist()):,d}')
    print(f'\t\tCancer: \t{cancer_patients_count:,d}')
    print(f'\t\tNo Cancer: \t{no_cancer_patients_count:,d}')

    print(f'\n\tRecords:')
    print(f'\t\tTotal: \t\t{len(df):,d}')
    print(f'\t\tCancer: \t{cancer_records_count:,d}')
    print(f'\t\tNo Cancer: \t{no_cancer_records_count:,d}')

#### Load Data

In [ ]:
# Load data into a Pandas DataFrame from one of two sources, controlled by
# config['DATA_SOURCE']:
#   'csv'      -> glob-load from config['DATA_FILE']           (v127 behaviour)
#   'bigquery' -> run config['SQL_FILE'] against BigQuery,
#                 with a local parquet cache keyed by SQL hash (v129 behaviour)
#
# v132: applies `_normalize_loaded_df` to BOTH paths so the returned DataFrame
# is bit-identical regardless of source. This eliminates the "same data, slightly
# different model" problem caused by row-order and dtype differences between
# pd.read_csv and bigquery.Client().query().to_dataframe().
import pandas as pd
import glob


def _normalize_loaded_df(df):
    """Make the loaded DataFrame source-independent.

    Two sources of variance between pd.read_csv and bq.to_dataframe:
      (1) Row order (CSV preserves file order; BQ Arrow stream doesn't guarantee
          any particular order). Even when downstream operations look stable,
          drop_duplicates(keep='first') and groupby().iloc[0] can produce
          slightly different per-patient feature values when duplicates exist
          and the surviving row's ethnicity/age/etc. differ marginally.
      (2) Dtype representation: CSV returns int columns with nulls as float64
          ("187862007.0" via astype(str)); BQ returns Int64 ("187862007" via
          astype(str)). When the vocab is built via .astype(str).value_counts(),
          the two sources can produce different keys.

    This function fixes both: stable sort + Int64-cast for known integer cols.
    """
    # ── 1. Stable row order
    candidate_sort_cols = [
        'patient_guid', 'event_date', 'snomed_c_t_concept_id',
        'med_code_id', 'event_type',
    ]
    sort_cols = [c for c in candidate_sort_cols if c in df.columns]
    if sort_cols:
        df = df.sort_values(by=sort_cols, kind='stable',
                            na_position='last').reset_index(drop=True)

    # ── 2. Coerce integer-typed columns to pandas Int64 (nullable)
    # so astype(str) gives "123" not "123.0" (the float64-with-NaN case).
    INT_COLS = [
        'cancer_class', 'patient_age', 'event_age',
        'snomed_c_t_concept_id', 'med_code_id', 'duration',
    ]
    for c in INT_COLS:
        if c not in df.columns:
            continue
        if df[c].dtype == 'Int64':
            continue  # already correct
        try:
            # Convert via float-aware path; .astype('Int64') handles NaN→NA.
            df[c] = df[c].astype('Int64')
        except (TypeError, ValueError):
            pass  # column not numeric; leave alone

    # ── 3. Strip whitespace from patient_guid (defensive; pre_process also does this)
    if 'patient_guid' in df.columns and df['patient_guid'].dtype == 'object':
        df['patient_guid'] = df['patient_guid'].str.strip()

    # ── 4. Parse event_date into datetime if CSV returned it as a string
    if 'event_date' in df.columns and df['event_date'].dtype == 'object':
        try:
            df['event_date'] = pd.to_datetime(df['event_date'])
        except Exception:
            pass

    return df


def _load_data_csv(config):
    pattern = config['DATA_FILE']
    files = sorted(glob.glob(pattern))
    print(f"CSV : pattern={pattern}  matched_files={len(files)}")
    if not files:
        raise FileNotFoundError(f"No CSV files matched pattern: {pattern}")
    df = pd.concat(
        [pd.read_csv(f, skip_blank_lines=True, on_bad_lines='skip', low_memory=False)
         for f in files],
        ignore_index=True,
    )
    print(f"  rows={len(df):,}  cols={len(df.columns)}")
    return df


def _load_data_bq(config):
    import hashlib
    from pathlib import Path
    from google.cloud import bigquery

    sql_path = Path(config['SQL_FILE'])
    if not sql_path.exists():
        raise FileNotFoundError(f"SQL file not found: {sql_path.resolve()}")

    sql = sql_path.read_text()
    use_cache     = bool(config.get('use_local_cache', True))
    force_refresh = bool(config.get('force_bq_refresh', False))
    cache_dir     = Path(config.get('cache_dir', 'cache'))
    project       = config.get('BQ_PROJECT', 'cthesigns-platform-475414-b7')
    location      = config.get('BQ_LOCATION', None)

    sql_hash   = hashlib.sha256(sql.encode('utf-8')).hexdigest()[:12]
    cache_file = cache_dir / f"{sql_path.stem}_{sql_hash}.parquet"

    if use_cache and cache_file.exists() and not force_refresh:
        size_mb = cache_file.stat().st_size / 1e6
        print(f"Cache HIT : {cache_file}  ({size_mb:.1f} MB)")
        df = pd.read_parquet(cache_file)
        print(f"  rows={len(df):,}  cols={len(df.columns)}")
        return df

    if not use_cache:
        print(f"Cache OFF (use_local_cache=False); querying BigQuery.")
    elif not cache_file.exists():
        print(f"Cache MISS: {cache_file} (not present); querying BigQuery.")
    else:
        print(f"Cache PRESENT but force_bq_refresh=True; querying BigQuery anyway.")

    print(f"BigQuery: requested project={project}  location={location}  sql={sql_path}")
    print(f"  ({len(sql.splitlines())} lines, {len(sql):,} chars,  sql_hash={sql_hash})")

    client = bigquery.Client(project=project, location=location)

    principal = "(unknown)"
    try:
        import google.auth
        from google.auth.transport.requests import Request as _AuthRequest
        creds, detected_project = google.auth.default()
        cred_type = type(creds).__name__
        try:
            creds.refresh(_AuthRequest())
        except Exception:
            pass
        principal = (getattr(creds, "service_account_email", None)
                     or getattr(creds, "_service_account_email", None)
                     or getattr(creds, "signer_email", None)
                     or principal)
        if not principal or principal == "default":
            principal = "(could not resolve principal; try `gcloud auth list`)"
        print(f"  auth: principal={principal}")
        print(f"        cred_type={cred_type}  detected_project={detected_project}")
    except Exception as e:
        print(f"  auth: diagnostic failed: {e}")
    print(f"  client: effective_project={client.project}  effective_location={client.location}")

    job_config = bigquery.QueryJobConfig(use_query_cache=True)
    job = client.query(sql, job_config=job_config)
    print(f"  job_id={job.job_id}  state={job.state}  job_location={job.location}")

    try:
        df = job.to_dataframe(progress_bar_type='tqdm')
    except Exception as exc:
        msg = str(exc)
        if 'Access Denied' in msg or '403' in msg or 'accessDenied' in msg:
            print()
            print('=' * 78)
            print('BigQuery returned 403 Access Denied.')
            print(f'  Authenticated as: {principal}')
            print(f'  Job project    : {client.project}')
            print(f'  Job location   : {job.location}')
            print()
            print('Two fixes:')
            print('  (A) gcloud auth application-default login --no-launch-browser')
            print(f"  (B) gcloud projects add-iam-policy-binding <DATA_PROJECT> \\")
            print(f"        --member='serviceAccount:{principal}' \\")
            print(f"        --role='roles/bigquery.dataViewer'")
            print()
            print('  Workaround: config["DATA_SOURCE"] = "csv" in the config cell.')
            print('=' * 78)
        raise

    bp = job.total_bytes_processed or 0
    bb = job.total_bytes_billed or 0
    bq_cached = bool(job.cache_hit)
    print(f"  rows={len(df):,}  cols={len(df.columns)}  "
          f"bytes_processed={bp/1e9:.3f} GB  bytes_billed={bb/1e9:.3f} GB  bq_cached={bq_cached}")

    if use_cache:
        cache_dir.mkdir(parents=True, exist_ok=True)
        df.to_parquet(cache_file, index=False)
        size_mb = cache_file.stat().st_size / 1e6
        print(f"Cache WROTE: {cache_file}  ({size_mb:.1f} MB)")

    return df


def load_data(config):
    """Load data and normalise it so the result is independent of source."""
    source = str(config.get('DATA_SOURCE', 'bigquery')).lower()
    print(f"Data source: {source}")
    if source == 'csv':
        df = _load_data_csv(config)
    elif source in ('bigquery', 'bq'):
        df = _load_data_bq(config)
    else:
        raise ValueError(
            f"Unknown DATA_SOURCE: {source!r}. Set config['DATA_SOURCE'] to "
            f"'csv' or 'bigquery'."
        )

    df = _normalize_loaded_df(df)
    # Brief dtype hash so you can confirm visually that both paths produce
    # the same shape/dtypes.
    dtypes_summary = ','.join(f"{c}:{str(t)}" for c, t in
                              sorted(df.dtypes.items())[:6])
    print(f"  normalised: rows={len(df):,}  cols={len(df.columns)}  "
          f"first_dtypes=[{dtypes_summary}...]")
    return df


#### compute stats

In [ ]:
# compute_stats
import matplotlib.pyplot as plt
import seaborn as sns

def _plot_distribution(data_cancer, data_non_cancer, xlabel, title):
    """Plot overlaid histograms using seaborn."""
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(data_non_cancer.dropna(), bins=40, color='#1f77b4', alpha=0.8, label='Non-Cancer', stat='count', edgecolor='none', ax=ax)
    sns.histplot(data_cancer.dropna(), bins=40, color='#ff7f0e', alpha=0.8, label='Cancer', stat='count', edgecolor='none', ax=ax)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Patients')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

def compute_stats(df_merged, config):
    df_merged_grouped = df_merged.groupby(['patient_id', 'has_cancer_diagnosis', 'EVENT_TYPE']).agg(
        snomed_seq_len=('snomed_id', 'count'),
        medications_seq_len=('MED_CODE_ID', 'count')
    ).reset_index()
    
    # Observation distributions
    obs_df_merged_grouped = df_merged_grouped[df_merged_grouped.EVENT_TYPE=='observation']
    cancer_obs = obs_df_merged_grouped[obs_df_merged_grouped.has_cancer_diagnosis == 1]
    non_cancer_obs = obs_df_merged_grouped[obs_df_merged_grouped.has_cancer_diagnosis == 0]
    
    _plot_distribution(
        cancer_obs.snomed_seq_len, non_cancer_obs.snomed_seq_len,
        'snomed_seq_len', 'Cancer vs Non-Cancer snomed_seq_len distributions')
    
    # Medication distributions (from medication rows, not observation rows)
    med_df_merged_grouped = df_merged_grouped[df_merged_grouped.EVENT_TYPE=='medication']
    cancer_med = med_df_merged_grouped[med_df_merged_grouped.has_cancer_diagnosis == 1]
    non_cancer_med = med_df_merged_grouped[med_df_merged_grouped.has_cancer_diagnosis == 0]
    
    _plot_distribution(
        cancer_med.medications_seq_len, non_cancer_med.medications_seq_len,
        'medications_seq_len', 'Cancer vs Non-Cancer medications_seq_len distributions')

    # Age distributions (one age per patient, deduplicated)
    if 'PATIENT_AGE' in df_merged.columns:
        patient_ages = df_merged.groupby(['patient_id', 'has_cancer_diagnosis'])['PATIENT_AGE'].first().reset_index()
        cancer_ages = patient_ages[patient_ages.has_cancer_diagnosis == 1]['PATIENT_AGE']
        non_cancer_ages = patient_ages[patient_ages.has_cancer_diagnosis == 0]['PATIENT_AGE']
        
        _plot_distribution(
            cancer_ages, non_cancer_ages,
            'Patient Age', 'Cancer vs Non-Cancer Age distributions')
        
        print(f"Age stats - Cancer:     mean={cancer_ages.mean():.1f}, std={cancer_ages.std():.1f}, min={cancer_ages.min()}, max={cancer_ages.max()}")
        print(f"Age stats - Non-Cancer: mean={non_cancer_ages.mean():.1f}, std={non_cancer_ages.std():.1f}, min={non_cancer_ages.min()}, max={non_cancer_ages.max()}")

    df_merged_grouped_agg = df_merged_grouped.groupby(['EVENT_TYPE', 'has_cancer_diagnosis']).agg(
        snomed_seq_len_mean=('snomed_seq_len', 'mean'),
        snomed_seq_len_min=('snomed_seq_len', 'min'),
        snomed_seq_len_max=('snomed_seq_len', 'max'),
        snomed_seq_len_std=('snomed_seq_len', 'std'),
        medications_seq_len_mean=('medications_seq_len', 'mean'),
        medications_seq_len_min=('medications_seq_len', 'min'),
        medications_seq_len_max=('medications_seq_len', 'max'),
        medications_seq_len_std=('medications_seq_len', 'std'),
    ).reset_index()
    
    df_merged_grouped_agg.loc[:, 'snomed_seq_len_mean'] = df_merged_grouped_agg.loc[:, 'snomed_seq_len_mean'].astype(int)
    df_merged_grouped_agg.loc[:, 'medications_seq_len_mean'] = df_merged_grouped_agg.loc[:, 'medications_seq_len_mean'].astype(int)
    
    display(df_merged_grouped_agg[df_merged_grouped_agg.EVENT_TYPE=='observation'][['EVENT_TYPE', 'has_cancer_diagnosis', 'snomed_seq_len_mean', 'snomed_seq_len_min', 
                                                                                    'snomed_seq_len_max', 'snomed_seq_len_std']])
    display(df_merged_grouped_agg[df_merged_grouped_agg.EVENT_TYPE=='medication'][['EVENT_TYPE', 'has_cancer_diagnosis', 'medications_seq_len_mean', 'medications_seq_len_min',
                                                                                  'medications_seq_len_max', 'medications_seq_len_std']])


#### Pre-processing

In [ ]:
# pre_process data
def pre_process(df, config):
    
    # upper case columns
    df.columns = df.columns.str.upper()
    
    # strip id to ensure uniqueness
    df['PATIENT_GUID'] = df['PATIENT_GUID'].str.strip()
    
    # rename columns
    df = df.rename(columns={'PATIENT_GUID':'patient_id',
                            'SEX':'gender',
                            'EVENT_AGE':'age_at_last_snomed_record',  # v144: per-event column; collapsed to LAST event in get_seq_data
                            'EVENT_DATE':'snomed_recorded_date',
                            'PATIENT_ETHNICITY':'ethnicity',
                            'SNOMED_C_T_CONCEPT_ID':'snomed_id',
                            'TERM':'snomed_text',
                            'CANCER_CLASS':'has_cancer_diagnosis'
                            })
    
    # print columns
    print(df.columns)
    
    # v113 Lever 2: drop event rows whose snomed_text matches a 'shortcut' pattern
    # (other-cancer screening, generic admin procedures, insurance medicals).
    if bool(config.get('exclude_shortcut_codes', False)):
        import re as _re
        shortcut_patterns = config.get('shortcut_text_patterns', [
            # breast variant: REMOVED mammogr / breast.*screen — they are the
            # PRIMARY signal for breast cancer, not a shortcut.
            # Other-cancer screening (still off-topic for breast)
            r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
            r'faecal.*occult', r'\bfobt\b',
            r'cervical.*smear', r'cervical.*screen',
            r'\bpsa\b', r'prostate.*screen',
            r'aaa.*screen', r'abdominal.*aortic.*screen',
            # Lung-specific (off-topic for breast cancer — kept as shortcut exclusions)
            r'bronchoscop', r'spirom', r'peak.*flow', r'\bcxr\b',
            r'chest.*x[-\s]?ray', r'ct.*thorax', r'pulmonary.*function',
            # Generic admin / utilisation proxies
            r'driving.*licen', r'\bfit\s*for\s*work\b',
            r'insurance.*medical', r'life assurance',
            r'\bbill.*fee\b', r'administrative procedure',
            r'did not attend', r'\bdna\b',
            # Other-system investigations unlikely to be breast-signal
            r'retinal.*screen', r'diabetic.*eye',
            r'dexa.*scan', r'bone.*density',
        ])
        text_col = 'snomed_text' if 'snomed_text' in df.columns else (
            'TERM' if 'TERM' in df.columns else None)
        if text_col and shortcut_patterns:
            pat = _re.compile('|'.join(shortcut_patterns), _re.IGNORECASE)
            text_lower = df[text_col].astype(str).str.lower()
            mask = text_lower.apply(lambda s: bool(pat.search(s)))
            n_drop = int(mask.sum())
            if n_drop > 0:
                df = df[~mask]
                print(f'  [exclude_shortcut_codes] dropped {n_drop} event rows matching '
                      f'{len(shortcut_patterns)} shortcut patterns (text column: {text_col})')
        else:
            print(f'  [exclude_shortcut_codes] skipped — '
                  f'text_col={text_col}, patterns_len={len(shortcut_patterns)}')

    # drop duplicates
    df = df.drop_duplicates()
    
    # print info
    print_df_info('Data - Raw', df)
    print(f'\n\tCancer Types:')
    for ct in df.CANCER_ID.unique().tolist():
        print(f'\t\t* {ct}')
    
    # compute_stats
    compute_stats(df, config)
    
    return df


#### Processing

In [ ]:
# Process data — symmetric column layout, branch-aware filtering.
#   obs rows:  snomed_id, snomed_text
#   med rows:  med_code_id, med_text
# EVENT_TYPE is preserved only when BOTH branches are enabled (otherwise it's implicit).
def process_data(df, config):
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))
    if not (use_snomed or use_med):
        raise ValueError("At least one of use_snomed_branch / use_med_branch must be True.")

    # Materialise med-only identity columns from MED_CODE_ID / DRUG_TERM.
    if use_med:
        med = (df['EVENT_TYPE'] == 'medication')
        df['med_code_id'] = pd.NA
        df['med_text'] = pd.NA
        df.loc[med, 'med_code_id'] = df.loc[med, 'MED_CODE_ID']
        df.loc[med, 'med_text'] = 'med_' + df.loc[med, 'DRUG_TERM'].astype(str)

    # Keep only rows belonging to enabled event types.
    if use_snomed and use_med:
        df = df[df['EVENT_TYPE'].isin(['observation', 'medication'])]
    elif use_snomed:
        df = df[df['EVENT_TYPE'] == 'observation']
    else:  # use_med only
        df = df[df['EVENT_TYPE'] == 'medication']

    # Drop raw source columns that have been mapped or aren't used downstream.
    drop_cols = ['VALUE', 'DURATION', 'MED_CODE_ID', 'DRUG_TERM',
                 'ASSOCIATED_TEXT', 'CANCER_ID', 'PATIENT_AGE']
    # EVENT_TYPE is only meaningful when both branches coexist.
    if not (use_snomed and use_med):
        drop_cols.append('EVENT_TYPE')
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # Normalise unknown ethnicity.
    unknown_ethnicity = df['ethnicity'].isna()
    df.loc[unknown_ethnicity, 'ethnicity'] = 'unknown'

    df['snomed_recorded_date'] = pd.to_datetime(df['snomed_recorded_date'])

    cols = ['patient_id', 'gender', 'ethnicity', 'snomed_recorded_date',
            'has_cancer_diagnosis', 'age_at_last_snomed_record']
    # v110: preserve ANCHOR_YEAR through the pipeline so plot_cohort_counts_per_year
    # can render on the final balanced cohort.
    if 'ANCHOR_YEAR' in df.columns:
        cols.append('ANCHOR_YEAR')
    if use_snomed:
        cols += ['snomed_id', 'snomed_text']
    if use_med:
        cols += ['med_code_id', 'med_text']
    if use_snomed and use_med:
        cols.append('EVENT_TYPE')
    df = df[cols]

    print_df_info('Processed Data:', df)
    print(f"Branches enabled — SNOMED: {use_snomed}  MED: {use_med}")
    return df


#### Data Filtering

In [ ]:
# Per-event-type dropna. Core cols apply to every row; branch-specific ID cols are
# only checked when that branch is enabled / present.
def get_df_filtered(df, config):
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    core_cols = ['patient_id', 'snomed_recorded_date', 'has_cancer_diagnosis',
                 'age_at_last_snomed_record', 'gender', 'ethnicity']
    if 'EVENT_TYPE' in df.columns:
        core_cols = core_cols + ['EVENT_TYPE']
    df_filtered = df.dropna(subset=core_cols)

    if use_snomed and use_med and 'EVENT_TYPE' in df_filtered.columns:
        obs_bad = (df_filtered['EVENT_TYPE'] == 'observation') & (df_filtered['snomed_id'].isna())
        med_bad = (df_filtered['EVENT_TYPE'] == 'medication') & (df_filtered['med_code_id'].isna())
        df_filtered = df_filtered[~(obs_bad | med_bad)]
    elif use_snomed:
        df_filtered = df_filtered.dropna(subset=['snomed_id'])
    elif use_med:
        df_filtered = df_filtered.dropna(subset=['med_code_id'])

    cancer_patient_ids = df_filtered[df_filtered.has_cancer_diagnosis == 1].patient_id.unique().tolist()
    no_cancer_patient_ids = df_filtered[df_filtered.has_cancer_diagnosis == 0].patient_id.unique().tolist()

    listed_in_both = df_filtered.patient_id.isin(cancer_patient_ids) & df_filtered.patient_id.isin(no_cancer_patient_ids)
    print(f'\n{len(df_filtered[listed_in_both].patient_id.unique().tolist())} patients listed in both cancer and no-cancer raw data -> filtering them out')
    print(f'{df_filtered[listed_in_both].patient_id.unique().tolist()}')
    df_filtered = df_filtered[~listed_in_both]

    print_df_info('df_filtered', df_filtered)
    return df_filtered


In [ ]:
# get_df_capped_shuffled
def get_df_capped_shuffled(df_filtered, config):
    # update the patients lists
    cancer_patient_ids = df_filtered[(df_filtered.has_cancer_diagnosis == 1)].patient_id.unique().tolist()
    no_cancer_patient_ids = df_filtered[(df_filtered.has_cancer_diagnosis == 0)].patient_id.unique().tolist()
    
    # select a capped subset with balanced data
    df_capped_shuffled = df_filtered[df_filtered.patient_id.isin(cancer_patient_ids[:config['MAX_CANCER_PATIENTS']]) | df_filtered.patient_id.isin(no_cancer_patient_ids[:config['MAX_NO_CANCER_PATIENTS']])]

    # shuffle rows & reset index
    df_capped_shuffled = df_capped_shuffled.sample(frac=1,random_state=config['SEED']).reset_index(drop=True)

    # print info
    print_df_info('df_capped_shuffled:', df_capped_shuffled)
    
    return df_capped_shuffled

#### Show Data

In [ ]:
def show_data(df, config):
    # Print a concise summary of the DataFrame
    print('\n\n')
    display(df.info())

    # Display the first rows of the DataFrame
    print('\n\n')
    display(df.head())

#### Feature Engineering for Sequences

Convert 'snomed_id' data, in addition to other categorical features ('gender', 'ethnicity') into sequences per patient for LSTM input, ensuring consistent sequence lengths through padding.

This involves creating a mapping for unique `snomed_id` values to contiguous integers.

In addition, we perform one-hot encoding of the `gender` and `ethnicity` columns using `pd.get_dummies()` and merging them back into the DataFrame.

This will prepare the features for subsequent sequence creation.

In [ ]:
# Build one vocabulary per ENABLED branch. Disabled branches get a trivial 2-entry
# vocabulary (PAD + <RARE>) so downstream code can always reference `vocabulary_size` etc.
def build_code_mapping(df, event_type, code_column, config):
    min_freq = config.get('min_code_frequency', 10)

    if 'EVENT_TYPE' in df.columns and event_type is not None:
        df_sub = df[df['EVENT_TYPE'] == event_type]
    else:
        df_sub = df

    code_counts = df_sub[code_column].dropna().astype(str).value_counts()
    total_codes = len(code_counts)

    frequent_codes = set(code_counts[code_counts >= min_freq].index)
    rare_codes = total_codes - len(frequent_codes)

    print(f"[{event_type or 'all'}] Total unique codes: {total_codes}")
    print(f"[{event_type or 'all'}] Codes appearing >= {min_freq} times (kept): {len(frequent_codes)}")
    print(f"[{event_type or 'all'}] Rare codes replaced with <RARE> token: {rare_codes}")

    code_to_int = {'PAD_VALUE': 0, '<RARE>': 1}
    for i, code in enumerate(sorted(frequent_codes)):
        code_to_int[code] = i + 2

    vocabulary_size = len(frequent_codes) + 2

    print(f"[{event_type or 'all'}] Vocabulary size (incl. padding + rare): {vocabulary_size}")
    return list(frequent_codes), code_to_int, vocabulary_size


def build_code_mappings(df, config):
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))
    has_event_type = 'EVENT_TYPE' in df.columns

    empty_map = {'PAD_VALUE': 0, '<RARE>': 1}
    snomed_ids, snomed_id_to_int, snomed_vocab_size = [], empty_map, 2
    med_ids, med_id_to_int, med_vocab_size = [], empty_map, 2

    if use_snomed:
        event_type = 'observation' if has_event_type else None
        snomed_ids, snomed_id_to_int, snomed_vocab_size = build_code_mapping(
            df, event_type, 'snomed_id', config)

    if use_med:
        event_type = 'medication' if has_event_type else None
        med_ids, med_id_to_int, med_vocab_size = build_code_mapping(
            df, event_type, 'med_code_id', config)

    return snomed_ids, snomed_id_to_int, snomed_vocab_size, med_ids, med_id_to_int, med_vocab_size


In [ ]:
# Encode codes. Each branch populates its own integer-id column independently.
#   snomed_id_encoded -> populated when SNOMED branch is enabled
#   med_id_encoded    -> populated when MED branch is enabled
# If only one branch is enabled, EVENT_TYPE is absent and the whole df is that branch.
def map_codes(df, snomed_id_to_int, med_id_to_int=None, config=None):
    use_snomed = bool(config.get('use_snomed_branch', True)) if config else True
    use_med = bool(config.get('use_med_branch', True)) if config else (med_id_to_int is not None)
    has_event_type = 'EVENT_TYPE' in df.columns

    if use_snomed:
        df['snomed_id_encoded'] = 0
        if has_event_type:
            obs_mask = df['EVENT_TYPE'] == 'observation'
            df.loc[obs_mask, 'snomed_id_encoded'] = df.loc[obs_mask, 'snomed_id'].astype(str).map(
                lambda x: snomed_id_to_int.get(x, snomed_id_to_int['<RARE>'])
            )
        else:
            df['snomed_id_encoded'] = df['snomed_id'].astype(str).map(
                lambda x: snomed_id_to_int.get(x, snomed_id_to_int['<RARE>'])
            )

    if use_med and med_id_to_int is not None:
        df['med_id_encoded'] = 0
        if has_event_type:
            med_mask = df['EVENT_TYPE'] == 'medication'
            df.loc[med_mask, 'med_id_encoded'] = df.loc[med_mask, 'med_code_id'].astype(str).map(
                lambda x: med_id_to_int.get(x, med_id_to_int['<RARE>'])
            )
        else:
            df['med_id_encoded'] = df['med_code_id'].astype(str).map(
                lambda x: med_id_to_int.get(x, med_id_to_int['<RARE>'])
            )

    return df


In [ ]:
# encode_df
def encode_df(df, config):
    # 1.2. One-hot encode 'gender' and 'ethnicity'
    df_encoded = pd.get_dummies(df, columns=['gender', 'ethnicity'], prefix=['gender', 'ethnicity'])

    # Display the head of the DataFrame to verify changes
    # df_encoded.head()
    
    return df_encoded

Next, we need to structure the data into sequences for each patient. This involves grouping by patient, sorting chronologically, and extracting the encoded SNOMED IDs, other numerical/one-hot encoded features, and the target variable into separate lists for each patient.

In [ ]:
def get_other_feature_cols(df_encoded, config):
    # Define the columns for other features (excluding patient_id, date_of_birth, snomed_recorded_date, snomed_text, snomed_id, has_cancer_diagnosis, snomed_id_encoded)
    # v144: 'age_at_last_snomed_record' is the patient's age at their LAST recorded
    # SNOMED event (built in get_seq_data via iloc[-1]); plus one-hot encoded 'gender_*'.
    other_feature_cols = ['age_at_last_snomed_record'] + \
                             [col for col in df_encoded.columns if col.startswith('gender_')] # or col.startswith('ethnicity_')]
    
    return other_feature_cols

In [ ]:
def get_seq_data(df_encoded, other_feature_cols, config):
    MAX_SNOMED = config['MAX_SNOMED_SEQ_LENGTH']
    MAX_MED = config['MAX_MED_SEQ_LENGTH']
    MIN_SNOMED = int(config.get('MIN_SNOMED_SEQ_LENGTH', 0))
    MIN_MED = int(config.get('MIN_MED_SEQ_LENGTH', 0))
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))
    has_event_type = 'EVENT_TYPE' in df_encoded.columns

    patient_ids = []
    snomed_id_sequences = []
    med_id_sequences = []
    other_feature_sequences = []
    y_patients = []
    patient_id_to_snomed_seq = dict()
    patient_id_to_med_seq = dict()
    patient_id_to_other_seq = dict()

    n_total = 0
    n_dropped_snomed = 0
    n_dropped_med = 0
    n_dropped_empty = 0

    for patient_id, group in df_encoded.groupby('patient_id'):
        n_total += 1
        group_sorted = group.sort_values(by='snomed_recorded_date')

        snomed_seq = []
        med_seq = []

        if use_snomed:
            if has_event_type:
                obs_group = group_sorted[group_sorted['EVENT_TYPE'] == 'observation']
                snomed_seq = obs_group['snomed_id_encoded'].tolist()
            else:
                snomed_seq = group_sorted['snomed_id_encoded'].tolist()

        if use_med:
            if has_event_type:
                med_group = group_sorted[group_sorted['EVENT_TYPE'] == 'medication']
                med_seq = med_group['med_id_encoded'].tolist()
            else:
                med_seq = group_sorted['med_id_encoded'].tolist()

        # Per-branch minimum-length filter (only enforced when branch is enabled).
        if use_snomed and len(snomed_seq) < MIN_SNOMED:
            n_dropped_snomed += 1
            continue
        if use_med and len(med_seq) < MIN_MED:
            n_dropped_med += 1
            continue

        # Safety net: require >=1 event on an enabled branch (catches min=0 + empty patient).
        total_enabled_events = (len(snomed_seq) if use_snomed else 0) + (len(med_seq) if use_med else 0)
        if total_enabled_events == 0:
            n_dropped_empty += 1
            continue

        patient_ids.append(patient_id)
        snomed_id_sequences.append(snomed_seq[-MAX_SNOMED:] if snomed_seq else [])
        med_id_sequences.append(med_seq[-MAX_MED:] if med_seq else [])

        # v144: use the LAST recorded event's row for the static features so the
        # 'age_at_last_snomed_record' value is the patient's age at their most recent
        # SNOMED observation (was iloc[0] / first-event age before v144).
        other_seq = group_sorted[other_feature_cols].iloc[-1].values.tolist()
        other_feature_sequences.append(other_seq)

        patient_id_to_snomed_seq[patient_id] = snomed_seq
        patient_id_to_med_seq[patient_id] = med_seq
        patient_id_to_other_seq[patient_id] = other_seq

        y_patients.append(group_sorted['has_cancer_diagnosis'].iloc[0])

    # Drop summary — shows the effect of the min-length thresholds.
    print(f"\nget_seq_data filter summary "
          f"(use_snomed={use_snomed}, use_med={use_med}, "
          f"MIN_SNOMED_SEQ_LENGTH={MIN_SNOMED}, MIN_MED_SEQ_LENGTH={MIN_MED}):")
    print(f"  Patients scanned: {n_total}")
    if use_snomed:
        print(f"  Dropped (SNOMED < {MIN_SNOMED}): {n_dropped_snomed}")
    if use_med:
        print(f"  Dropped (MED    < {MIN_MED}): {n_dropped_med}")
    print(f"  Dropped (empty on enabled branches): {n_dropped_empty}")
    print(f"  Kept: {len(patient_ids)}")

    print(f"other_feature_sequences: {len(other_feature_sequences)}")
    print(f"snomed_id_sequences: {len(snomed_id_sequences)}")
    print(f"med_id_sequences: {len(med_id_sequences)}")
    if snomed_id_sequences:
        print(f"First snomed_id_seq length: {len(snomed_id_sequences[0])}")
    if med_id_sequences:
        print(f"First med_id_seq length: {len(med_id_sequences[0])}")
    print(f"First other_feature_vec length: {len(other_feature_sequences[0])}")
    print(f"other_feature_cols: {len(other_feature_cols)}")

    if snomed_id_sequences:
        print(f"\nFirst snomed_id_seq: \n\n{snomed_id_sequences[0]}")
    if med_id_sequences:
        print(f"\nFirst med_id_seq: \n\n{med_id_sequences[0]}")
    print(f"\nFirst other_feature_vec (static patient features): \n\n{other_feature_sequences[0]}")

    return (patient_ids, snomed_id_sequences, med_id_sequences, other_feature_sequences,
            y_patients, patient_id_to_snomed_seq, patient_id_to_med_seq, patient_id_to_other_seq)


In [ ]:
# sequence length distribution
def seq_distribution(snomed_id_sequences, med_id_sequences, config):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    pd.Series([len(s) for s in snomed_id_sequences]).hist(bins=30, ax=axes[0])
    axes[0].set_title('SNOMED sequence length distribution')
    axes[0].set_xlabel('length')
    if med_id_sequences and any(len(s) > 0 for s in med_id_sequences):
        pd.Series([len(s) for s in med_id_sequences]).hist(bins=30, ax=axes[1])
        axes[1].set_title('Medication sequence length distribution')
        axes[1].set_xlabel('length')
    else:
        axes[1].set_title('Medication sequences (none)')
    plt.tight_layout()
    plt.show()


In [ ]:
from keras.utils import pad_sequences

def get_padded_sequences(snomed_id_sequences, med_id_sequences, other_feature_sequences, config):
    max_snomed = config['MAX_SNOMED_SEQ_LENGTH']
    max_med = config['MAX_MED_SEQ_LENGTH']

    X_snomed_ids_padded = pad_sequences(
        snomed_id_sequences, maxlen=max_snomed, padding='post', truncating='pre', value=0
    )

    # Empty med sequences are padded to all-zeros (vocab id 0 = padding).
    if med_id_sequences and any(len(s) > 0 for s in med_id_sequences):
        med_seqs_safe = [s if len(s) > 0 else [0] for s in med_id_sequences]
        X_med_ids_padded = pad_sequences(
            med_seqs_safe, maxlen=max_med, padding='post', truncating='pre', value=0
        )
    else:
        X_med_ids_padded = np.zeros((len(snomed_id_sequences), max_med), dtype=np.int32)

    X_other_features_padded = np.array(other_feature_sequences, dtype=np.float32)

    print(f"Shape of X_snomed_ids_padded: {X_snomed_ids_padded.shape}")
    print(f"Shape of X_med_ids_padded: {X_med_ids_padded.shape}")
    print(f"Shape of X_other_features_padded: {X_other_features_padded.shape}")

    print("\nExample padded snomed_id sequence (first patient, first 10):")
    print(X_snomed_ids_padded[0][:10])
    print("\nExample padded med_id sequence (first patient, first 10):")
    print(X_med_ids_padded[0][:10])
    print("\nExample other features vector (first patient):")
    print(X_other_features_padded[0])

    return X_snomed_ids_padded, X_med_ids_padded, X_other_features_padded


In [ ]:
def get_y_patients_numpy(y_patients, config):
    # Convert y_patients to a NumPy array
    y_patients = np.array(y_patients)
    
    print(f"Shape of y_patients: {y_patients.shape}")
    
    return y_patients

#### Split Data into Training, Validation, and Testing Sets


Split the preprocessed patient sequences and their corresponding target labels into training, validation (10% of the data), and testing (10% of the data) datasets, ensuring patient data is not mixed across sets.

In order to do that, we need to import the `train_test_split` function from `sklearn.model_selection` to split the patient sequences and target labels into training, validation, and testing datasets, ensuring patient data is not mixed across sets as per the subtask instructions.



In [ ]:
from sklearn.model_selection import train_test_split

def split_data(patient_ids, X_snomed_ids_padded, X_med_ids_padded, X_other_features_padded, y_patients, config):
    split_seed = config.get('SPLIT_SEED', config.get('SEED', 42))

    # First split: 80% train, 20% temp (val+test)
    (patient_ids_train, patient_ids_temp,
     X_snomed_ids_train, X_snomed_ids_temp,
     X_med_ids_train, X_med_ids_temp,
     X_other_features_train, X_other_features_temp,
     y_train, y_temp) = train_test_split(
        patient_ids, X_snomed_ids_padded, X_med_ids_padded, X_other_features_padded, y_patients,
        test_size=0.20, random_state=split_seed, stratify=y_patients
    )

    # Second split: 50/50 of temp -> val, test
    (patient_ids_val, patient_ids_test,
     X_snomed_ids_val, X_snomed_ids_test,
     X_med_ids_val, X_med_ids_test,
     X_other_features_val, X_other_features_test,
     y_val, y_test) = train_test_split(
        patient_ids_temp, X_snomed_ids_temp, X_med_ids_temp, X_other_features_temp, y_temp,
        test_size=0.50, random_state=split_seed, stratify=y_temp
    )

    print(f"Shapes of the datasets after splitting:")
    print(f"X_snomed_ids_train: {X_snomed_ids_train.shape}")
    print(f"X_med_ids_train: {X_med_ids_train.shape}")
    print(f"X_other_features_train: {X_other_features_train.shape}")
    print(f"y_train: {y_train.shape}\n")

    print(f"X_snomed_ids_val: {X_snomed_ids_val.shape}")
    print(f"X_med_ids_val: {X_med_ids_val.shape}")
    print(f"X_other_features_val: {X_other_features_val.shape}")
    print(f"y_val: {y_val.shape}\n")

    print(f"X_snomed_ids_test: {X_snomed_ids_test.shape}")
    print(f"X_med_ids_test: {X_med_ids_test.shape}")
    print(f"X_other_features_test: {X_other_features_test.shape}")
    print(f"y_test: {y_test.shape}")

    def _report(y, name):
        c = int(np.sum(y == 1)); n = int(np.sum(y == 0))
        total = c + n
        pct = 100.0 * c / total if total else 0.0
        print(f"{name}: {c} cancer / {n} no-cancer ({pct:.1f}% positive)")

    _report(y_train, 'Train')
    _report(y_val, 'Val')
    _report(y_test, 'Test')

    return (patient_ids_train, patient_ids_val, patient_ids_test,
            X_snomed_ids_train, X_snomed_ids_val, X_snomed_ids_test,
            X_med_ids_train, X_med_ids_val, X_med_ids_test,
            X_other_features_train, X_other_features_val, X_other_features_test,
            y_train, y_val, y_test)


#### Full Data Pipeline

In [ ]:
# Full Data Pipeline — split into common (run once) and variant (per data-level params)

def plot_cohort_counts_per_year(df, config, title_suffix=''):
    """Line chart of patient counts per anchor year — cancer line uses each
    patient's earliest-diagnosis year, non-cancer line uses each patient's
    latest-observation-event year (v38's anchor_year). Both lines share the same
    x-axis ticks. Year range set by config['cohort_plot_year_start'] and
    ['cohort_plot_year_end'] (inclusive).
    Tolerates pre- and post-pre_process column names so the same helper can be
    called on the raw loaded DataFrame (PATIENT_GUID, CANCER_CLASS, ANCHOR_YEAR)
    AND on the post-processed balanced DataFrame (patient_id, has_cancer_diagnosis,
    ANCHOR_YEAR)."""
    import matplotlib.pyplot as plt

    def _pick(cands):
        for c in cands:
            if c in df.columns:
                return c
        return None
    patient_col = _pick(['patient_id', 'PATIENT_GUID', 'PATIENT_ID', 'patient_guid'])
    class_col   = _pick(['has_cancer_diagnosis', 'CANCER_CLASS', 'HAS_CANCER_DIAGNOSIS', 'cancer_class'])
    year_col    = _pick(['ANCHOR_YEAR', 'anchor_year'])
    if not (patient_col and class_col and year_col):
        print(f'[plot_cohort_counts_per_year] skipped — missing columns '
              f'(patient={patient_col}, class={class_col}, year={year_col})')
        return

    year_start = int(config.get('cohort_plot_year_start', 2010))
    year_end   = int(config.get('cohort_plot_year_end',   2025))
    years = list(range(year_start, year_end + 1))

    by_patient = df[[patient_col, class_col, year_col]].drop_duplicates()
    cancer_counts_s    = by_patient[by_patient[class_col] == 1].groupby(year_col).size()
    noncancer_counts_s = by_patient[by_patient[class_col] == 0].groupby(year_col).size()
    cancer_counts    = [int(cancer_counts_s.get(y, 0))    for y in years]
    noncancer_counts = [int(noncancer_counts_s.get(y, 0)) for y in years]

    # v110 fix: seaborn grouped bar chart; palette matches the age-distribution plot
    # (Cancer = '#ff7f0e' orange, Non-Cancer = '#1f77b4' blue).
    import seaborn as sns
    plot_df = pd.DataFrame({
        'year':  list(years) * 2,
        'count': cancer_counts + noncancer_counts,
        'class': (['Cancer (earliest diagnosis year)'] * len(years)
                  + ['Non-Cancer (matched observation year)'] * len(years)),
    })
    plt.figure(figsize=(max(10, 0.55 * len(years) + 2), 5))
    ax = sns.barplot(
        data=plot_df, x='year', y='count', hue='class',
        palette={
            'Cancer (earliest diagnosis year)':     '#ff7f0e',
            'Non-Cancer (matched observation year)': '#1f77b4',
        },
        edgecolor='none', alpha=0.8,
    )
    ax.set_xlabel('Year')
    ax.set_ylabel('Patient count')
    title = f'Cohort counts per anchor year  ({year_start}-{year_end})'
    if title_suffix:
        title = f'{title} — {title_suffix}'
    ax.set_title(title)
    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(title=None)
    plt.tight_layout()
    plt.show()


def prepare_data_common(config):
    loaded_df = load_data(config)
    # v110: initial cohort-audit line chart on the freshly loaded data (pre-processing).
    plot_cohort_counts_per_year(loaded_df, config, title_suffix='initial (raw loaded data)')
    df_pre_processed = pre_process(loaded_df, config)
    df_processed = process_data(df_pre_processed, config)
    df_filtered = get_df_filtered(df_processed, config)
    df = get_df_capped_shuffled(df_filtered, config)
    # v110: cohort-audit line chart on the final balanced training cohort.
    plot_cohort_counts_per_year(df, config, title_suffix='final balanced cohort')
    df_encoded = encode_df(df, config)
    other_feature_cols = get_other_feature_cols(df_encoded, config)
    return {
        'df': df, 'df_filtered': df_filtered, 'df_encoded': df_encoded,
        'other_feature_cols': other_feature_cols
    }


def prepare_data_variant(common_data, config):
    df = common_data['df']
    df_encoded = common_data['df_encoded']
    other_feature_cols = common_data['other_feature_cols']

    (unique_snomed_ids, snomed_id_to_int, snomed_vocabulary_size,
     unique_med_ids, med_id_to_int, med_vocabulary_size) = build_code_mappings(df, config)

    df_mapped = map_codes(df_encoded.copy(), snomed_id_to_int, med_id_to_int, config)

    (patient_ids, snomed_id_sequences, med_id_sequences, other_feature_sequences,
     y_patients, patient_id_to_snomed_seq, patient_id_to_med_seq,
     patient_id_to_other_seq) = get_seq_data(df_mapped, other_feature_cols, config)

    X_snomed_ids_padded, X_med_ids_padded, X_other_features_padded = get_padded_sequences(
        snomed_id_sequences, med_id_sequences, other_feature_sequences, config)

    y_patients = get_y_patients_numpy(y_patients, config)

    (patient_ids_train, patient_ids_val, patient_ids_test,
     X_snomed_ids_train, X_snomed_ids_val, X_snomed_ids_test,
     X_med_ids_train, X_med_ids_val, X_med_ids_test,
     X_other_features_train, X_other_features_val, X_other_features_test,
     y_train, y_val, y_test) = split_data(
        patient_ids, X_snomed_ids_padded, X_med_ids_padded, X_other_features_padded, y_patients, config)

    return {
        **common_data,
        'snomed_id_to_int': snomed_id_to_int,
        'med_id_to_int': med_id_to_int,
        'patient_ids_train': patient_ids_train, 'patient_ids_val': patient_ids_val, 'patient_ids_test': patient_ids_test,
        'X_snomed_ids_padded': X_snomed_ids_padded,
        'X_med_ids_padded': X_med_ids_padded,
        'X_other_features_padded': X_other_features_padded,
        'X_snomed_ids_train': X_snomed_ids_train, 'X_snomed_ids_val': X_snomed_ids_val, 'X_snomed_ids_test': X_snomed_ids_test,
        'X_med_ids_train': X_med_ids_train, 'X_med_ids_val': X_med_ids_val, 'X_med_ids_test': X_med_ids_test,
        'X_other_features_train': X_other_features_train, 'X_other_features_val': X_other_features_val, 'X_other_features_test': X_other_features_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'snomed_id_sequences': snomed_id_sequences,
        'med_id_sequences': med_id_sequences,
        'other_feature_sequences': other_feature_sequences,
        'y_patients': y_patients, 'patient_ids': patient_ids,
        'patient_id_to_snomed_seq': patient_id_to_snomed_seq,
        'patient_id_to_med_seq': patient_id_to_med_seq,
        'patient_id_to_other_seq': patient_id_to_other_seq,
        'max_snomed_seq_length': X_snomed_ids_padded.shape[1],
        'max_med_seq_length': X_med_ids_padded.shape[1],
        'number_of_other_features': X_other_features_padded.shape[1],
        'vocabulary_size': snomed_vocabulary_size,
        'med_vocabulary_size': med_vocabulary_size,
        'df_mapped': df_mapped
    }


def prepare_data(config):
    common = prepare_data_common(config)
    return prepare_data_variant(common, config)


### Build Keras LSTM Model

Design and compile a Keras LSTM model suitable for binary classification, considering the shape of the input sequences and the type of features used.

To do that, we need to import the necessary Keras modules to define the model architecture, including `Input`, `Embedding`, `LSTM`, `Dense`, and `Model`. This is the first step towards building the LSTM model.

In [ ]:
# Custom Keras metric: weighted average of sensitivity and specificity
# Also defines `model_inputs(config, X_snomed, X_med, X_other)` — a centralised helper that
# builds the feed dict for model.fit / model.predict, including only the branch inputs that
# the config enables. Used by every training, evaluation, explainability, and prediction
# path so disabled branches are automatically omitted.
import tensorflow as tf
import keras


def model_inputs(config, X_snomed=None, X_med=None, X_other=None):
    """Build a model input dict that matches the enabled-branch architecture."""
    inputs = {}
    if config.get('use_snomed_branch', True) and X_snomed is not None:
        inputs['snomed_id_input'] = X_snomed
    if config.get('use_med_branch', True) and X_med is not None:
        inputs['med_id_input'] = X_med
    if X_other is not None:
        inputs['other_features_input'] = X_other
    return inputs


@keras.saving.register_keras_serializable()
class WeightedSensSpec(tf.keras.metrics.Metric):
    """Custom metric: weighted average = (sw * sensitivity + spw * specificity) / (sw + spw).
    Returns a value between 0 and 1, where 1 is perfect."""

    def __init__(self, name='weighted_sens_spec', threshold=0.5, sensitivity_weight=5.0, specificity_weight=1.0, **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.sensitivity_weight = sensitivity_weight
        self.specificity_weight = specificity_weight
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')
        self.true_negatives = self.add_weight(name='tn', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
        y_pred = tf.cast(tf.reshape(y_pred, [-1]) >= self.threshold, tf.float32)

        self.true_positives.assign_add(tf.reduce_sum(y_true * y_pred))
        self.false_negatives.assign_add(tf.reduce_sum(y_true * (1 - y_pred)))
        self.true_negatives.assign_add(tf.reduce_sum((1 - y_true) * (1 - y_pred)))
        self.false_positives.assign_add(tf.reduce_sum((1 - y_true) * y_pred))

    def result(self):
        sensitivity = self.true_positives / (self.true_positives + self.false_negatives + tf.keras.backend.epsilon())
        specificity = self.true_negatives / (self.true_negatives + self.false_positives + tf.keras.backend.epsilon())
        return (self.sensitivity_weight * sensitivity + self.specificity_weight * specificity) / (self.sensitivity_weight + self.specificity_weight)

    def reset_state(self):
        self.true_positives.assign(0)
        self.false_negatives.assign(0)
        self.true_negatives.assign(0)
        self.false_positives.assign(0)

    def get_config(self):
        config = super().get_config()
        config['threshold'] = self.threshold
        config['sensitivity_weight'] = self.sensitivity_weight
        config['specificity_weight'] = self.specificity_weight
        return config


In [ ]:
# Build Model — one branch per enabled event type (SNOMED observations, medications)
# plus a static-feature branch. The fusion layer takes only the enabled contexts.
from keras.layers import Input, Embedding, LSTM, Dense, concatenate, Dropout, SpatialDropout1D, Softmax, Multiply, Layer
from keras.models import Model
from keras.optimizers import AdamW
from keras.losses import BinaryFocalCrossentropy, BinaryCrossentropy
from keras.regularizers import l2
import keras


@keras.saving.register_keras_serializable()
class SumOverTime(Layer):
    """Sum over the time axis (axis=1). Serializable replacement for Lambda."""
    def call(self, x):
        return keras.ops.sum(x, axis=1)


def _attention_pool(lstm_out, name_prefix):
    attention_scores = Dense(1, activation='tanh', name=f'{name_prefix}_attention_dense')(lstm_out)
    attention_weights = Softmax(axis=1, name=f'{name_prefix}_attention_softmax')(attention_scores)
    weighted = Multiply(name=f'{name_prefix}_attention_multiply')([lstm_out, attention_weights])
    context = SumOverTime(name=f'{name_prefix}_attention_context')(weighted)
    return context


def build_model(config, vocabulary_size, med_vocabulary_size=None,
                max_snomed_seq_length=None, max_med_seq_length=None,
                number_of_other_features=None,
                X_snomed_ids_padded=None, X_med_ids_padded=None, X_other_features_padded=None):

    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))
    if not (use_snomed or use_med):
        raise ValueError("At least one of use_snomed_branch / use_med_branch must be True.")

    if use_snomed and max_snomed_seq_length is None:
        max_snomed_seq_length = X_snomed_ids_padded.shape[1]
    if use_med and max_med_seq_length is None:
        max_med_seq_length = X_med_ids_padded.shape[1] if X_med_ids_padded is not None else config['MAX_MED_SEQ_LENGTH']
    if number_of_other_features is None:
        number_of_other_features = X_other_features_padded.shape[1]
    if med_vocabulary_size is None:
        med_vocabulary_size = 2

    embedding_dim = config['embedding_dim']
    med_embedding_dim = config.get('med_embedding_dim', embedding_dim)
    med_lstm_units = config.get('med_lstm_units', max(8, config['snomed_lstm_units'] // 2))
    embedding_dropout = config.get('embedding_dropout', 0.3)
    dense_dropout = config.get('dense_dropout', 0.4)
    l2_reg = config.get('l2_reg', 1e-3)
    weight_decay = config.get('weight_decay', 1e-3)
    label_smoothing = config.get('label_smoothing', 0.1)
    use_focal_loss = config.get('use_focal_loss', False)
    mask_rate = config.get('mask_rate', 0.0)

    print(f"Branches enabled — SNOMED: {use_snomed}  MED: {use_med}")
    if use_snomed:
        print(f"Max SNOMED seq length: {max_snomed_seq_length}  |  SNOMED vocab size: {vocabulary_size}"
              f"  |  SNOMED embedding dim: {embedding_dim}  |  SNOMED LSTM units: {config['snomed_lstm_units']}")
    if use_med:
        print(f"Max MED seq length:    {max_med_seq_length}  |  MED vocab size:    {med_vocabulary_size}"
              f"  |  MED embedding dim:    {med_embedding_dim}  |  MED LSTM units:    {med_lstm_units}")
    print(f"Number of other features: {number_of_other_features}")
    print(f"Embedding dropout: {embedding_dropout}, Dense dropout: {dense_dropout}")
    print(f"L2: {l2_reg}, Weight decay: {weight_decay}, Label smoothing: {label_smoothing}")
    print(f"Focal loss: {use_focal_loss} (gamma=2.0, stacked with label_smoothing when on), Mask rate: {mask_rate}")

    contexts = []
    inputs = []

    if use_snomed:
        snomed_id_input = Input(shape=(max_snomed_seq_length,), name='snomed_id_input')
        inputs.append(snomed_id_input)
        snomed_emb = Embedding(input_dim=vocabulary_size, output_dim=embedding_dim, name='snomed_embedding')(snomed_id_input)
        snomed_emb = SpatialDropout1D(embedding_dropout, name='snomed_embedding_dropout')(snomed_emb)
        snomed_lstm = LSTM(units=config['snomed_lstm_units'], return_sequences=True, name='snomed_lstm')(snomed_emb)
        contexts.append(_attention_pool(snomed_lstm, 'snomed'))

    if use_med:
        med_id_input = Input(shape=(max_med_seq_length,), name='med_id_input')
        inputs.append(med_id_input)
        med_emb = Embedding(input_dim=med_vocabulary_size, output_dim=med_embedding_dim, name='med_embedding')(med_id_input)
        med_emb = SpatialDropout1D(embedding_dropout, name='med_embedding_dropout')(med_emb)
        med_lstm = LSTM(units=med_lstm_units, return_sequences=True, name='med_lstm')(med_emb)
        contexts.append(_attention_pool(med_lstm, 'med'))

    # Static features — always present.
    other_features_input = Input(shape=(number_of_other_features,), name='other_features_input')
    inputs.append(other_features_input)
    other_features_dense = Dense(units=config['other_dense_units'], activation='relu',
                                 kernel_regularizer=l2(l2_reg), name='other_features_dense')(other_features_input)
    contexts.append(other_features_dense)

    if len(contexts) > 1:
        fused = concatenate(contexts, name='concatenation_layer')
    else:
        fused = contexts[0]

    x = Dropout(dense_dropout, name='dense_dropout')(fused)
    x = Dense(units=config['dense_1_units'], activation='relu',
              kernel_regularizer=l2(l2_reg), name='dense_1')(x)
    output_layer = Dense(units=1, activation='sigmoid', name='output_layer')(x)

    model = Model(inputs=inputs, outputs=output_layer)

    # v139: focal loss now carries label_smoothing too — the focusing factor
    # (down-weight easy examples) and soft targets (don't over-confidence)
    # address different problems and stack cleanly on Keras's
    # BinaryFocalCrossentropy. Set label_smoothing=0 to disable smoothing.
    if use_focal_loss:
        loss = BinaryFocalCrossentropy(gamma=2.0, label_smoothing=label_smoothing)
    else:
        loss = BinaryCrossentropy(label_smoothing=label_smoothing)

    optimizer = AdamW(learning_rate=config['learning_rate'], weight_decay=weight_decay)
    model.compile(optimizer=optimizer, loss=loss, metrics=[config['optimizer_metric']])

    print("Model compiled successfully.")
    model.summary()
    return model


### Train LSTM Model

Train the built LSTM model using the prepared training data and use the validation set for monitoring performance during training.

To train the LSTM model, we use the `fit` method, providing the training and validation data, specifying the number of epochs and batch size, and storing the training history as requested.

In [ ]:
# Train Model — with sequence masking augmentation on enabled branches.
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight


def mask_sequences(X, mask_rate, mask_value=1):
    """Randomly replace codes with <RARE> token during training. Vocab id 1 = <RARE>."""
    if mask_rate <= 0:
        return X
    X_masked = X.copy()
    mask = np.random.random(X_masked.shape) < mask_rate
    mask &= (X_masked != 0)
    X_masked[mask] = mask_value
    return X_masked


def train_model(model,
                X_snomed_ids_train, X_med_ids_train, X_other_features_train, y_train,
                X_snomed_ids_val, X_med_ids_val, X_other_features_val, y_val,
                config):
    epochs = config['epochs']
    batch_size = config['batch_size']
    mask_rate = config.get('mask_rate', 0.15)
    med_mask_rate = config.get('med_mask_rate', mask_rate)
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    callbacks = [
        EarlyStopping(
            monitor=config['monitor_metric'], mode='max',
            patience=config.get('early_stopping_patience', 7),
            restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(
            monitor=config['monitor_metric'], mode='max', factor=0.5,
            patience=config.get('lr_reduction_patience', 2),
            min_lr=1e-6, verbose=1),
    ]

    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weight_dict = dict(enumerate(class_weights))
    if 'cancer_class_weight' in config:
        class_weight_dict[1] = config['cancer_class_weight']
    print(f"Class weights: {class_weight_dict}")
    if use_snomed:
        print(f"SNOMED mask rate: {mask_rate}")
    if use_med:
        print(f"MED    mask rate: {med_mask_rate}")

    X_snomed_masked = mask_sequences(X_snomed_ids_train, mask_rate) if use_snomed else X_snomed_ids_train
    X_med_masked = mask_sequences(X_med_ids_train, med_mask_rate) if use_med else X_med_ids_train

    train_inputs = model_inputs(config, X_snomed=X_snomed_masked, X_med=X_med_masked,
                                X_other=X_other_features_train)
    val_inputs = model_inputs(config, X_snomed=X_snomed_ids_val, X_med=X_med_ids_val,
                              X_other=X_other_features_val)

    history = model.fit(
        train_inputs, y_train,
        epochs=epochs, batch_size=batch_size,
        validation_data=(val_inputs, y_val),
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )
    print("Model training completed. Best weights restored from early stopping.")
    return model, history


### Save the Model

In [ ]:
def save_model(model, config):
    saved_model_path = config['saved_model_path']
    model.save(saved_model_path)  # The file needs to end with the .keras extension
    return model

### Load the Model

In [ ]:
# Load model
def load_model(config):
    model = keras.models.load_model(config['saved_model_path'])
    return model

### Evaluate Model Performance

#### Core Evaluation

Evaluate the trained LSTM model's performance on the test set using appropriate metrics such as sensitivity, and specificity, accuracy, precision, recall, and F1-score.


In [ ]:
# Evaluate Model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


def evaluate_model_on_test_data(model, X_snomed_ids_test, X_med_ids_test, X_other_features_test, y_test, config):
    y_pred_proba = model.predict(model_inputs(
        config, X_snomed=X_snomed_ids_test, X_med=X_med_ids_test, X_other=X_other_features_test))

    # v147_k7 calibration patch: save raw probas + true labels
    import os as _os_cal
    _proba_dir = 'probas'
    _os_cal.makedirs(_proba_dir, exist_ok=True)
    _model_tag = config.get('saved_model_path', 'unknown.keras').split('/')[-1].replace('.keras', '')
    np.save(f'{_proba_dir}/{_model_tag}_y_proba_test.npy', np.asarray(y_pred_proba).ravel())
    np.save(f'{_proba_dir}/{_model_tag}_y_test.npy', np.asarray(y_test).ravel())
    print(f'[v147_k7 calibration] saved probas: {_proba_dir}/{_model_tag}_y_proba_test.npy')
    y_pred = (y_pred_proba > config['proba_threshold']).astype(int).ravel()

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)

    cm = confusion_matrix(y_test, y_pred)
    TN, FP, FN, TP = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    npv = TN / (TN + FN) if (TN + FN) > 0 else 0

    print(f"TP: {TP}\nTN: {TN}\nFP: {FP}\nFN: {FN}")
    print(f"\nModel Evaluation on Test Set:")
    print(f"Accuracy: {accuracy:.2%}")
    print(f"Precision: {precision:.2%}")
    print(f"Recall: {recall:.2%}")
    print(f"F1-Score: {f1:.2%}")
    print(f"ROC-AUC: {auc:.2%}")
    print(f"PPV: {ppv:.2%}")
    print(f"NPV: {npv:.2%}")
    print(f"Sensitivity: {sensitivity:.2%}")
    print(f"Specificity: {specificity:.2%}")

    return (y_pred_proba, y_pred, accuracy, precision, recall, f1, auc,
            cm, TN, FP, FN, TP, sensitivity, specificity, ppv, npv)


In [ ]:
# confusion matrix with details
import seaborn as sns
import matplotlib.pyplot as plt

def show_confusion_matrix(cm, config):
    ax= plt.subplot()
    # sns.heatmap(cm, annot=True, fmt='g', ax=ax);  #annot=True to annotate cells, ftm='g' to disable scientific notation

    group_names = ['True Neg','False Pos','False Neg','True Pos']
    group_counts = ['{0:0.0f}'.format(value) for value in cm.flatten()]
    group_percentages = ['{0:.2%}'.format(value) for value in cm.flatten()/np.sum(cm)]
    labels = [f'{v1}\n{v2}\n{v3}' for v1, v2, v3 in zip(group_names,group_counts,group_percentages)]
    labels = np.asarray(labels).reshape(2,2)
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues')

    # labels, title and ticks
    ax.set_xlabel('Predicted labels');ax.set_ylabel('True labels');
    ax.set_title('Confusion Matrix');
    ax.xaxis.set_ticklabels(['no cancer', 'cancer']); ax.yaxis.set_ticklabels(['no cancer', 'cancer']);

In [ ]:
# Display ROC curve
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def show_roc_curve(y_pred_proba, y_test, auc, config):
    y_pred_proba_1d = y_pred_proba.ravel()
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_1d)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc:.3f})')
    plt.plot([0, 1], [0, 1], linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
def find_optimal_threshold(model,
                           X_snomed_ids_val, X_med_ids_val, X_other_features_val, y_val,
                           config=None):
    """Pick probability threshold. If use_sensitivity_floor=True, maximize specificity subject to
    sensitivity >= sensitivity_floor. Otherwise, maximize weighted (sw*sens + spw*spec)."""
    from sklearn.metrics import roc_curve

    y_val_proba = model.predict(model_inputs(
        config, X_snomed=X_snomed_ids_val, X_med=X_med_ids_val, X_other=X_other_features_val))
    fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)
    sens = tpr
    spec = 1 - fpr

    use_floor = bool(config.get('use_sensitivity_floor', False)) if config else False
    if use_floor:
        floor = float(config.get('sensitivity_floor', 0.95))
        eligible = sens >= floor
        if np.any(eligible):
            eligible_idx = np.where(eligible)[0]
            optimal_idx = eligible_idx[np.argmax(spec[eligible_idx])]
            print(f"Sensitivity-floor threshold (sens >= {floor:.2f}, maximize spec): {thresholds[optimal_idx]:.4f}")
        else:
            optimal_idx = int(np.argmax(sens))
            print(f"No threshold meets sensitivity floor {floor:.2f}. Falling back to max sensitivity.")
    else:
        sw = config.get('threshold_sensitivity_weight', config.get('sensitivity_weight', 2.0)) if config else 2.0
        spw = config.get('threshold_specificity_weight', config.get('specificity_weight', 1.0)) if config else 1.0
        scores = (sw * sens + spw * spec) / (sw + spw)
        optimal_idx = int(np.argmax(scores))
        print(f"Optimal threshold ({sw} * sens + {spw} * spec): {thresholds[optimal_idx]:.4f}")

    optimal_threshold = thresholds[optimal_idx]
    print(f"  Sensitivity: {sens[optimal_idx]:.4f}, Specificity: {spec[optimal_idx]:.4f}")
    return optimal_threshold


def evaluate_model(model, X_snomed_ids_test, X_med_ids_test, X_other_features_test, y_test, config):
    (y_pred_proba, y_pred, accuracy, precision, recall, f1, auc,
     cm, TN, FP, FN, TP, sensitivity, specificity, ppv, npv) = evaluate_model_on_test_data(
        model, X_snomed_ids_test, X_med_ids_test, X_other_features_test, y_test, config)

    show_confusion_matrix(cm, config)
    show_roc_curve(y_pred_proba, y_test, auc, config)


#### Run the Model on a selected Patient

In [ ]:
# Individual-patient prediction with per-branch encoded columns, respecting branch toggles.
def predict_cancer_for_patient(model, patient_id, other_feature_cols,
                               max_snomed_seq_length, max_med_seq_length,
                               df_encoded, config, verbose=True):
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    patient_data = df_encoded[df_encoded['patient_id'] == patient_id]
    if len(patient_data) == 0:
        if verbose:
            print(f"Patient {patient_id} not found in dataset.")
        return None, None, None, None, None

    patient_data = patient_data.sort_values(by='snomed_recorded_date')
    actual_diagnosis = patient_data['has_cancer_diagnosis'].iloc[0]

    has_event_type = 'EVENT_TYPE' in patient_data.columns
    snomed_seq_full = []
    med_seq_full = []
    if use_snomed:
        if has_event_type:
            snomed_seq_full = patient_data[patient_data['EVENT_TYPE'] == 'observation']['snomed_id_encoded'].tolist()
        else:
            snomed_seq_full = patient_data['snomed_id_encoded'].tolist()
    if use_med:
        if has_event_type:
            med_seq_full = patient_data[patient_data['EVENT_TYPE'] == 'medication']['med_id_encoded'].tolist()
        else:
            med_seq_full = patient_data['med_id_encoded'].tolist()

    snomed_seq = snomed_seq_full[-max_snomed_seq_length:] if use_snomed else []
    med_seq = med_seq_full[-max_med_seq_length:] if use_med else []

    X_snomed_ids_patient = (
        pad_sequences([snomed_seq] if len(snomed_seq) > 0 else [[0]],
                      maxlen=max_snomed_seq_length, padding='post', truncating='pre', value=0)
        if use_snomed else None
    )
    X_med_ids_patient = (
        pad_sequences([med_seq] if len(med_seq) > 0 else [[0]],
                      maxlen=max_med_seq_length, padding='post', truncating='pre', value=0)
        if use_med else None
    )

    patient_other_features = patient_data[other_feature_cols].iloc[0].values.astype(np.float32)
    X_other_features_patient = patient_other_features.reshape(1, -1)

    y_pred_proba_patient = float(model.predict(
        model_inputs(config, X_snomed=X_snomed_ids_patient, X_med=X_med_ids_patient,
                     X_other=X_other_features_patient),
        verbose=0
    )[0][0])
    y_pred_patient = int(y_pred_proba_patient > config['proba_threshold'])

    snomed_nonpad = int((X_snomed_ids_patient != 0).sum()) if X_snomed_ids_patient is not None else 0
    med_nonpad = int((X_med_ids_patient != 0).sum()) if X_med_ids_patient is not None else 0

    if verbose:
        print(f"\nPatient {patient_id}")
        print(f"  Actual diagnosis: {actual_diagnosis}")
        print(f"  Predicted label : {y_pred_patient}  |  probability: {y_pred_proba_patient:.4f}"
              f"  |  threshold: {config['proba_threshold']:.4f}")
        if use_snomed:
            print(f"  SNOMED branch   : {snomed_nonpad} non-padding tokens (of {max_snomed_seq_length})"
                  f"  |  raw history: {len(snomed_seq_full)} events"
                  + (f"  -> truncated to last {max_snomed_seq_length}" if len(snomed_seq_full) > max_snomed_seq_length else ""))
        else:
            print(f"  SNOMED branch   : DISABLED")
        if use_med:
            print(f"  MED branch      : {med_nonpad} non-padding tokens (of {max_med_seq_length})"
                  f"  |  raw history: {len(med_seq_full)} events"
                  + (f"  -> truncated to last {max_med_seq_length}" if len(med_seq_full) > max_med_seq_length else ""))
        else:
            print(f"  MED branch      : DISABLED")

    return actual_diagnosis, y_pred_patient, y_pred_proba_patient, snomed_nonpad, med_nonpad


#### Verify individual results on a test subset 

In [ ]:
def verify_results_on_a_subset_of_test_data(model, patient_ids_test,
                                             X_snomed_ids_test, X_med_ids_test, X_other_features_test,
                                             y_test, other_feature_cols,
                                             max_snomed_seq_length, max_med_seq_length,
                                             df_encoded, config):
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    n_subset = min(int(config.get('individual_verification_n_subset', 20)), len(y_test))
    X_snomed_sel = X_snomed_ids_test[:n_subset] if use_snomed else None
    X_med_sel = X_med_ids_test[:n_subset] if use_med else None
    X_other_sel = X_other_features_test[:n_subset]
    y_sel = y_test[:n_subset]

    y_pred_proba_batch = model.predict(model_inputs(
        config, X_snomed=X_snomed_sel, X_med=X_med_sel, X_other=X_other_sel), verbose=0).ravel()
    y_pred = (y_pred_proba_batch > config['proba_threshold']).astype(int)

    accuracy = accuracy_score(y_sel, y_pred)
    precision = precision_score(y_sel, y_pred)
    recall = recall_score(y_sel, y_pred)
    f1 = f1_score(y_sel, y_pred)
    auc = roc_auc_score(y_sel, y_pred_proba_batch)

    cm = confusion_matrix(y_sel, y_pred)
    TN, FP, FN, TP = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

    print(f"\nBatch evaluation on subset ({n_subset} patients):")
    print(f"  Accuracy: {accuracy:.2%}  Precision: {precision:.2%}  Recall: {recall:.2%}")
    print(f"  F1: {f1:.2%}  ROC-AUC: {auc:.2%}")
    print(f"  Sensitivity: {sensitivity:.2%}  Specificity: {specificity:.2%}")
    print(f"  Branches enabled — SNOMED: {use_snomed}  MED: {use_med}")

    # v104: confusion-matrix plot suppressed (metrics above are sufficient)

    c_patient_id_list = patient_ids_test[:n_subset]
    print(f'\nRe-predicting {len(c_patient_id_list)} patients via the per-patient path '
          f'(verifies branch-wise sequence extraction)')

    c_actual_list = []
    c_predicted_list = []
    c_probability_list = []
    c_snomed_nonpad = []
    c_med_nonpad = []

    for patient_id in c_patient_id_list:
        actual, pred, proba, s_np, m_np = predict_cancer_for_patient(
            model, patient_id, other_feature_cols,
            max_snomed_seq_length, max_med_seq_length, df_encoded, config,
            verbose=False)
        if actual is not None:
            c_actual_list.append(actual)
            c_predicted_list.append(pred)
            c_probability_list.append(proba)
            c_snomed_nonpad.append(s_np)
            c_med_nonpad.append(m_np)

    table = {
        "patient_id": c_patient_id_list[:len(c_actual_list)],
        "actual": c_actual_list,
        "predicted": c_predicted_list,
        "probability": np.round(c_probability_list, 4),
    }
    if use_snomed:
        table["snomed_nonpad"] = c_snomed_nonpad
    if use_med:
        table["med_nonpad"] = c_med_nonpad
    check_df = pd.DataFrame(table)

    if len(c_probability_list) == len(y_pred_proba_batch):
        diffs = np.abs(np.array(c_probability_list) - y_pred_proba_batch)
        n_mismatches = int((diffs > 1e-4).sum())
        print(f"  Batch vs per-patient — max |diff|: {diffs.max():.6f}  |  mean |diff|: {diffs.mean():.6f}"
              f"  |  patients differing by > 1e-4: {n_mismatches}/{len(diffs)}")
        if n_mismatches == 0:
            print("  Per-patient extraction matches the batch path — enabled-branch wiring is consistent.")
    else:
        print(f"  (Skipped consistency check: {len(c_probability_list)} per-patient vs {len(y_pred_proba_batch)} batch)")

    if len(check_df):
        print(f"\nBranch coverage across subset ({len(check_df)} patients):")
        if use_snomed:
            print(f"  SNOMED non-padding: mean={np.mean(c_snomed_nonpad):.1f}  min={min(c_snomed_nonpad)}  max={max(c_snomed_nonpad)}")
        if use_med:
            print(f"  MED    non-padding: mean={np.mean(c_med_nonpad):.1f}  min={min(c_med_nonpad)}  max={max(c_med_nonpad)}  "
                  f"|  patients w/ 0 meds: {int(np.sum(np.array(c_med_nonpad) == 0))}")

    pd.set_option('display.max_rows', 100)
    display(check_df.head(100))

    for p, q in [(0, 0), (1, 1), (0, 1), (1, 0)]:
        subset = check_df[(check_df['actual'] == p) & (check_df['predicted'] == q)]
        print(f"\nActual={p}, Predicted={q}: {len(subset)} patients")
        display(subset)


#### Demographics - Train and Test patients


In [ ]:
# Demographics percentage breakdown for train and test patients

def demographic_breakdown(patient_ids, df):
    sub = df[df['patient_id'].isin(patient_ids)].copy()

    # one row per patient
    patient_level = sub.sort_values('snomed_recorded_date').groupby('patient_id').last().reset_index()

    # Ethnicity %
    ethnicity_pct = (patient_level['ethnicity']
                     .value_counts(normalize=True)
                     .sort_index()*100)

    # Gender %
    gender_pct = (patient_level['gender']
                  .value_counts(normalize=True)
                  .sort_index()*100)

    # Age groups
    age = patient_level['age_at_last_snomed_record']
    bins = [0,40,50,60,70,120]
    labels = ['<40','40-49','50-59','60-69','70+']
    age_group = pd.cut(age, bins=bins, labels=labels, right=False)

    age_pct = (age_group.value_counts(normalize=True)
               .sort_index()*100)

    return ethnicity_pct, gender_pct, age_pct

# Demographics breakdown as DataFrames
def series_to_demo_df(series, category, value_name):
    s = series.rename(value_name).reset_index()
    group_col = s.columns[0]
    return s.rename(columns={group_col: 'Group'}).assign(Category=category)[['Category', 'Group', value_name]]

def show_train_test_demographics(df, patient_ids_train, patient_ids_test, config):
    eth_train, gen_train, age_train = demographic_breakdown(patient_ids_train, df)
    eth_test, gen_test, age_test = demographic_breakdown(patient_ids_test, df)
    
    train_demographics_df = pd.concat(
    [
        series_to_demo_df(eth_train, 'Ethnicity', 'Train %'),
        series_to_demo_df(gen_train, 'Gender', 'Train %'),
        series_to_demo_df(age_train, 'Age', 'Train %'),
    ],
    ignore_index=True
    )

    test_demographics_df = pd.concat(
        [
            series_to_demo_df(eth_test, 'Ethnicity', 'Test %'),
            series_to_demo_df(gen_test, 'Gender', 'Test %'),
            series_to_demo_df(age_test, 'Age', 'Test %'),
        ],
        ignore_index=True
    )

    combined_demographics_df = train_demographics_df.merge(
        test_demographics_df,
        on=['Category', 'Group'],
        how='outer'
    ).sort_values(['Category', 'Group']).reset_index(drop=True)

    train_demographics_df['Train %'] = train_demographics_df['Train %'].round(2)
    test_demographics_df['Test %'] = test_demographics_df['Test %'].round(2)
    combined_demographics_df[['Train %', 'Test %']] = combined_demographics_df[['Train %', 'Test %']].round(2)
    
    # Combined demographics dataframe with blanks for 0/NaN and Gender shown as M/F
    combined_demographics_display_df = combined_demographics_df.copy()

    # Standardize gender labels to M/F where possible
    gender_map = {
        'male': 'M', 'm': 'M',
        'female': 'F', 'f': 'F'
    }

    gender_mask = combined_demographics_display_df['Category'].eq('Gender')
    combined_demographics_display_df.loc[gender_mask, 'Group'] = (
        combined_demographics_display_df.loc[gender_mask, 'Group']
        .astype(str)
        .str.strip()
        .str.lower()
        .map(gender_map)
        .fillna(combined_demographics_display_df.loc[gender_mask, 'Group'])
    )

    # Show 0 values and NaN as blanks for display
    for col in ['Train %', 'Test %']:
        combined_demographics_display_df[col] = combined_demographics_display_df[col].apply(
            lambda x: '' if pd.isna(x) or x == 0 else f'{x:.2f}'
        )

    print("Combined demographics dataframe (formatted)")
    display(combined_demographics_display_df)

### Explainability


In [ ]:
# Explainability — IG on static features + TimeSHAP-style temporal ablation for enabled
# branches. Adds tqdm progress bars on the hot loops and suppresses repeated TF C++ stderr
# warnings (e.g. "NodeDef mentions attribute use_unbounded_threadpool ...") that spam the
# output during many small model.predict calls.
import tensorflow as tf
import os
import sys
import warnings
from contextlib import contextmanager
from collections import defaultdict

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):  # graceful fallback if tqdm isn't installed
        return iterable if iterable is not None else range(0)


@contextmanager
def _suppress_tf_cpp_stderr():
    """Redirect fd 2 (stderr) to /dev/null for the duration of the block.

    This is the only reliable way to silence the TF C++ LOG(ERROR) messages (e.g. the
    `node_def_util.cc: NodeDef mentions attribute use_unbounded_threadpool ...` warning)
    after TF has been imported — `tf.get_logger().setLevel(...)` and
    `TF_CPP_MIN_LOG_LEVEL` have no effect at that point. We scope it as tightly as possible
    (around single model.predict calls) so legitimate Python-level tracebacks still surface.
    """
    try:
        sys.stderr.flush()
    except Exception:
        pass
    saved_fd = os.dup(2)
    try:
        with open(os.devnull, 'w') as devnull:
            os.dup2(devnull.fileno(), 2)
            try:
                yield
            finally:
                os.dup2(saved_fd, 2)
    finally:
        os.close(saved_fd)


# Silence the benign Keras optimizer-variable-count UserWarning that fires on model load.
warnings.filterwarnings('ignore', message='.*Skipping variable loading for optimizer.*')


def predict_proba(model, X_snomed_ids, X_med_ids, X_other_features, config, batch_size=512):
    with _suppress_tf_cpp_stderr():
        preds = model.predict(
            model_inputs(config, X_snomed=X_snomed_ids, X_med=X_med_ids, X_other=X_other_features),
            batch_size=batch_size, verbose=0)
    return np.asarray(preds).reshape(-1)


@tf.function
def _predict_tf_branches(model, snomed_ids, med_ids, other_features, use_snomed, use_med):
    feed = {'other_features_input': other_features}
    if use_snomed:
        feed['snomed_id_input'] = snomed_ids
    if use_med:
        feed['med_id_input'] = med_ids
    out = model(feed, training=False)
    return tf.reshape(out, (-1,))


def compute_healthy_baseline_other(X_other_train, y_train):
    X = np.asarray(X_other_train).astype(np.float32)
    y = np.asarray(y_train).reshape(-1)
    healthy = X[y == 0]
    if len(healthy) == 0:
        raise ValueError('No healthy (y==0) samples found.')
    return np.mean(healthy, axis=0, keepdims=True)


def compute_healthy_baseline_codes(X_codes_train, y_train, pad_value=0, branch_name='codes'):
    X = np.asarray(X_codes_train)
    y = np.asarray(y_train).reshape(-1)
    Xh = X[y == 0]
    if len(Xh) == 0:
        raise ValueError('No healthy (y==0) samples found.')
    T = X.shape[1]
    baseline = np.full((T,), int(pad_value), dtype=np.int32)
    for t in tqdm(range(T), desc=f'Healthy baseline ({branch_name})', leave=False):
        col = Xh[:, t]
        col = col[col != pad_value]
        if col.size > 0:
            vals, counts = np.unique(col, return_counts=True)
            baseline[t] = int(vals[np.argmax(counts)])
    return baseline[None, :]


def compute_training_baseline_other(X_other_train):
    """SHAP-style unconditional baseline for static features — mean across *all*
    training patients, not restricted to y==0. This is the single-point analogue
    to `shap.sample(X_train, 100).mean(axis=0)` for a standard SHAP explainer.
    Use `config['explainability_baseline_type'] = 'shap_training'` to select it."""
    X = np.asarray(X_other_train).astype(np.float32)
    if len(X) == 0:
        raise ValueError('No training samples found.')
    return np.mean(X, axis=0, keepdims=True)


def compute_training_baseline_codes(X_codes_train, pad_value=0, branch_name='codes'):
    """Unconditional per-position mode of non-pad codes across *all* training
    patients. Used as the single-point baseline for SHAP-style temporal ablation
    when explainability_baseline_type='shap_eg' (same as v105s's sequence baseline
    — we don't do K-sample ablation because it's expensive and rarely re-ranks
    top concepts; use 'shap_eg' to get the real SHAP behaviour on static features,
    which is where the method most diverges from IG)."""
    X = np.asarray(X_codes_train)
    if len(X) == 0:
        raise ValueError('No training samples found.')
    T = X.shape[1]
    baseline = np.full((T,), int(pad_value), dtype=np.int32)
    for t in tqdm(range(T), desc=f'Training baseline ({branch_name})', leave=False):
        col = X[:, t]
        col = col[col != pad_value]
        if col.size > 0:
            vals, counts = np.unique(col, return_counts=True)
            baseline[t] = int(vals[np.argmax(counts)])
    return baseline[None, :]


def build_shap_background_other(X_other_train, size=100, seed=42):
    """SHAP background for static features — `size` randomly sampled training rows,
    unconditional on class. Mirrors `shap.sample(X_train, 100)` from the SHAP docs."""
    X = np.asarray(X_other_train).astype(np.float32)
    n = min(int(size), len(X))
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(len(X), size=n, replace=False)
    return X[idx]  # (n, F)


def expected_gradients_other_features(model, X_snomed, X_med, X_other, config,
                                       background_other, n_samples=100, seed=0):
    """Expected Gradients (SHAP.GradientExplainer equivalent, hand-rolled in TF).

      phi_j ≈ mean_k [ (x_j - b_k_j) * df/dx_j at (b_k + alpha_k * (x - b_k)) ]

    with (b_k, alpha_k) sampled from (background_other, Uniform[0,1]). One batched
    GradientTape call covers all K samples per patient — cost is similar to an
    n_samples-step IG on a single baseline."""
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    X_other_tf = tf.convert_to_tensor(X_other, dtype=tf.float32)      # (1, F)
    bg = tf.convert_to_tensor(background_other, dtype=tf.float32)      # (N_bg, F)
    N_bg = int(bg.shape[0])
    if N_bg == 0:
        raise ValueError('Empty SHAP background.')

    rng = np.random.default_rng(int(seed))
    bg_idx = rng.integers(0, N_bg, size=int(n_samples))
    alphas = rng.uniform(0.0, 1.0, size=int(n_samples)).astype(np.float32)

    b_samples = tf.gather(bg, bg_idx)                                  # (K, F)
    alphas_tf = tf.constant(alphas.reshape(-1, 1))                     # (K, 1)
    x_tiled = tf.tile(X_other_tf, [int(n_samples), 1])                 # (K, F)
    interpolated = b_samples + alphas_tf * (x_tiled - b_samples)       # (K, F)

    if use_snomed:
        snomed_rep = tf.tile(tf.convert_to_tensor(X_snomed, dtype=tf.int32),
                             [int(n_samples), 1])
    else:
        snomed_rep = tf.zeros((int(n_samples), 1), dtype=tf.int32)
    if use_med:
        med_rep = tf.tile(tf.convert_to_tensor(X_med, dtype=tf.int32),
                          [int(n_samples), 1])
    else:
        med_rep = tf.zeros((int(n_samples), 1), dtype=tf.int32)

    with _suppress_tf_cpp_stderr():
        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            preds = _predict_tf_branches(model, snomed_rep, med_rep, interpolated, use_snomed, use_med)
        grads = tape.gradient(preds, interpolated)                     # (K, F)

    x_minus_b = x_tiled - b_samples                                    # (K, F)
    per_sample_attr = x_minus_b * grads                                # (K, F)
    eg = tf.reduce_mean(per_sample_attr, axis=0)                       # (F,)
    return eg.numpy()


def expected_gradients_batched(model, X_snomed_all, X_med_all, X_other_all, config,
                                background_other, n_samples=100,
                                patient_chunk_size=32, seed_base=0,
                                show_progress=True):
    """Vectorised Expected Gradients across many patients.

    Instead of one `tape.gradient` call per patient, coalesce P = patient_chunk_size
    patients × K = n_samples MC samples into one batched call of shape (P*K, F).
    Amortises the TF tape + graph-trace overhead P×. Mathematically identical to
    calling `expected_gradients_other_features` once per patient (same sampling
    distribution, same (x - b_k) * grad_k formula).

    Returns:
        eg_all: (N_patients, F) per-patient EG attributions (float32).
    """
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    N = int(X_other_all.shape[0])
    F = int(X_other_all.shape[1])
    K = int(n_samples)
    P = max(1, int(patient_chunk_size))

    bg = tf.convert_to_tensor(background_other, dtype=tf.float32)         # (N_bg, F)
    N_bg = int(bg.shape[0])
    if N_bg == 0:
        raise ValueError('Empty SHAP background.')

    T_s = int(X_snomed_all.shape[1]) if use_snomed and X_snomed_all is not None else 1
    T_m = int(X_med_all.shape[1]) if use_med and X_med_all is not None else 1

    eg_all = np.zeros((N, F), dtype=np.float32)
    chunks = list(range(0, N, P))
    iterator = (tqdm(chunks, desc=f'EG batched (P={P}, K={K})', leave=True)
                if show_progress else chunks)

    for start in iterator:
        end = min(start + P, N)
        P_eff = end - start

        # Sample (baseline_idx, alpha) per (patient_in_chunk, k).
        rng = np.random.default_rng(int(seed_base) + start)
        bg_idx = rng.integers(0, N_bg, size=(P_eff, K))                    # (P, K)
        alphas = rng.uniform(0.0, 1.0, size=(P_eff, K)).astype(np.float32) # (P, K)

        # Gather baselines as (P, K, F) via numpy advanced indexing.
        b_samples_np = np.asarray(background_other, dtype=np.float32)[bg_idx]  # (P, K, F)
        b_samples = tf.convert_to_tensor(b_samples_np, dtype=tf.float32)

        # Tile each patient's x across K samples.
        Xo_np = np.asarray(X_other_all[start:end], dtype=np.float32)       # (P, F)
        x_expanded = tf.expand_dims(tf.convert_to_tensor(Xo_np), 1)        # (P, 1, F)
        x_tiled = tf.tile(x_expanded, [1, K, 1])                           # (P, K, F)

        alphas_tf = tf.constant(alphas[:, :, None])                        # (P, K, 1)
        interpolated = b_samples + alphas_tf * (x_tiled - b_samples)       # (P, K, F)

        # Flatten to (P*K, F) — the tensor tape watches.
        interpolated_flat = tf.reshape(interpolated, [P_eff * K, F])

        # Tile SNOMED and MED sequences per patient × K samples.
        if use_snomed and X_snomed_all is not None:
            Xs_np = np.asarray(X_snomed_all[start:end], dtype=np.int32)
            Xs_exp = tf.expand_dims(tf.convert_to_tensor(Xs_np), 1)
            Xs_tiled = tf.tile(Xs_exp, [1, K, 1])
            Xs_flat = tf.reshape(Xs_tiled, [P_eff * K, T_s])
        else:
            Xs_flat = tf.zeros((P_eff * K, 1), dtype=tf.int32)
        if use_med and X_med_all is not None:
            Xm_np = np.asarray(X_med_all[start:end], dtype=np.int32)
            Xm_exp = tf.expand_dims(tf.convert_to_tensor(Xm_np), 1)
            Xm_tiled = tf.tile(Xm_exp, [1, K, 1])
            Xm_flat = tf.reshape(Xm_tiled, [P_eff * K, T_m])
        else:
            Xm_flat = tf.zeros((P_eff * K, 1), dtype=tf.int32)

        with _suppress_tf_cpp_stderr():
            with tf.GradientTape() as tape:
                tape.watch(interpolated_flat)
                preds = _predict_tf_branches(
                    model, Xs_flat, Xm_flat, interpolated_flat, use_snomed, use_med)
            grads_flat = tape.gradient(preds, interpolated_flat)           # (P*K, F)

        grads = tf.reshape(grads_flat, [P_eff, K, F])
        x_minus_b = x_tiled - b_samples                                    # (P, K, F)
        per_sample_attr = x_minus_b * grads
        eg_chunk = tf.reduce_mean(per_sample_attr, axis=1)                 # (P, F)
        eg_all[start:end] = eg_chunk.numpy()

    return eg_all


def integrated_gradients_other_features(model, X_snomed, X_med, X_other, config,
                                        baseline_other=None, m_steps=50):
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    X_other_tf = tf.convert_to_tensor(X_other, dtype=tf.float32)
    if baseline_other is None:
        baseline_other = tf.zeros_like(X_other_tf)
    else:
        baseline_other = tf.convert_to_tensor(baseline_other, dtype=tf.float32)

    alphas = tf.linspace(0.0, 1.0, m_steps + 1)
    alphas_x = tf.reshape(alphas, (-1, 1))
    baseline_rep = tf.repeat(baseline_other, repeats=m_steps + 1, axis=0)
    input_rep = tf.repeat(X_other_tf, repeats=m_steps + 1, axis=0)
    interpolated = baseline_rep + alphas_x * (input_rep - baseline_rep)

    if use_snomed:
        snomed_rep = tf.repeat(tf.convert_to_tensor(X_snomed, dtype=tf.int32), repeats=m_steps + 1, axis=0)
    else:
        snomed_rep = tf.zeros((m_steps + 1, 1), dtype=tf.int32)
    if use_med:
        med_rep = tf.repeat(tf.convert_to_tensor(X_med, dtype=tf.int32), repeats=m_steps + 1, axis=0)
    else:
        med_rep = tf.zeros((m_steps + 1, 1), dtype=tf.int32)

    with _suppress_tf_cpp_stderr():
        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            preds = _predict_tf_branches(model, snomed_rep, med_rep, interpolated, use_snomed, use_med)
        grads = tape.gradient(preds, interpolated)
    grads = (grads[:-1] + grads[1:]) / 2.0
    avg_grads = tf.reduce_mean(grads, axis=0)
    ig = (X_other_tf[0] - baseline_other[0]) * avg_grads
    return ig.numpy()


def temporal_ablation_batch(model, X_snomed, X_med, X_other, branch, baseline_codes, config,
                             predict_batch_size=1024, chunk_patients=32, show_progress=True):
    """Batched temporal ablation on one branch. Each iteration of the progress bar is one
    coalesced (M*T)-variant predict call for `chunk_patients` patients."""
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    if branch == 'snomed':
        if not use_snomed:
            raise ValueError("Cannot ablate SNOMED — use_snomed_branch is disabled.")
        T = X_snomed.shape[1]
    elif branch == 'med':
        if not use_med:
            raise ValueError("Cannot ablate MED — use_med_branch is disabled.")
        T = X_med.shape[1]
    else:
        raise ValueError(f"branch must be 'snomed' or 'med', got {branch!r}")

    X_other = np.asarray(X_other)
    N = X_other.shape[0]

    baseline = np.asarray(baseline_codes)
    per_patient_baseline = None  # populated if baseline has shape (N, T)
    if baseline.ndim == 0:
        baseline_row = np.full(T, int(baseline), dtype=np.int32)
    elif baseline.ndim == 1:
        baseline_row = baseline.astype(np.int32)
    elif baseline.ndim == 2 and baseline.shape[0] == N and baseline.shape[1] == T:
        # v106 fix: per-patient baseline sequence (SHAP marginal_sample mode)
        per_patient_baseline = baseline.astype(np.int32)
        baseline_row = None  # not used in this branch
    else:
        baseline_row = baseline.ravel().astype(np.int32)
        if baseline_row.size != T:
            baseline_row = np.resize(baseline_row, T)

    orig_preds = predict_proba(
        model,
        X_snomed if use_snomed else None,
        X_med if use_med else None,
        X_other, config, batch_size=predict_batch_size)

    deltas = np.zeros((N, T), dtype=np.float32)

    chunks = list(range(0, N, chunk_patients))
    iterator = (tqdm(chunks, desc=f'Ablating {branch.upper()} (N={N}, T={T})', leave=False)
                if show_progress else chunks)

    for start in iterator:
        end = min(start + chunk_patients, N)
        M = end - start

        Xo_rep = np.repeat(X_other[start:end], T, axis=0)

        Xs_rep = np.repeat(X_snomed[start:end], T, axis=0) if use_snomed else None
        Xm_rep = np.repeat(X_med[start:end], T, axis=0) if use_med else None

        row_idx = np.arange(M * T)
        col_idx = np.tile(np.arange(T), M)
        if per_patient_baseline is not None:
            # Per-patient baseline: replace position t in patient p with that patient's
            # sampled baseline at the same position (SHAP marginal_sample mode).
            chunk_bl = per_patient_baseline[start:end]                   # (M, T)
            row_bl = np.repeat(chunk_bl, T, axis=0)                      # (M*T, T)
            fill_vals = row_bl[row_idx, col_idx]                          # (M*T,)
            if branch == 'snomed':
                Xs_rep[row_idx, col_idx] = fill_vals
            else:
                Xm_rep[row_idx, col_idx] = fill_vals
        elif branch == 'snomed':
            Xs_rep[row_idx, col_idx] = baseline_row[col_idx]
        else:
            Xm_rep[row_idx, col_idx] = baseline_row[col_idx]

        preds_masked = predict_proba(model, Xs_rep, Xm_rep, Xo_rep, config, batch_size=predict_batch_size)
        preds_masked = preds_masked.reshape(M, T)
        deltas[start:end] = orig_preds[start:end, None] - preds_masked

    return orig_preds, deltas


def temporal_ablation_snomed(model, X_snomed_1, X_med_1, X_other_1, config, baseline_snomed=0):
    orig, deltas = temporal_ablation_batch(
        model, X_snomed_1, X_med_1, X_other_1,
        branch='snomed', baseline_codes=baseline_snomed, config=config,
        predict_batch_size=512, chunk_patients=1, show_progress=False)
    return float(orig[0]), deltas[0]


def temporal_ablation_med(model, X_snomed_1, X_med_1, X_other_1, config, baseline_med=0):
    orig, deltas = temporal_ablation_batch(
        model, X_snomed_1, X_med_1, X_other_1,
        branch='med', baseline_codes=baseline_med, config=config,
        predict_batch_size=512, chunk_patients=1, show_progress=False)
    return float(orig[0]), deltas[0]


# v143: feature-level ablation for STATIC features. Mirrors temporal_ablation_batch
# but masks one static feature at a time (replacing it with the baseline value)
# instead of one timestep in a sequence branch. Returns deltas in the same
# probability-space units as temporal_ablation_batch, so static and SNOMED
# attributions can be compared/plotted on the same scale.
#
# Note: ablation deltas are NOT additive (masking one feature isn't independent
# of the others), so sum(delta_i) != f(x) - f(baseline) in general. That's the
# tradeoff vs IG, which is additive by construction (completeness axiom) but
# uses gradient-based attributions that aren't directly comparable to ablation.
def static_feature_ablation_batch(model, X_snomed, X_med, X_other, baseline_other,
                                    config, predict_batch_size=1024,
                                    chunk_patients=32, show_progress=True):
    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    X_other = np.asarray(X_other, dtype=np.float32)
    N, F = X_other.shape
    baseline_row = np.asarray(baseline_other, dtype=np.float32).reshape(-1)
    if baseline_row.size != F:
        raise ValueError(f'baseline_other must have {F} values, got {baseline_row.size}')

    orig_preds = predict_proba(
        model,
        X_snomed if use_snomed else None,
        X_med if use_med else None,
        X_other, config, batch_size=predict_batch_size)

    deltas = np.zeros((N, F), dtype=np.float32)
    chunks = list(range(0, N, chunk_patients))
    iterator = (tqdm(chunks, desc=f'Ablating STATIC (N={N}, F={F})', leave=False)
                if show_progress else chunks)

    for start in iterator:
        end = min(start + chunk_patients, N)
        M = end - start
        Xo_rep = np.repeat(X_other[start:end], F, axis=0)
        row_idx = np.arange(M * F)
        col_idx = np.tile(np.arange(F), M)
        Xo_rep[row_idx, col_idx] = baseline_row[col_idx]

        Xs_rep = np.repeat(X_snomed[start:end], F, axis=0) if use_snomed else None
        Xm_rep = np.repeat(X_med[start:end], F, axis=0) if use_med else None

        preds_masked = predict_proba(model, Xs_rep, Xm_rep, Xo_rep, config,
                                       batch_size=predict_batch_size)
        preds_masked = preds_masked.reshape(M, F)
        deltas[start:end] = orig_preds[start:end, None] - preds_masked

    return orig_preds, deltas


def static_feature_ablation_single(model, X_snomed_1, X_med_1, X_other_1, config,
                                     baseline_other):
    orig, deltas = static_feature_ablation_batch(
        model, X_snomed_1, X_med_1, X_other_1,
        baseline_other=baseline_other, config=config,
        predict_batch_size=512, chunk_patients=1, show_progress=False)
    return float(orig[0]), deltas[0]


# v147: collapse one-hot static groups for display. Operates on
# (column_name, signed_delta) pairs. Default rules:
#   'gender_*'                      -> 'gender'  (aggregated, mean signed delta)
#   'age_at_last_snomed_record'     -> 'age'     (renamed, no aggregation needed
#                                                  since it's a single column)
# Other columns pass through unchanged. Used by the explainability charts so
# the report shows one row per logical static concept rather than one per
# one-hot column. Not used inside ablation itself — the ablation still acts
# on each raw column independently; only the chart presentation is collapsed.
def _aggregate_static_features(items_with_col, agg_method='mean'):
    rename_map = {'age_at_last_snomed_record': 'age'}
    group_prefixes = {'gender_': 'gender'}
    sums = {}
    counts = {}
    for col, val in items_with_col:
        if col in rename_map:
            disp = rename_map[col]
        else:
            disp = col
            for prefix, gname in group_prefixes.items():
                if col.startswith(prefix):
                    disp = gname
                    break
        sums[disp] = sums.get(disp, 0.0) + float(val)
        counts[disp] = counts.get(disp, 0) + 1
    if agg_method == 'mean':
        return [(k, sums[k] / counts[k]) for k in sums]
    return list(sums.items())


def _build_int_to_text(code_to_int, df, event_type, code_column, text_column):
    int_to_code = {v: k for k, v in code_to_int.items() if k != 'PAD_VALUE'}
    int_to_code[0] = 'PAD'

    if 'EVENT_TYPE' in df.columns and event_type is not None:
        df_sub = df[df['EVENT_TYPE'] == event_type]
    else:
        df_sub = df
    if code_column not in df_sub.columns:
        code_to_text = {}
    else:
        df_sub = df_sub.dropna(subset=[code_column])
        code_to_text = df_sub.groupby(df_sub[code_column].astype(str))[text_column].first().to_dict()

    int_to_text = {}
    for i, code in int_to_code.items():
        if code == 'PAD':
            int_to_text[i] = '[PAD]'
        elif code == '<RARE>':
            int_to_text[i] = '<RARE>'
        else:
            int_to_text[i] = str(code_to_text.get(str(code), code))
    return int_to_text


def _plot_barh_concepts(title, items, xlabel):
    import matplotlib.pyplot as plt
    if not items:
        print(f'(no concepts to plot for "{title}")')
        return
    labels = [x[0] for x in items][::-1]
    values = [x[1] for x in items][::-1]
    plt.figure(figsize=(10, max(4, 0.3 * len(items) + 1)))
    plt.barh(range(len(items)), values)
    plt.yticks(range(len(items)), labels, fontsize=8)
    plt.axvline(0, linewidth=1)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.show()



# ==================== EXPLAINABILITY RATING ====================
# Domain-relevant (breast-cancer in this notebook) SNOMED-text regex patterns
# used by the automated clinical-relevance score. Override via
# config['relevance_patterns'] if needed.
# RELEVANCE_PATTERNS fallback — the real list is provided per-domain via
# config['relevance_patterns'] (see CANCER_PROFILES in the config cell).
# Left empty here so a missing config entry doesn't silently match anything.
RELEVANCE_PATTERNS = []


# --- Authoritative SNOMED codelist loader (OpenCodelists-sourced) ---
# Expects CSVs with columns `id,name,active,notes` (standard OpenCodelists SNOMED
# export). Rows with non-digit IDs (CTV3 etc.) and inactive rows are skipped.
_CONCEPT_CACHE = {}


def _load_concept_ids(paths):
    """Load authoritative SNOMED concept IDs from one or more OpenCodelists CSVs.
    Returns (id_set, {id: name}). Missing files print a warning and are skipped."""
    import csv
    import os
    ids = set()
    id_to_name = {}
    loaded_msgs = []
    for p in paths:
        if not os.path.exists(p):
            print(f'  [concept codelist not found: {p} — skipping]')
            continue
        try:
            with open(p, newline='', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                n_kept = 0
                for row in reader:
                    sid = (row.get('id') or '').strip()
                    if not sid or not sid.isdigit():
                        continue  # skip CTV3/Read rows
                    active = str(row.get('active', 'y')).strip().lower()
                    if active in ('n', 'no', '0', 'false'):
                        continue
                    ids.add(sid)
                    id_to_name.setdefault(sid, (row.get('name') or '').strip())
                    n_kept += 1
            loaded_msgs.append(f'{os.path.basename(p)} ({n_kept})')
        except Exception as e:
            print(f'  [failed to load codelist {p}: {e!r}]')
    if loaded_msgs:
        print(f'  Relevance codelists loaded: {", ".join(loaded_msgs)} '
              f'— {len(ids)} unique SNOMED IDs')
    return ids, id_to_name


def _load_concept_ids_cached(paths):
    key = tuple(paths)
    if key not in _CONCEPT_CACHE:
        _CONCEPT_CACHE[key] = _load_concept_ids(paths)
    return _CONCEPT_CACHE[key]



# v113 Lever 4: semantic-relevance scorer (optional).
# When config['use_semantic_relevance'] is True, concepts that don't match the
# authoritative codelist OR regex get a last-chance check against a biomedical
# sentence-embedding cosine similarity to a domain (breast-cancer) reference. Requires
# sentence-transformers; falls back to no-semantic-matches if not installed.
_SEMANTIC_SCORER_CACHE = {}  # key: (model_name, tuple(reference_phrases)) -> 'failed' | callable


def _get_semantic_scorer(model_name='sentence-transformers/all-MiniLM-L6-v2',
                          threshold=0.5, reference_phrases=None, verbose=True):
    if reference_phrases is None or len(reference_phrases) == 0:
        if verbose:
            print('  [semantic relevance] no reference_phrases supplied — '
                  'skipping semantic tier (provide config["semantic_reference_phrases"]).')
        return None
    cache_key = (model_name, tuple(reference_phrases), float(threshold))
    cached = _SEMANTIC_SCORER_CACHE.get(cache_key)
    if cached == 'failed':
        return None
    if callable(cached):
        return cached
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as _imp_err:
        if verbose:
            print(f'  [semantic relevance] sentence-transformers not importable: {_imp_err!r}')
            print('  [semantic relevance] pip install sentence-transformers to enable; skipping.')
        _SEMANTIC_SCORER_CACHE[cache_key] = 'failed'
        return None
    try:
        if verbose:
            print(f'  [semantic relevance] loading {model_name!r} '
                  f'(first use downloads ~80 MB)...')
        model = SentenceTransformer(model_name)
    except Exception as _load_err:
        if verbose:
            print(f'  [semantic relevance] model load failed: {_load_err!r}')
        _SEMANTIC_SCORER_CACHE[cache_key] = 'failed'
        return None
    ref_emb = model.encode(list(reference_phrases), normalize_embeddings=True).mean(axis=0)
    ref_norm = float(np.linalg.norm(ref_emb))
    if ref_norm > 0:
        ref_emb = ref_emb / ref_norm

    def _scorer(concept_names, sim_threshold=threshold):
        if not concept_names:
            return []
        embs = model.encode(list(concept_names), normalize_embeddings=True)
        sims = embs @ ref_emb
        return [(float(s), bool(s >= sim_threshold)) for s in sims]

    _SEMANTIC_SCORER_CACHE[cache_key] = _scorer
    return _scorer


def _faithfulness_aopc(model, X_snomed_sample, X_other_sample, snomed_deltas_all,
                       orig_preds_all, baseline_codes, config,
                       n_patients=100, k_values=(1, 5, 10, 20), seed=0, X_med_sample=None):
    """Faithfulness via AOPC / MoRF: when we cumulatively ablate the top-k high-|delta|
    timesteps, the prediction should drop MORE than when we ablate random-k timesteps.
    One batched predict covers all (patient, ordering, k) variants.
    Returns a [0,1] score plus the raw drops so the user can audit."""
    N, T = X_snomed_sample.shape
    n = min(int(n_patients), N)
    if n <= 0:
        return {'score': 0.0, 'note': 'no patients', 'n_patients': 0}
    pick = np.random.default_rng(seed).choice(N, size=n, replace=False)

    Xs = X_snomed_sample[pick]
    Xo = X_other_sample[pick]
    d = np.abs(snomed_deltas_all[pick])
    orig = orig_preds_all[pick]

    baseline = np.asarray(baseline_codes)
    if baseline.ndim == 0:
        baseline_row = np.full(T, int(baseline), dtype=np.int32)
    elif baseline.ndim == 1:
        baseline_row = baseline.astype(np.int32)
    else:
        baseline_row = baseline.ravel().astype(np.int32)
        if baseline_row.size != T:
            baseline_row = np.resize(baseline_row, T)

    sorted_idx = np.argsort(-d, axis=1)
    rng_r = np.random.default_rng(seed + 1)
    rand_idx = np.stack([rng_r.permutation(T) for _ in range(n)])

    k_values = tuple(int(k) for k in k_values if 1 <= int(k) <= T)
    if not k_values:
        return {'score': 0.0, 'note': 'no valid k values', 'n_patients': n}

    variants_Xs, variants_Xm, variants_Xo = [], [], []
    for k in k_values:
        for indices in (sorted_idx, rand_idx):
            Xs_k = Xs.copy()
            cols = indices[:, :k]
            rows_broadcast = np.arange(n)[:, None]
            Xs_k[rows_broadcast, cols] = baseline_row[cols]
            variants_Xs.append(Xs_k); variants_Xo.append(Xo)
            if X_med_sample is not None:
                variants_Xm.append(X_med_sample[pick])
    Xs_all = np.concatenate(variants_Xs, axis=0)
    Xo_all = np.concatenate(variants_Xo, axis=0)
    Xm_all = np.concatenate(variants_Xm, axis=0) if variants_Xm else None

    bs = int(config.get('explainability_predict_batch_size', 1024))
    preds = predict_proba(model, Xs_all, Xm_all, Xo_all, config, batch_size=bs)
    preds = preds.reshape(len(k_values), 2, n)
    drop_top = (orig[None, :] - preds[:, 0, :]).mean(axis=1)
    drop_rand = (orig[None, :] - preds[:, 1, :]).mean(axis=1)
    aopc_top = float(np.mean(drop_top))
    aopc_rand = float(np.mean(drop_rand))
    score = 0.0 if aopc_top <= 1e-6 else float(np.clip((aopc_top - aopc_rand) / aopc_top, 0.0, 1.0))

    return {
        'score': score,
        'aopc_top_mean_drop': aopc_top,
        'aopc_random_mean_drop': aopc_rand,
        'drop_top_by_k': {int(k): float(v) for k, v in zip(k_values, drop_top)},
        'drop_rand_by_k': {int(k): float(v) for k, v in zip(k_values, drop_rand)},
        'n_patients': n,
        'k_values': list(k_values),
    }


def _clinical_relevance_score(top_concepts, patterns, top_k=10,
                               authoritative_id_set=None, concept_to_snomed_id=None,
                               semantic_scorer=None):
    """Fraction of top-k global concepts that are domain-relevant.

    Matching priority per concept:
      1. If the concept's SNOMED ID (via concept_to_snomed_id) is in
         authoritative_id_set -> matched as 'auth'.
      2. Else if its text matches any regex in patterns -> matched as 'regex'.
      3. Else -> unmatched.

    Score = 0.5 * unweighted matched-fraction + 0.5 * weighted-by-|delta|
    matched-fraction. A concept counts once (auth takes priority over regex).
    """
    import re
    top = top_concepts[:top_k]
    if not top:
        return {'score': 0.0, 'unweighted_fraction': 0.0, 'weighted_fraction': 0.0,
                'matched_count': 0, 'matched_count_authoritative': 0,
                'matched_count_regex': 0, 'total': 0,
                'matched_labels': [], 'matched_labels_authoritative': [],
                'matched_labels_regex': [], 'unmatched_labels': []}
    pat = re.compile('|'.join(patterns), re.IGNORECASE) if patterns else None
    id_set = authoritative_id_set or set()
    c2id = concept_to_snomed_id or {}

    matched_auth = []
    matched_regex = []
    matched_sem = []
    unmatched = []
    sem_sim_by_concept = {}
    total_w = 0.0
    match_w = 0.0
    # v146: weights can now be SIGNED (mean signed delta from _aggregate).
    # Use |w| for the weighted-fraction so positive/negative contributions
    # don't cancel and reduce the denominator to ~0. Importance is magnitude.
    # First pass: classify each concept into auth/regex/residual.
    residual = []
    stash = []
    for text, w in top:
        aw = abs(float(w))
        total_w += aw
        sid = c2id.get(text, '')
        if sid and sid in id_set:
            stash.append((text, aw, 'auth'))
        elif pat and pat.search(text):
            stash.append((text, aw, 'regex'))
        else:
            stash.append((text, aw, None))
            residual.append(text)
    # Batch semantic scoring on the residual only.
    sem_results = {}
    if semantic_scorer and residual:
        scored = semantic_scorer(residual)
        for text, (sim, is_match) in zip(residual, scored):
            sem_results[text] = (sim, is_match)
            sem_sim_by_concept[text] = sim
    # Second pass: tally.
    for text, w, tier in stash:
        if tier == 'auth':
            matched_auth.append((text, w)); match_w += w
        elif tier == 'regex':
            matched_regex.append((text, w)); match_w += w
        elif tier is None and text in sem_results and sem_results[text][1]:
            matched_sem.append((text, w, sem_results[text][0])); match_w += w
        else:
            unmatched.append(text)
    total_w = total_w or 1e-9
    matched_count = len(matched_auth) + len(matched_regex) + len(matched_sem)
    unweighted = matched_count / len(top)
    weighted = match_w / total_w
    score = 0.5 * unweighted + 0.5 * weighted
    return {
        'score': float(np.clip(score, 0.0, 1.0)),
        'unweighted_fraction': float(unweighted),
        'weighted_fraction': float(weighted),
        'matched_count': matched_count,
        'matched_count_authoritative': len(matched_auth),
        'matched_count_regex': len(matched_regex),
        'matched_count_semantic': len(matched_sem),
        'total': len(top),
        'matched_labels': ([t for t, _ in matched_auth]
                           + [t for t, _ in matched_regex]
                           + [t for t, _, _ in matched_sem]),
        'matched_labels_authoritative': [t for t, _ in matched_auth],
        'matched_labels_regex': [t for t, _ in matched_regex],
        'matched_labels_semantic': [(t, float(s)) for t, _, s in matched_sem],
        'semantic_similarities': {k: float(v) for k, v in sem_sim_by_concept.items()},
        'unmatched_labels': unmatched,
    }


def _build_relevance_matchers(config, data, int_to_concept_text):
    """Return (patterns, auth_id_set, concept_to_snomed_id, semantic_scorer) for
    auth/regex/semantic SNOMED concept matching. Mirrors the matcher
    construction inside _evaluate_explainability_quality so the same matching
    logic drives both the relevance score and the v138 relevance-filtered
    explainability charts."""
    patterns = list(config.get('relevance_patterns', RELEVANCE_PATTERNS))
    concept_to_snomed_id = {}
    try:
        snomed_id_to_int = data.get('snomed_id_to_int', {})
        int_to_snomed_id_str = {v: str(k) for k, v in snomed_id_to_int.items()
                                  if k != 'PAD_VALUE'}
        for i, txt in int_to_concept_text.items():
            sid = int_to_snomed_id_str.get(i, '')
            if sid and txt != '[PAD]':
                concept_to_snomed_id[txt] = sid
    except Exception as _c2sid_err:
        print(f'  [could not build concept_to_snomed_id: {_c2sid_err!r}]')

    codelist_paths = list(config.get('concept_codelist_paths', []))
    auth_id_set = set()
    if codelist_paths:
        try:
            auth_id_set, _ = _load_concept_ids_cached(tuple(codelist_paths))
        except Exception as _auth_err:
            print(f'  [codelist load failed, falling back to regex-only: {_auth_err!r}]')

    semantic_scorer = None
    if bool(config.get('use_semantic_relevance', False)):
        try:
            semantic_scorer = _get_semantic_scorer(
                model_name=config.get('semantic_relevance_model',
                                       'sentence-transformers/all-MiniLM-L6-v2'),
                threshold=float(config.get('semantic_relevance_threshold', 0.5)),
                reference_phrases=config.get('semantic_reference_phrases'),
            )
        except Exception as _sem_err:
            print(f'  [semantic relevance] init failed, proceeding without: {_sem_err!r}')
            semantic_scorer = None
    return patterns, auth_id_set, concept_to_snomed_id, semantic_scorer


def _filter_to_relevant_concepts(items, patterns, auth_id_set,
                                  concept_to_snomed_id, semantic_scorer):
    """Subset (concept_text, value) items to those matching auth/regex/semantic.
    Order-preserving. Used by the v138 relevance-filtered charts."""
    import re
    if not items:
        return []
    pat = re.compile('|'.join(patterns), re.IGNORECASE) if patterns else None
    id_set = auth_id_set or set()
    c2id = concept_to_snomed_id or {}
    matched_set = set()
    residual_idx = []
    residual_texts = []
    for i, (text, _w) in enumerate(items):
        sid = c2id.get(text, '')
        if sid and sid in id_set:
            matched_set.add(i)
        elif pat and pat.search(text):
            matched_set.add(i)
        else:
            residual_idx.append(i)
            residual_texts.append(text)
    if semantic_scorer and residual_texts:
        try:
            scored = semantic_scorer(residual_texts)
            for j, (_sim, is_match) in enumerate(scored):
                if is_match:
                    matched_set.add(residual_idx[j])
        except Exception as _sem_err:
            print(f'  [semantic relevance filter failed, skipping residual: {_sem_err!r}]')
    return [items[i] for i in range(len(items)) if i in matched_set]



def _local_global_consistency_score(X_codes, deltas, int_to_text, top_global_concepts,
                                    patient_top_k=10, global_top_n=50):
    """Average over patients of: fraction of their patient-level top-k concepts that
    appear in the global top-N. Skips patients with fewer than patient_top_k distinct
    non-padding concepts (rare at T=100)."""
    top_global_set = set(lab for lab, _ in top_global_concepts[:global_top_n])
    if not top_global_set:
        return {'score': 0.0, 'n_patients_scored': 0,
                'patient_top_k': int(patient_top_k), 'global_top_n': int(global_top_n)}
    abs_d = np.abs(deltas)
    N, T = X_codes.shape
    fracs = []
    for i in range(N):
        best_by_label = {}
        for t in range(T):
            enc = int(X_codes[i, t])
            if enc == 0:
                continue
            lab = int_to_text.get(enc, '[UNK]')
            v = float(abs_d[i, t])
            if v > best_by_label.get(lab, 0.0):
                best_by_label[lab] = v
        if len(best_by_label) < patient_top_k:
            continue
        patient_top = sorted(best_by_label.items(), key=lambda x: x[1], reverse=True)[:patient_top_k]
        patient_labels = {lab for lab, _ in patient_top}
        fracs.append(len(patient_labels & top_global_set) / len(patient_labels))
    return {
        'score': float(np.mean(fracs)) if fracs else 0.0,
        'n_patients_scored': len(fracs),
        'patient_top_k': int(patient_top_k),
        'global_top_n': int(global_top_n),
    }


def _evaluate_explainability_quality(model, data, config,
                                     X_snomed_sample, X_other_sample,
                                     snomed_deltas_all, orig_preds_all,
                                     top_snomed_global, int_to_snomed_text,
                                     baseline_codes, X_med_sample=None):
    """Run faithfulness + clinical-relevance + consistency; combine into a single
    0–100 rating using configurable weights. Prints a compact report."""
    import time
    weights = dict(config.get('explainability_rating_weights',
                              {'faithfulness': 0.5, 'relevance': 0.3, 'consistency': 0.2}))
    patterns = list(config.get('relevance_patterns', RELEVANCE_PATTERNS))
    aopc_n = int(config.get('explainability_aopc_n_patients', 100))
    aopc_ks = tuple(config.get('explainability_aopc_k_values', (1, 5, 10, 20)))
    rel_top_k = int(config.get('explainability_relevance_top_k', 10))
    cons_top_k = int(config.get('explainability_consistency_top_k', 10))
    cons_top_n = int(config.get('explainability_consistency_top_global_n', 50))

    print('\n=== Explainability quality evaluation ===')

    t0 = time.time()
    faith = _faithfulness_aopc(
        model, X_snomed_sample, X_other_sample, snomed_deltas_all, orig_preds_all,
        baseline_codes, config, n_patients=aopc_n, k_values=aopc_ks,
        X_med_sample=X_med_sample)
    print(f'  Faithfulness (AOPC, n={faith.get("n_patients", 0)} pts, k={list(aopc_ks)}) '
          f'— score: {faith["score"]:.3f}  |  drop@top={faith.get("aopc_top_mean_drop", 0):.4f} '
          f'vs drop@rand={faith.get("aopc_random_mean_drop", 0):.4f}  '
          f'[{time.time()-t0:.1f}s]')

    # Build concept->SNOMED-ID mapping so auth-ID matching can win over regex.
    concept_to_snomed_id = {}
    try:
        snomed_id_to_int = data.get('snomed_id_to_int', {})
        int_to_snomed_id_str = {v: str(k) for k, v in snomed_id_to_int.items()
                                 if k != 'PAD_VALUE'}
        for i, txt in int_to_snomed_text.items():
            sid = int_to_snomed_id_str.get(i, '')
            if sid and txt != '[PAD]':
                concept_to_snomed_id[txt] = sid
    except Exception as _c2sid_err:
        print(f'  [could not build concept_to_snomed_id: {_c2sid_err!r}]')

    codelist_paths = list(config.get('concept_codelist_paths', []))
    auth_id_set = set()
    if codelist_paths:
        try:
            auth_id_set, _ = _load_concept_ids_cached(tuple(codelist_paths))
        except Exception as _auth_err:
            print(f'  [codelist load failed, falling back to regex-only: {_auth_err!r}]')

    # v113 Lever 4: optional semantic-embedding third-tier match
    semantic_scorer = None
    if bool(config.get('use_semantic_relevance', False)):
        try:
            semantic_scorer = _get_semantic_scorer(
                model_name=config.get('semantic_relevance_model',
                                       'sentence-transformers/all-MiniLM-L6-v2'),
                threshold=float(config.get('semantic_relevance_threshold', 0.5)),
                reference_phrases=config.get('semantic_reference_phrases'),
            )
        except Exception as _sem_err:
            print(f'  [semantic relevance] init failed, proceeding without: {_sem_err!r}')
            semantic_scorer = None

    rel = _clinical_relevance_score(top_snomed_global, patterns, top_k=rel_top_k,
                                     authoritative_id_set=auth_id_set,
                                     concept_to_snomed_id=concept_to_snomed_id,
                                     semantic_scorer=semantic_scorer)
    print(f'  Clinical relevance (top-{rel_top_k}) — score: {rel["score"]:.3f}  '
          f'| matched {rel["matched_count"]}/{rel["total"]} '
          f'(auth={rel["matched_count_authoritative"]}, '
          f'regex={rel["matched_count_regex"]}, '
          f'sem={rel.get("matched_count_semantic", 0)}, '
          f'unw={rel["unweighted_fraction"]:.2f}, w={rel["weighted_fraction"]:.2f})')
    if rel['matched_labels_authoritative']:
        show = rel['matched_labels_authoritative'][:5]
        more = max(0, len(rel['matched_labels_authoritative']) - 5)
        print(f'    auth-matched: {", ".join(show)}' + (f'  +{more} more' if more else ''))
    if rel['matched_labels_regex']:
        show = rel['matched_labels_regex'][:5]
        more = max(0, len(rel['matched_labels_regex']) - 5)
        print(f'    regex-matched: {", ".join(show)}' + (f'  +{more} more' if more else ''))
    if rel.get('matched_labels_semantic'):
        show = rel['matched_labels_semantic'][:5]
        more = max(0, len(rel['matched_labels_semantic']) - 5)
        formatted = ', '.join(f'{t} (sim={s:.2f})' for t, s in show)
        print(f'    semantic-matched: {formatted}' + (f'  +{more} more' if more else ''))
    if rel['unmatched_labels']:
        show = rel['unmatched_labels'][:5]
        more = max(0, len(rel['unmatched_labels']) - 5)
        print(f'    unmatched: {", ".join(show)}' + (f'  +{more} more' if more else ''))

    cons = _local_global_consistency_score(
        X_snomed_sample, snomed_deltas_all, int_to_snomed_text,
        top_snomed_global, patient_top_k=cons_top_k, global_top_n=cons_top_n)
    print(f'  Local↔global consistency (patient top-{cons_top_k} ∩ global top-{cons_top_n}) '
          f'— score: {cons["score"]:.3f}  |  patients scored: {cons["n_patients_scored"]}')

    wf = float(weights.get('faithfulness', 0.5))
    wr = float(weights.get('relevance', 0.3))
    wc = float(weights.get('consistency', 0.2))
    w_total = wf + wr + wc
    rating = 0.0 if w_total <= 0 else 100.0 * (wf * faith['score'] + wr * rel['score'] + wc * cons['score']) / w_total
    print(f'  Explainability Rating: {rating:.1f} / 100  '
          f'(weights — faith:{wf:.2f}, relevance:{wr:.2f}, consistency:{wc:.2f})')

    return {
        'rating_0_100': float(rating),
        'weights': {'faithfulness': wf, 'relevance': wr, 'consistency': wc},
        'faithfulness': faith,
        'clinical_relevance': rel,
        'consistency': cons,
    }



def evaluate_trial_explainability(model, data, config):
    """Model-level-only explainability + rating for a single tuning trial.

    Skips patient-level IG plots to keep per-trial cost low: just builds the
    SNOMED healthy baseline, runs one batched ablation over the sampled test
    patients, aggregates the global concept importance, then calls
    `_evaluate_explainability_quality`. Returns the rating dict (or None if
    SNOMED is disabled). Prints a compact rating block for the trial.
    """
    use_snomed = bool(config.get('use_snomed_branch', True))
    if not use_snomed:
        print('  (SNOMED branch disabled — skipping explainability rating.)')
        return None

    X_snomed_ids_train = data['X_snomed_ids_train']
    X_snomed_ids_test = data['X_snomed_ids_test']
    X_med_ids_train = data.get('X_med_ids_train')
    X_med_ids_test = data.get('X_med_ids_test')
    X_other_features_test = data['X_other_features_test']
    y_train = data['y_train']
    snomed_id_to_int = data['snomed_id_to_int']
    df = data['df']

    BASE_SNOMED_HEALTHY = compute_healthy_baseline_codes(
        X_snomed_ids_train, y_train, pad_value=0, branch_name='SNOMED')
    int_to_snomed_text = _build_int_to_text(
        snomed_id_to_int, df, 'observation', 'snomed_id', 'snomed_text')

    rng = np.random.default_rng(42)
    n_test = len(X_other_features_test)
    sample_n = min(int(config.get('explainability_sample_n', 400)), n_test)
    sample_idx = rng.choice(n_test, size=sample_n, replace=False)
    X_snomed_sample = X_snomed_ids_test[sample_idx]
    X_med_sample = X_med_ids_test[sample_idx] if X_med_ids_test is not None else None
    X_other_sample = X_other_features_test[sample_idx]

    predict_bs = int(config.get('explainability_predict_batch_size', 1024))
    chunk_pts = int(config.get('explainability_chunk_patients', 32))

    orig_preds_all, snomed_deltas_all = temporal_ablation_batch(
        model, X_snomed_sample, X_med_sample, X_other_sample,
        branch='snomed', baseline_codes=BASE_SNOMED_HEALTHY, config=config,
        predict_batch_size=predict_bs, chunk_patients=chunk_pts, show_progress=False)

    # Aggregate global concept importance (same logic as run_explainability._aggregate)
    scores = defaultdict(float); counts = defaultdict(int)
    abs_deltas = np.abs(snomed_deltas_all)
    mask = X_snomed_sample != 0
    for i in range(X_snomed_sample.shape[0]):
        row_mask = mask[i]
        if not row_mask.any():
            continue
        encs = X_snomed_sample[i][row_mask]
        ds = abs_deltas[i][row_mask]
        for enc, d in zip(encs, ds):
            label = int_to_snomed_text.get(int(enc), '[UNK]')
            scores[label] += float(d); counts[label] += 1
    means = {lab: scores[lab] / max(1, counts[lab]) for lab in scores}
    # v139 fix: keep the full ranking so explainability_relevance_top_k > 25
    # and consistency global_top_n=50 are honored. Display/'top_concepts'
    # field stays at 10/25 for backward compatibility with tuner reports.
    top_snomed_global_full = sorted(means.items(), key=lambda x: x[1], reverse=True)
    top_snomed_global = top_snomed_global_full[:25]

    result = _evaluate_explainability_quality(
        model, data, config,
        X_snomed_sample=X_snomed_sample,
        X_other_sample=X_other_sample,
        snomed_deltas_all=snomed_deltas_all,
        orig_preds_all=orig_preds_all,
        top_snomed_global=top_snomed_global_full,
        int_to_snomed_text=int_to_snomed_text,
        baseline_codes=BASE_SNOMED_HEALTHY,
        X_med_sample=X_med_sample,
    )
    result['top_concepts'] = top_snomed_global[:10]
    return result


def run_explainability(model, data, config):
    """Patient-level and model-level explainability for enabled branches only.

    Progress bars cover:
      - Healthy baseline construction (per timestep).
      - Model-level IG accumulation (per patient).
      - Model-level temporal ablation (per chunk of patients, batched across timesteps).
    TF C++ ERROR stderr messages are suppressed around each model.predict call via
    fd-level redirection (see `_suppress_tf_cpp_stderr`).
    """
    import matplotlib.pyplot as plt
    import time

    use_snomed = bool(config.get('use_snomed_branch', True))
    use_med = bool(config.get('use_med_branch', True))

    X_snomed_ids_train = data['X_snomed_ids_train']
    X_snomed_ids_test = data['X_snomed_ids_test']
    X_med_ids_train = data['X_med_ids_train']
    X_med_ids_test = data['X_med_ids_test']
    X_other_features_train = data['X_other_features_train']
    X_other_features_test = data['X_other_features_test']
    y_train = data['y_train']
    patient_ids_test = data['patient_ids_test']
    snomed_id_to_int = data['snomed_id_to_int']
    med_id_to_int = data['med_id_to_int']
    other_feature_cols = data['other_feature_cols']
    df = data['df']

    # v105-shap: three baseline modes.
    #   'healthy'       -> class-conditional single point (v105 default)
    #   'shap_training' -> unconditional single point (v105s)
    #   'shap_eg'       -> Expected Gradients over a 100-row training background (this one)
    baseline_type = str(config.get('explainability_baseline_type', 'healthy')).lower()
    BACKGROUND_OTHER = None  # populated for shap_eg mode
    if baseline_type == 'shap_eg':
        _bg_size = int(config.get('shap_background_size', 100))
        _bg_seed = int(config.get('SEED', 42))
        BACKGROUND_OTHER = build_shap_background_other(
            X_other_features_train, size=_bg_size, seed=_bg_seed)
        # EG's completeness axiom is sum(phi) ≈ f(x) - E_b[f(b)], but the downstream
        # plots and temporal-ablation code still want a single-point reference.
        # Use the background *mean* for static features and the unconditional mode
        # for sequences — plots stay interpretable, and EG attributions use the
        # full background distribution anyway.
        BASE_OTHER_HEALTHY = np.mean(BACKGROUND_OTHER, axis=0, keepdims=True)
        BASE_SNOMED_HEALTHY = (compute_training_baseline_codes(X_snomed_ids_train, pad_value=0, branch_name='SNOMED')
                              if use_snomed else None)
        BASE_MED_HEALTHY = (compute_training_baseline_codes(X_med_ids_train, pad_value=0, branch_name='MED')
                           if use_med else None)
        baseline_desc = (f'SHAP Expected Gradients (background size={BACKGROUND_OTHER.shape[0]}, '
                         f'static) + unconditional mode (sequences)')
    elif baseline_type == 'shap_training':
        BASE_OTHER_HEALTHY = compute_training_baseline_other(X_other_features_train)
        BASE_SNOMED_HEALTHY = (compute_training_baseline_codes(X_snomed_ids_train, pad_value=0, branch_name='SNOMED')
                              if use_snomed else None)
        BASE_MED_HEALTHY = (compute_training_baseline_codes(X_med_ids_train, pad_value=0, branch_name='MED')
                           if use_med else None)
        baseline_desc = 'SHAP-style (unconditional training mean/mode, single-point)'
    else:
        BASE_OTHER_HEALTHY = compute_healthy_baseline_other(X_other_features_train, y_train)
        BASE_SNOMED_HEALTHY = (compute_healthy_baseline_codes(X_snomed_ids_train, y_train, pad_value=0, branch_name='SNOMED')
                              if use_snomed else None)
        BASE_MED_HEALTHY = (compute_healthy_baseline_codes(X_med_ids_train, y_train, pad_value=0, branch_name='MED')
                           if use_med else None)
        baseline_desc = 'healthy (class-conditional, y==0)'
    shapes = [f'other: {BASE_OTHER_HEALTHY.shape}']
    if use_snomed: shapes.append(f'SNOMED: {BASE_SNOMED_HEALTHY.shape}')
    if use_med: shapes.append(f'MED: {BASE_MED_HEALTHY.shape}')
    print('Healthy baselines — ' + ', '.join(shapes))
    print(f"Branches enabled — SNOMED: {use_snomed}  MED: {use_med}")

    # (sanity-check moved below, after int_to_snomed_text is built — see below)

    int_to_snomed_text = (_build_int_to_text(snomed_id_to_int, df, 'observation', 'snomed_id', 'snomed_text')
                          if use_snomed else {})
    int_to_med_text = (_build_int_to_text(med_id_to_int, df, 'medication', 'med_code_id', 'med_text')
                       if use_med else {})

    # v138: build relevance matchers once so both the SNOMED chart blocks and
    # _evaluate_explainability_quality see the same auth/regex/semantic logic.
    show_charts = bool(config.get('show_explainability_charts', True))
    show_filtered = bool(config.get('show_relevance_filtered_charts', True))
    filtered_top_n = int(config.get('relevance_filtered_top_n', 50))
    rel_patterns = []
    rel_auth_set = set()
    rel_concept_to_id = {}
    rel_semantic = None
    if use_snomed and show_filtered:
        try:
            (rel_patterns, rel_auth_set, rel_concept_to_id, rel_semantic) = (
                _build_relevance_matchers(config, data, int_to_snomed_text))
        except Exception as _rm_err:
            print(f'  [v138 relevance matchers] build failed, skipping filtered charts: {_rm_err!r}')
            show_filtered = False

    # --- Baseline sanity-check (moved below int_to_snomed_text so top-5 codes resolve) ---
    try:
        _y_arr = np.asarray(y_train).reshape(-1)
        _n_healthy = int((_y_arr == 0).sum()); _n_total = int(_y_arr.size)
        print(f'\n=== Baseline sanity check (type: {baseline_desc}) ===')
        if baseline_type == 'shap_eg':
            print(f'  Built from {int(BACKGROUND_OTHER.shape[0])} random training rows '
                  f'(unconditional on class; used as the EG background distribution)')
        elif baseline_type == 'shap_training':
            print(f'  Built from all {_n_total} training patients (unconditional / SHAP-style single-point)')
        else:
            print(f'  Built from {_n_healthy} / {_n_total} training patients (y == 0, class-conditional)')
        print(f'  Static-feature baseline mean: {BASE_OTHER_HEALTHY.ravel().tolist()}')
        if use_snomed:
            _vals, _cnts = np.unique(BASE_SNOMED_HEALTHY.ravel(), return_counts=True)
            _order = np.argsort(-_cnts)[:5]
            print(f'  SNOMED baseline: shape {BASE_SNOMED_HEALTHY.shape}, '
                  f'{int(len(_vals))} unique codes across timesteps')
            print(f'  Top-5 most-common baseline codes:')
            for _j in _order:
                _enc = int(_vals[_j])
                _lbl = int_to_snomed_text.get(_enc, "?")
                _lbl_short = (_lbl[:70] + '...') if len(_lbl) > 70 else _lbl
                print(f'    enc={_enc:>6d}  used @ {int(_cnts[_j]):>3d} timesteps  — {_lbl_short}')
    except Exception as _sanity_err:
        print(f'  [baseline sanity-check skipped: {_sanity_err!r}]')

    # ============== PATIENT-LEVEL ==============
    patient_idx = config.get('explainability_patient_idx', 0)
    patient_id = patient_ids_test[patient_idx]
    X_snomed_patient = X_snomed_ids_test[patient_idx:patient_idx+1] if use_snomed else None
    X_med_patient = X_med_ids_test[patient_idx:patient_idx+1] if use_med else None
    X_other_patient = X_other_features_test[patient_idx:patient_idx+1]
    print(f'\nExplaining patient: {patient_id}')
    print(f'Model prediction: {predict_proba(model, X_snomed_patient, X_med_patient, X_other_patient, config)[0]:.4f}')

    # v143: static attribution via feature-level ablation, same method/units
    # as the SNOMED temporal ablation below. ig_patient is now (F,) of signed
    # deltas: p(orig) - p(masked when feature i replaced by baseline value).
    # The IG/EG completeness check from earlier versions doesn't apply
    # (ablation isn't additive across features) and was removed in v143.
    _, ig_patient = static_feature_ablation_single(
        model, X_snomed_patient, X_med_patient, X_other_patient, config,
        baseline_other=BASE_OTHER_HEALTHY)
    _attr_method = 'static-feature ablation'
    print(f'  [static attribution] method: {_attr_method} '
          f'(baseline = {baseline_desc})')

    # v141: compute SNOMED concept contributions first (if enabled), then
    # render ONE combined chart of static + SNOMED contributions. The
    # standalone static chart (top-15 |IG|) and SNOMED chart (top-20 signed
    # delta) from v140 are merged into a single signed-contribution chart.
    concept_contrib = []
    top_snomed = []
    if use_snomed:
        orig_pred, snomed_deltas = temporal_ablation_snomed(
            model, X_snomed_patient, X_med_patient, X_other_patient, config,
            baseline_snomed=BASE_SNOMED_HEALTHY)
        print(f'Original prediction: {orig_pred:.4f}')
        print(f'Top-5 influential SNOMED timesteps: {np.argsort(np.abs(snomed_deltas))[::-1][:5].tolist()}')
        for t, (enc, delta) in enumerate(zip(X_snomed_patient[0], snomed_deltas)):
            enc = int(enc)
            if enc == 0:
                continue
            concept_contrib.append((t, int_to_snomed_text.get(enc, '[UNK]'), float(delta)))
        concept_contrib.sort(key=lambda x: abs(x[2]), reverse=True)
        top_snomed = concept_contrib[:20]
        display(pd.DataFrame(top_snomed, columns=['timestep', 'snomed_concept', 'delta']))

        # v145/v146: concept-level dedup for the patient charts. Same recipe
        # used at model-level (see _aggregate below): mean signed delta per
        # occurrence. Repeats of the same concept in a patient's record are
        # collapsed by averaging — admin codes that occur 8x don't get
        # 8x larger bars. Sign is kept; concepts the model uses both ways
        # shrink toward zero (intentional — direction-ambivalent codes
        # are usually noisy admin/follow-up records).
        _concept_sum = {}
        _concept_count = {}
        for _, _label, _d in concept_contrib:
            _concept_sum[_label] = _concept_sum.get(_label, 0.0) + float(_d)
            _concept_count[_label] = _concept_count.get(_label, 0) + 1
        _concept_mean = {lab: _concept_sum[lab] / _concept_count[lab]
                          for lab in _concept_sum}
        concept_contrib_unique = sorted(_concept_mean.items(),
                                         key=lambda x: abs(x[1]), reverse=True)
        if len(concept_contrib_unique) < len(concept_contrib):
            print(f'  [v146 concept-level dedup] {len(concept_contrib)} timestep entries '
                  f'-> {len(concept_contrib_unique)} unique concepts '
                  f'(mean signed delta per occurrence)')

    if show_charts:
        _combined_k = int(config.get('combined_chart_top_k', 30))
        # v147: collapse one-hot static groups (gender_M+gender_F -> gender, etc.)
        _static_pairs = [(col, float(ig_patient[i]))
                          for i, col in enumerate(other_feature_cols)]
        _static_items = [(f'[static] {disp}', val)
                          for disp, val in _aggregate_static_features(_static_pairs)]
        # v145: concept_contrib_unique collapses repeat-timestep duplicates.
        _snomed_items = ([(f'[snomed] {label}', float(d))
                            for label, d in concept_contrib_unique]
                          if use_snomed else [])
        _combined = _static_items + _snomed_items
        _combined.sort(key=lambda x: abs(x[1]), reverse=True)
        _combined_top = _combined[:_combined_k]
        _suffix = 'Static + SNOMED' if use_snomed else 'Static'
        _plot_barh_concepts(
            f'Patient {patient_id} — Top {len(_combined_top)} {_suffix} contributions (signed)',
            _combined_top,
            'signed contribution; + = increases risk')

    # v138/v142: relevance-filtered patient chart — top-N SNOMED filtered to
    # auth/regex/semantic matches, plus all static features (hand-curated
    # demographics; intrinsically relevant — not subject to the SNOMED
    # matchers). Combined and re-sorted by |signed contribution|.
    if show_filtered and (use_snomed or len(other_feature_cols) > 0):
        _filtered_items = []
        if use_snomed:
            # v145: pull from concept_contrib_unique so each concept appears
            # at most once before the top-N slice (otherwise admin codes that
            # repeat many times would crowd the top-N with duplicates).
            _patient_topN = concept_contrib_unique[:filtered_top_n]
            _patient_items = [(c, d) for c, d in _patient_topN]
            _patient_matched = _filter_to_relevant_concepts(
                _patient_items, rel_patterns, rel_auth_set, rel_concept_to_id, rel_semantic)
            print(f'  [v138 relevance-filtered] patient SNOMED: '
                  f'{len(_patient_matched)}/{len(_patient_topN)} of top-{filtered_top_n} matched '
                  f'(unique concepts; v145 dedup applied)')
            _filtered_items.extend([(f'[snomed] {c}', d) for c, d in _patient_matched])
        # v142: static features always included.
        # v147: collapse one-hot static groups for display.
        _static_pairs_p = [(col, float(ig_patient[i]))
                            for i, col in enumerate(other_feature_cols)]
        _filtered_items.extend([(f'[static] {disp}', val)
                                  for disp, val in _aggregate_static_features(_static_pairs_p)])
        _filtered_items.sort(key=lambda x: abs(x[1]), reverse=True)
        _suffix = ('SNOMED + Static' if use_snomed else 'Static')
        _plot_barh_concepts(
            f'Patient {patient_id} — Clinically Relevant Contributions ({_suffix}, '
            f'{len(_filtered_items)} items)',
            _filtered_items,
            'signed contribution; + = increases risk')

    if use_med:
        orig_pred_m, med_deltas = temporal_ablation_med(
            model, X_snomed_patient, X_med_patient, X_other_patient, config,
            baseline_med=BASE_MED_HEALTHY)
        med_contrib = []
        for t, (enc, delta) in enumerate(zip(X_med_patient[0], med_deltas)):
            enc = int(enc)
            if enc == 0:
                continue
            med_contrib.append((t, int_to_med_text.get(enc, '[UNK]'), float(delta)))
        med_contrib.sort(key=lambda x: abs(x[2]), reverse=True)
        top_med = med_contrib[:20]
        if top_med:
            display(pd.DataFrame(top_med, columns=['timestep', 'med_concept', 'delta']))
            if show_charts:
                _plot_barh_concepts(
                    f'Patient {patient_id} — MED Concept Contributions (delta when masked)',
                    [(c, d) for _, c, d in top_med],
                    'delta = p(orig) - p(masked); + = increases risk')
        else:
            print(f'Patient {patient_id} has no non-padding medications — skipping MED concept plot.')

    # ============== MODEL-LEVEL ==============
    rng = np.random.default_rng(42)
    n_test = len(X_other_features_test)
    sample_n = min(config.get('explainability_sample_n', 100), n_test)
    sample_idx = rng.choice(n_test, size=sample_n, replace=False)
    X_snomed_sample = X_snomed_ids_test[sample_idx] if use_snomed else None
    X_med_sample = X_med_ids_test[sample_idx] if use_med else None
    X_other_sample = X_other_features_test[sample_idx]
    print(f'\nModel-level explainability using {sample_n} test patients')

    # v143: model-level static attribution via feature-level ablation, same
    # method/units as the SNOMED temporal ablation below. ig_global is now
    # mean over patients of |delta_i| where delta_i = p(orig) - p(masked
    # when feature i replaced by baseline). Same probability-delta units
    # as SNOMED's mean |delta|.
    t0 = time.time()
    _static_chunk_pts = int(config.get('explainability_chunk_patients', 32))
    _, _static_deltas_all = static_feature_ablation_batch(
        model, X_snomed_sample, X_med_sample, X_other_sample,
        baseline_other=BASE_OTHER_HEALTHY, config=config,
        predict_batch_size=int(config.get('explainability_predict_batch_size', 1024)),
        chunk_patients=_static_chunk_pts, show_progress=True)
    # v146: signed mean per feature (same recipe as SNOMED). Sort/select
    # downstream uses |ig_global[i]| so a feature ambivalent across patients
    # (positive delta in some, negative in others) ranks low.
    ig_global = np.mean(_static_deltas_all, axis=0).astype(np.float64)
    print(f'Global static-feature ablation done in {time.time()-t0:.1f}s '
          f'(N={len(X_other_sample)}, F={X_other_sample.shape[1]})')

    # v141: when SNOMED is enabled the static-feature chart is folded into a
    # combined static+SNOMED chart rendered below (after SNOMED aggregation).
    # When SNOMED is off, fall back to the v140 static-only chart here.
    if not use_snomed and show_charts:
        # v147: aggregate one-hot groups for display, then reuse the shared
        # bar-plot helper (signed bars + zero axvline come for free).
        _fallback_pairs = [(col, float(ig_global[i]))
                            for i, col in enumerate(other_feature_cols)]
        _fallback_agg = _aggregate_static_features(_fallback_pairs)
        _fallback_agg.sort(key=lambda x: abs(x[1]), reverse=True)
        _fallback_top = _fallback_agg[:min(20, len(_fallback_agg))]
        _plot_barh_concepts(
            f'Model-level — Top {len(_fallback_top)} Static-Feature Drivers (mean signed delta)',
            _fallback_top,
            'Mean signed delta = p(orig) - p(masked); + = increases risk')

    predict_bs = int(config.get('explainability_predict_batch_size', 1024))
    chunk_pts = int(config.get('explainability_chunk_patients', 32))

    def _aggregate(X_codes, deltas, int_to_text):
        # v146: mean SIGNED delta per occurrence — same recipe as the
        # patient-level dedup. Bars have direction: + = masking lowered
        # the prediction (concept is a risk signal), - = masking raised
        # it (concept is protective). Sort by |mean| so direction-
        # ambivalent concepts drop down the ranking.
        scores = defaultdict(float); counts = defaultdict(int)
        mask = X_codes != 0
        for i in range(X_codes.shape[0]):
            row_mask = mask[i]
            if not row_mask.any():
                continue
            encs = X_codes[i][row_mask]
            ds = deltas[i][row_mask]   # signed, not abs
            for enc, d in zip(encs, ds):
                label = int_to_text.get(int(enc), '[UNK]')
                scores[label] += float(d)
                counts[label] += 1
        means = {lab: scores[lab] / max(1, counts[lab]) for lab in scores}
        return sorted(means.items(), key=lambda x: abs(x[1]), reverse=True)

    orig_preds_all = None
    snomed_deltas_all = None
    top_snomed_global = []
    if use_snomed:
        t0 = time.time()
        # v106 fix: in shap_eg mode, optionally use per-patient marginal samples.
        _snomed_baseline_for_ablation = BASE_SNOMED_HEALTHY
        _shap_abl_mode = str(config.get('shap_ablation_mode', 'per_position_mode')).lower()
        if baseline_type == 'shap_eg' and _shap_abl_mode == 'marginal_sample':
            # Build a (sample_n, T) per-patient baseline by sampling random rows
            # from the training SNOMED sequences (unconditional on class).
            _rng_bl = np.random.default_rng(int(config.get('SEED', 42)) + 7)
            _bg_snomed_size = min(int(config.get('shap_background_size', 1000)),
                                   len(X_snomed_ids_train))
            _bg_snomed_pool_idx = _rng_bl.choice(
                len(X_snomed_ids_train), size=_bg_snomed_size, replace=False)
            _bg_snomed_pool = np.asarray(X_snomed_ids_train)[_bg_snomed_pool_idx]
            # One sampled baseline patient per test patient
            _pick = _rng_bl.integers(0, _bg_snomed_size, size=sample_n)
            _snomed_baseline_for_ablation = _bg_snomed_pool[_pick].astype(np.int32)
            print(f'  [shap_ablation_mode=marginal_sample — per-patient baselines '
                  f'sampled from {_bg_snomed_size} training rows]')
        orig_preds_all, snomed_deltas_all = temporal_ablation_batch(
            model, X_snomed_sample, X_med_sample, X_other_sample,
            branch='snomed', baseline_codes=_snomed_baseline_for_ablation, config=config,
            predict_batch_size=predict_bs, chunk_patients=chunk_pts, show_progress=True)
        print(f'Batched SNOMED ablation ({sample_n} pts x {X_snomed_sample.shape[1]} steps) done in {time.time()-t0:.1f}s')
        # v139 fix: aggregate the full ranking once. Slice to 25 only for the
        # default chart/display; pass the full list to the evaluator so its
        # explainability_relevance_top_k (default 30, can be 50+) and the
        # consistency-score's global_top_n (default 50) actually take effect.
        top_snomed_global_full = _aggregate(X_snomed_sample, snomed_deltas_all, int_to_snomed_text)
        top_snomed_global = top_snomed_global_full[:25]
        # v146: combined model-level chart — both halves now use 'mean signed
        # delta per occurrence' (same recipe as patient-level dedup). Sort
        # by |mean| so direction-ambivalent items drop down the ranking.
        if show_charts:
            _combined_k = int(config.get('combined_chart_top_k', 30))
            # v147: collapse one-hot static groups for display.
            _static_pairs_g = [(col, float(ig_global[i]))
                                for i, col in enumerate(other_feature_cols)]
            _static_items_g = [(f'[static] {disp}', val)
                                for disp, val in _aggregate_static_features(_static_pairs_g)]
            _snomed_items_g = [(f'[snomed] {label}', float(score))
                                for label, score in top_snomed_global_full]
            _combined_g = _static_items_g + _snomed_items_g
            _combined_g.sort(key=lambda x: abs(x[1]), reverse=True)
            _combined_g_top = _combined_g[:_combined_k]
            _plot_barh_concepts(
                f'Model-level — Top {len(_combined_g_top)} Static + SNOMED drivers '
                f'(mean signed delta)',
                _combined_g_top,
                'Mean signed delta = p(orig) - p(masked); + = increases risk')
        display(pd.DataFrame(top_snomed_global, columns=['snomed_concept', 'mean_signed_delta']).head(10))
        # v138/v142: relevance-filtered model-level chart — top-N SNOMED
        # filtered to auth/regex/semantic matches, plus all static features
        # (intrinsically relevant; not subject to the SNOMED matchers).
        # Combined and re-sorted by mean |attribution|.
        if show_filtered:
            _global_topN = top_snomed_global_full[:filtered_top_n]
            _global_matched = _filter_to_relevant_concepts(
                _global_topN, rel_patterns, rel_auth_set, rel_concept_to_id, rel_semantic)
            print(f'  [v138 relevance-filtered] model-level SNOMED: '
                  f'{len(_global_matched)}/{len(_global_topN)} of top-{filtered_top_n} matched')
            _filtered_items_g = [(f'[snomed] {label}', float(score))
                                   for label, score in _global_matched]
            # v147: collapse one-hot static groups for display.
            _static_pairs_gf = [(col, float(ig_global[i]))
                                  for i, col in enumerate(other_feature_cols)]
            _filtered_items_g.extend([(f'[static] {disp}', val)
                                        for disp, val in _aggregate_static_features(_static_pairs_gf)])
            # v146: sort by |mean| so direction-ambivalent items rank lower.
            _filtered_items_g.sort(key=lambda x: abs(x[1]), reverse=True)
            _plot_barh_concepts(
                f'Model-level — Clinically Relevant Drivers (SNOMED + Static, '
                f'{len(_filtered_items_g)} items)',
                _filtered_items_g,
                'Mean signed delta = p(orig) - p(masked); + = increases risk')
    else:
        print('SNOMED branch disabled — skipping SNOMED ablation.')

    if use_med:
        t0 = time.time()
        _, med_deltas_all = temporal_ablation_batch(
            model, X_snomed_sample, X_med_sample, X_other_sample,
            branch='med', baseline_codes=BASE_MED_HEALTHY, config=config,
            predict_batch_size=predict_bs, chunk_patients=chunk_pts, show_progress=True)
        print(f'Batched MED ablation    ({sample_n} pts x {X_med_sample.shape[1]} steps) done in {time.time()-t0:.1f}s')
        top_med_global = _aggregate(X_med_sample, med_deltas_all, int_to_med_text)[:25]
        if top_med_global:
            if show_charts:
                _plot_barh_concepts(
                    'Model-level MED Concept Importance (mean |delta| when masked)',
                    top_med_global, 'Mean |delta| contribution')
            display(pd.DataFrame(top_med_global, columns=['med_concept', 'mean_signed_delta']).head(10))
        else:
            print('No medication concepts found in sample — skipping MED model-level plot.')
    else:
        print('MED branch disabled — skipping MED ablation.')

    # --- Automated explainability rating (gated on config['rate_explainability']) ---
    if use_snomed and bool(config.get('rate_explainability', True)) and orig_preds_all is not None:
        try:
            _evaluate_explainability_quality(
                model, data, config,
                X_snomed_sample=X_snomed_sample,
                X_other_sample=X_other_sample,
                snomed_deltas_all=snomed_deltas_all,
                orig_preds_all=orig_preds_all,
                top_snomed_global=top_snomed_global_full,  # v139 fix: full ranking, not the 25-slice
                int_to_snomed_text=int_to_snomed_text,
                baseline_codes=BASE_SNOMED_HEALTHY,
                X_med_sample=X_med_sample,  # v147_k7: dual-branch rating bug fix
            )
        except Exception as _rating_err:
            print(f'  [explainability rating skipped: {_rating_err!r}]')

    print('Explainability analysis complete.')


### Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning using Keras Tuner — dual-branch (SNOMED + MED).
# v111 ranges: shifted around v110's winner (LR 1.72e-4, LSTM 64, emb_drop 0.30,
# dense_drop 0.45 on v38_v2 → AUC 81.2, sens 96.1, spec 66.3).
# Strategy: favour configurations near v110's and v100's winners; narrower ranges
# = faster convergence on the 25-trial budget. Lower LR + larger batch = room to
# push best epoch past the 5 we saw on v110. See build_v111.py for per-range deltas.
import keras_tuner as kt
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras
import itertools


class SensSpecTuner(kt.BayesianOptimization):
    """Custom tuner that optimizes the weighted average of sensitivity and specificity
    (or maximizes specificity subject to a sensitivity floor)."""

    def __init__(self, *args, validation_data=None,
                 sensitivity_weight=5.0, specificity_weight=1.0,
                 use_sensitivity_floor=False, sensitivity_floor=0.95, **kwargs):
        super().__init__(*args, **kwargs)
        self.val_data = validation_data
        self.sw = sensitivity_weight
        self.spw = specificity_weight
        self.use_sensitivity_floor = use_sensitivity_floor
        self.sensitivity_floor = sensitivity_floor
        self.best_score = -1
        self.best_sens = 0
        self.best_spec = 0
        self.best_threshold = 0
        self.best_trial_params = {}
        self.all_trial_results = []

    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters
        cancer_class_weight = hp.Float('cancer_class_weight', 1.0, 2.0, step=0.25)
        batch_size = hp.Choice('batch_size', [32, 64])  # v134: dropped 128 — smaller batches generalise better
        kwargs['batch_size'] = batch_size
        kwargs['class_weight'] = {0: 1.0, 1: cancer_class_weight}
        return super().run_trial(trial, *args, **kwargs)

    def _pick_threshold(self, tpr, fpr, thresholds):
        sens = tpr
        spec = 1 - fpr
        if self.use_sensitivity_floor:
            eligible = sens >= self.sensitivity_floor
            if np.any(eligible):
                idxs = np.where(eligible)[0]
                best = idxs[np.argmax(spec[idxs])]
                return thresholds[best], sens[best], spec[best]
            # fallback: highest sens
            best = int(np.argmax(sens))
            return thresholds[best], sens[best], spec[best]
        scores = (self.sw * sens + self.spw * spec) / (self.sw + self.spw)
        best = int(np.argmax(scores))
        return thresholds[best], sens[best], spec[best]

    def on_trial_end(self, trial):
        super().on_trial_end(trial)
        from sklearn.metrics import confusion_matrix as sk_confusion_matrix
        from sklearn.metrics import roc_curve

        try:
            model = self.load_model(trial)
        except Exception:
            return
        if self.val_data is None:
            return

        val_x, val_y = self.val_data
        y_proba = model.predict(val_x, verbose=0)
        y_true = val_y.ravel()

        fpr, tpr, thresholds = roc_curve(y_true, y_proba)
        threshold, sens, spec = self._pick_threshold(tpr, fpr, thresholds)

        y_pred = (y_proba.ravel() >= threshold).astype(int)
        cm = sk_confusion_matrix(y_true, y_pred, labels=[0, 1])
        TN, FP, FN, TP = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
        sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0

        if self.use_sensitivity_floor:
            score = spec if sens >= self.sensitivity_floor else sens - 1.0
        else:
            score = (self.sw * sens + self.spw * spec) / (self.sw + self.spw)

        hp = trial.hyperparameters
        self.all_trial_results.append({
            'trial_id': trial.trial_id,
            'threshold': float(threshold),
            'sensitivity': float(sens),
            'specificity': float(spec),
            'score': float(score),
            'model_params': {key: hp.get(key) for key in [
                'embedding_dim', 'snomed_lstm_units', 'med_lstm_units', 'med_embedding_dim',
                'other_dense_units', 'dense_1_units', 'learning_rate',
                'embedding_dropout', 'dense_dropout', 'l2_reg',
                'weight_decay', 'label_smoothing', 'mask_rate',
                'cancer_class_weight', 'batch_size']}
        })

        print(f"\n>>> Trial {trial.trial_id} ended  |  Threshold: {threshold:.4f}  |  Sensitivity: {sens:.2%}  |  Specificity: {spec:.2%}  |  Score: {score:.4f}")

        if score > self.best_score:
            self.best_score = score
            self.best_sens = sens
            self.best_spec = spec
            self.best_threshold = threshold
            self.best_trial_params = self.all_trial_results[-1]['model_params']

        print(f">>> Best so far  |  Threshold: {self.best_threshold:.4f}  |  Sensitivity: {self.best_sens:.2%}  |  Specificity: {self.best_spec:.2%}  |  Score: {self.best_score:.4f}")
        for k, v in self.best_trial_params.items():
            print(f">>>   {k}: {v}")
        print()


def run_tuning(config):
    """Run hyperparameter tuning with both data-level and model-level params for dual-branch model."""
    enable_better_reproducibility(config)
    check_gpus(config)

    data_param_grid = {
        'min_code_frequency': [30],
        'MAX_SNOMED_SEQ_LENGTH': [config['MAX_SNOMED_SEQ_LENGTH']],
        'MAX_MED_SEQ_LENGTH': [config['MAX_MED_SEQ_LENGTH']],
        'sensitivity_weight': [1.0],
    }
    data_param_names = list(data_param_grid.keys())
    data_combos = list(itertools.product(*data_param_grid.values()))

    print(f"Data-level param combinations: {len(data_combos)}")
    print(f"Model-level trials per combo: {config.get('tuner_max_trials', 30)}")
    print(f"Total trials: {len(data_combos) * config.get('tuner_max_trials', 30)}")

    print("\n>>> Loading and preprocessing data (once for all combos)...")
    common_data = prepare_data_common(config)
    print(">>> Common data ready.\n")

    use_floor = bool(config.get('use_sensitivity_floor', False))
    floor = float(config.get('sensitivity_floor', 0.95))
    global_best_score = -1
    global_best_result = {}
    all_results = []
    all_tuning_models = []    # per-combo top-K (model, trial_info, data, data_params)
    data_cache = {}

    from sklearn.metrics import confusion_matrix as sk_confusion_matrix
    from sklearn.metrics import roc_curve

    for combo_idx, combo in enumerate(data_combos):
        data_params = dict(zip(data_param_names, combo))
        print(f"\n{'='*80}")
        print(f"Data combo {combo_idx+1}/{len(data_combos)}: {data_params}")
        print(f"{'='*80}")

        trial_config = config.copy()
        for k, v in data_params.items():
            trial_config[k] = v

        sw = trial_config['sensitivity_weight']
        spw = trial_config.get('specificity_weight', 1.0)
        trial_config['optimizer_metric'] = WeightedSensSpec(
            name='weighted_sens_spec', sensitivity_weight=sw, specificity_weight=spw)

        cache_key = (data_params['min_code_frequency'],
                     data_params['MAX_SNOMED_SEQ_LENGTH'],
                     data_params['MAX_MED_SEQ_LENGTH'])
        if cache_key in data_cache:
            print(f">>> Reusing cached data for {cache_key}")
            data = data_cache[cache_key]
        else:
            print(f">>> Processing data for {cache_key}...")
            try:
                data = prepare_data_variant(common_data, trial_config)
                data_cache[cache_key] = data
            except Exception as e:
                print(f"  Data pipeline failed: {e}")
                continue

        max_snomed_seq = data['max_snomed_seq_length']
        max_med_seq = data['max_med_seq_length']
        n_other = data['number_of_other_features']
        snomed_vocab = data['vocabulary_size']
        med_vocab = data['med_vocabulary_size']
        monitor_metric = trial_config.get('monitor_metric', 'val_weighted_sens_spec')

        def model_builder(hp):
            model_config = trial_config.copy()
            # ── v136 tuning ranges ──
            # Shifted from v134 in the directions the v134 Rank 3 winner saturated.
            # Rank 3 hit upper bounds on mask_rate / embedding_dropout / dense_dropout,
            # lower bound on label_smoothing, and upper third on l2_reg. Each axis
            # below extends in the direction Rank 3 was pulled, with the new range
            # still containing Rank 3's value so the BO can reproduce it if optimal.
            model_config['embedding_dim'] = hp.Choice('embedding_dim', [16, 24, 32])
            model_config['med_embedding_dim'] = hp.Choice('med_embedding_dim', [8, 16, 24])
            model_config['snomed_lstm_units'] = hp.Choice('snomed_lstm_units', [32, 48, 64, 96])
            model_config['med_lstm_units'] = hp.Choice('med_lstm_units', [16, 24, 32])
            model_config['other_dense_units'] = hp.Choice('other_dense_units', [8, 16, 32])              # +32 (Rank 3 hit upper [8,16])
            model_config['dense_1_units'] = hp.Choice('dense_1_units', [16, 32, 48, 64])
            model_config['learning_rate'] = hp.Float('learning_rate', 5e-5, 5e-4, sampling='log')        # v136 revision: keep v134's range. v135 showed best_epoch=2 at LR=2.7e-4 — higher LR risks noisier explainability.
            model_config['embedding_dropout'] = hp.Float('embedding_dropout', 0.30, 0.55, step=0.05)     # 0.20–0.45 → 0.30–0.55
            model_config['dense_dropout'] = hp.Float('dense_dropout', 0.40, 0.65, step=0.05)             # 0.30–0.55 → 0.40–0.65
            model_config['l2_reg'] = hp.Float('l2_reg', 1e-3, 1e-2, sampling='log')                      # 1e-4–5e-3 → 1e-3–1e-2 (Rank 3 was 3.5e-3)
            model_config['weight_decay'] = hp.Float('weight_decay', 5e-4, 1e-2, sampling='log')          # unchanged (Rank 3 was 1.6e-3 mid)
            model_config['label_smoothing'] = hp.Float('label_smoothing', 0.0, 0.15, step=0.05)          # 0.05–0.30 → 0.0–0.15 (Rank 3 hit lower bound)
            model_config['mask_rate'] = hp.Float('mask_rate', 0.30, 0.55, step=0.05)                     # 0.10–0.40 → 0.30–0.55 (Rank 3 hit upper bound)
            model_config['use_focal_loss'] = False
            model_config['optimizer_metric'] = WeightedSensSpec(
                name='weighted_sens_spec', sensitivity_weight=sw, specificity_weight=spw)
            return build_model(
                model_config, snomed_vocab, med_vocabulary_size=med_vocab,
                max_snomed_seq_length=max_snomed_seq,
                max_med_seq_length=max_med_seq,
                number_of_other_features=n_other,
            )

        val_data = (
            model_inputs(trial_config,
                         X_snomed=data['X_snomed_ids_val'],
                         X_med=data['X_med_ids_val'],
                         X_other=data['X_other_features_val']),
            data['y_val']
        )

        project_name = (f"{trial_config.get('tuner_project_name', 'tuning')}"
                        f"_mcf{data_params['min_code_frequency']}"
                        f"_snomedseq{data_params['MAX_SNOMED_SEQ_LENGTH']}"
                        f"_medseq{data_params['MAX_MED_SEQ_LENGTH']}"
                        f"_sw{data_params['sensitivity_weight']}")

        tuner = SensSpecTuner(
            model_builder,
            objective=kt.Objective(monitor_metric, direction='max'),
            max_trials=trial_config.get('tuner_max_trials', 30),
            executions_per_trial=1,
            directory='tuner_results',
            project_name=project_name,
            overwrite=True,
            validation_data=val_data,
            sensitivity_weight=sw,
            specificity_weight=spw,
            use_sensitivity_floor=use_floor,
            sensitivity_floor=floor,
        )

        callbacks = [
            EarlyStopping(monitor=monitor_metric, mode='max',
                          patience=trial_config.get('early_stopping_patience', 7),
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor=monitor_metric, mode='max', factor=0.5,
                              patience=trial_config.get('lr_reduction_patience', 2),
                              min_lr=1e-6, verbose=1)
        ]

        tuner.search(
            model_inputs(trial_config,
                         X_snomed=data['X_snomed_ids_train'],
                         X_med=data['X_med_ids_train'],
                         X_other=data['X_other_features_train']),
            data['y_train'],
            epochs=trial_config.get('tuner_epochs', 20),
            validation_data=val_data,
            callbacks=callbacks,
            verbose=1
        )

        for tr in tuner.all_trial_results:
            tr_with_data = tr.copy()
            tr_with_data['data_params'] = data_params.copy()
            all_results.append(tr_with_data)

        # --- v111: collect top-K trained models from this combo for end-of-tuning rating ---
        if bool(config.get('rate_top_tuning_trials', True)):
            k = min(int(config.get('tuning_top_n_rate', 10)),
                    len(tuner.all_trial_results))
            combo_sorted = sorted(
                tuner.all_trial_results, key=lambda x: x['score'], reverse=True)[:k]
            for tr in combo_sorted:
                try:
                    trial_obj = tuner.oracle.get_trial(tr['trial_id'])
                    model_obj = tuner.load_model(trial_obj)
                    all_tuning_models.append({
                        'model': model_obj,
                        'trial_info': tr,
                        'data_params': dict(data_params),
                        'data': data,
                    })
                except Exception as _load_err:
                    print(f'  [could not load model for trial '
                          f'{tr.get("trial_id", "?")}: {_load_err!r}]')

        best_model = tuner.get_best_models(num_models=1)[0]
        best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

        y_val_proba = best_model.predict(val_data[0], verbose=0)
        fpr, tpr, thresholds = roc_curve(data['y_val'], y_val_proba)

        # same threshold-picking logic as tuner
        sens_arr = tpr; spec_arr = 1 - fpr
        if use_floor:
            eligible = sens_arr >= floor
            if np.any(eligible):
                idxs = np.where(eligible)[0]
                optimal_idx = idxs[int(np.argmax(spec_arr[idxs]))]
            else:
                optimal_idx = int(np.argmax(sens_arr))
        else:
            scores_curve = (sw * sens_arr + spw * spec_arr) / (sw + spw)
            optimal_idx = int(np.argmax(scores_curve))
        threshold = thresholds[optimal_idx]

        y_val_pred = (y_val_proba.ravel() >= threshold).astype(int)
        cm = sk_confusion_matrix(data['y_val'], y_val_pred, labels=[0, 1])
        TN, FP, FN, TP = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
        sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        score = spec if use_floor and sens >= floor else (sw * sens + spw * spec) / (sw + spw)

        model_params = {key: best_hp.get(key) for key in [
            'embedding_dim', 'snomed_lstm_units', 'med_lstm_units', 'med_embedding_dim',
            'other_dense_units', 'dense_1_units', 'learning_rate',
            'embedding_dropout', 'dense_dropout', 'l2_reg',
            'cancer_class_weight', 'batch_size']}

        print(f"\n>>> Combo {combo_idx+1} best  |  Threshold: {threshold:.4f}  |  Sensitivity: {sens:.2%}  |  Specificity: {spec:.2%}  |  Score: {score:.4f}")
        print(f">>>   Data params: {data_params}")
        print(f">>>   Model params: {model_params}")

        if score > global_best_score:
            global_best_score = score
            global_best_result = {
                'data_params': data_params, 'model_params': model_params,
                'threshold': threshold, 'sensitivity': sens, 'specificity': spec, 'score': score
            }

        print(f"\n>>> GLOBAL BEST  |  Score: {global_best_score:.4f}  |  Sensitivity: {global_best_result.get('sensitivity', 0):.2%}  |  Specificity: {global_best_result.get('specificity', 0):.2%}")
        print(f">>>   Data: {global_best_result.get('data_params', {})}")
        print(f">>>   Model: {global_best_result.get('model_params', {})}")

    all_results_sorted = sorted(all_results, key=lambda r: r['score'], reverse=True)
    top_n = min(10, len(all_results_sorted))

    print('\n' + '='*80)
    print(f'TUNING COMPLETE — TOP {top_n} RESULTS')
    print('='*80)

    for rank, result in enumerate(all_results_sorted[:top_n]):
        print(f"\n--- Rank {rank+1}  |  Threshold: {result['threshold']:.4f}  |  Sensitivity: {result['sensitivity']:.2%}  |  Specificity: {result['specificity']:.2%}  |  Score: {result['score']:.4f} ---")
        print(f"  Data params:")
        for k, v in result['data_params'].items():
            print(f"    {k}: {v}")
        print(f"  Model params:")
        for k, v in result['model_params'].items():
            print(f"    {k}: {v}")

    # --- v111: rate the global top-N tuning trials (uses evaluate_trial_explainability from the explainability cell) ---
    if bool(config.get('rate_top_tuning_trials', True)) and all_tuning_models:
        top_n_rate = min(int(config.get('tuning_top_n_rate', 10)), len(all_tuning_models))
        ranked = sorted(all_tuning_models,
                        key=lambda r: r['trial_info']['score'], reverse=True)[:top_n_rate]

        print('\n' + '=' * 80)
        print(f'EXPLAINABILITY RATING — TOP {top_n_rate} TUNING TRIALS')
        print('=' * 80)

        trial_ratings = []
        for rank, rec in enumerate(ranked):
            info = rec['trial_info']
            trial_cfg = dict(config)
            trial_cfg.update(rec['data_params'])
            trial_cfg.update(info.get('model_params', {}))
            trial_cfg['rate_explainability'] = True

            print(f'\n--- Rank {rank+1}  trial_id={info.get("trial_id")}  '
                  f'sens={info["sensitivity"]:.2%}  spec={info["specificity"]:.2%}  '
                  f'score={info["score"]:.4f}  threshold={info["threshold"]:.4f} ---')
            try:
                res = evaluate_trial_explainability(rec['model'], rec['data'], trial_cfg)
                if res is not None:
                    trial_ratings.append({
                        'rank': rank + 1,
                        'trial_id': info.get('trial_id'),
                        'sensitivity': info['sensitivity'],
                        'specificity': info['specificity'],
                        'score': info['score'],
                        'threshold': info['threshold'],
                        'rating': res['rating_0_100'],
                        'faith': res['faithfulness']['score'],
                        'relev': res['clinical_relevance']['score'],
                        'cons': res['consistency']['score'],
                        'matched': res['clinical_relevance'].get('matched_labels', [])[:3],
                    })
            except Exception as _rate_err:
                print(f'  [rating skipped: {_rate_err!r}]')

        if trial_ratings:
            print('\n' + '=' * 80)
            print('EXPLAINABILITY RATING SUMMARY — TOP TUNING TRIALS')
            print('=' * 80)
            print(f'{"Rank":<5}{"Trial":<14}{"Sens":<9}{"Spec":<9}{"Thr":<8}'
                  f'{"Score":<9}{"Rating":<9}{"Faith":<8}{"Relev":<8}{"Cons":<8}  Top-3 matched')
            print('-' * 110)
            for r in trial_ratings:
                matched_preview = ', '.join(r['matched'])[:40] if r['matched'] else '(none)'
                print(f'{r["rank"]:<5}{str(r["trial_id"])[:12]:<14}'
                      f'{r["sensitivity"]:<9.2%}{r["specificity"]:<9.2%}'
                      f'{r["threshold"]:<8.4f}{r["score"]:<9.4f}'
                      f'{r["rating"]:<9.1f}{r["faith"]:<8.3f}{r["relev"]:<8.3f}{r["cons"]:<8.3f}'
                      f'  {matched_preview}')

    return all_results_sorted


### Run

In [ ]:
# run(config) — uses shared prepare_data() and build_model() — dual-branch.
def run(config):

    enable_better_reproducibility(config)
    check_gpus(config)

    data = prepare_data(config)
    # ── Optionally load saved artifacts (predict-only reproducibility) ──
    # If config['load_saved_artifacts']=True and artifact file exists, override
    # the rebuilt encoder/threshold with the SAVED ones from training time.
    # This guarantees inference matches the original training run regardless of
    # machine/library/BQ-data drift.
    if config.get('load_saved_artifacts', False):
        try:
            import joblib as _joblib
            _artifacts_path = config['saved_model_path'].replace('.keras', '_artifacts.joblib')
            _saved = _joblib.load(_artifacts_path)
            print(f"[artifacts] LOADING saved encoder + threshold from: {_artifacts_path}")
            # Override data dict's encoders with saved ones
            data['snomed_id_to_int'] = _saved['snomed_id_to_int']
            data['med_id_to_int']    = _saved['med_id_to_int']
            data['snomed_vocab_size'] = _saved['snomed_vocab_size']
            data['med_vocab_size']    = _saved['med_vocab_size']
            # Override threshold so evaluate_model uses saved one
            config['proba_threshold'] = _saved['threshold']
            print(f"[artifacts] threshold restored: {_saved['threshold']:.4f}")
            print(f"[artifacts] snomed vocab: {len(_saved['snomed_id_to_int'])}, med vocab: {len(_saved['med_id_to_int'])}")
        except Exception as _e:
            print(f"[artifacts] WARN: load_saved_artifacts=True but failed to load ({_e})")

    df = data['df']
    df_filtered = data['df_filtered']
    df_encoded = data['df_encoded']
    df_mapped = data['df_mapped']
    other_feature_cols = data['other_feature_cols']
    snomed_id_to_int = data['snomed_id_to_int']
    med_id_to_int = data['med_id_to_int']
    patient_ids = data['patient_ids']
    snomed_id_sequences = data['snomed_id_sequences']
    med_id_sequences = data['med_id_sequences']
    other_feature_sequences = data['other_feature_sequences']
    y_patients = data['y_patients']
    X_snomed_ids_padded = data['X_snomed_ids_padded']
    X_med_ids_padded = data['X_med_ids_padded']
    X_other_features_padded = data['X_other_features_padded']
    patient_ids_train = data['patient_ids_train']
    patient_ids_val = data['patient_ids_val']
    patient_ids_test = data['patient_ids_test']
    X_snomed_ids_train = data['X_snomed_ids_train']
    X_snomed_ids_val = data['X_snomed_ids_val']
    X_snomed_ids_test = data['X_snomed_ids_test']
    X_med_ids_train = data['X_med_ids_train']
    X_med_ids_val = data['X_med_ids_val']
    X_med_ids_test = data['X_med_ids_test']
    X_other_features_train = data['X_other_features_train']
    X_other_features_val = data['X_other_features_val']
    X_other_features_test = data['X_other_features_test']
    y_train = data['y_train']
    y_val = data['y_val']
    y_test = data['y_test']
    vocabulary_size = data['vocabulary_size']
    med_vocabulary_size = data['med_vocabulary_size']
    max_snomed_seq_length = data['max_snomed_seq_length']
    max_med_seq_length = data['max_med_seq_length']

    # Show sequence length distributions
    seq_distribution(snomed_id_sequences, med_id_sequences, config)

    # Build model
    model = build_model(
        config, vocabulary_size, med_vocabulary_size=med_vocabulary_size,
        X_snomed_ids_padded=X_snomed_ids_padded,
        X_med_ids_padded=X_med_ids_padded,
        X_other_features_padded=X_other_features_padded,
    )

    history = None
    if config['train_model']:
        model, history = train_model(
            model,
            X_snomed_ids_train, X_med_ids_train, X_other_features_train, y_train,
            X_snomed_ids_val, X_med_ids_val, X_other_features_val, y_val,
            config)

    if config['train_model'] and history is not None:
        import matplotlib.pyplot as plt
        metric_name = 'weighted_sens_spec'
        if metric_name in history.history:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            ax1.plot(history.history[metric_name], label=f'Train {metric_name}')
            ax1.plot(history.history[f'val_{metric_name}'], label=f'Val {metric_name}')
            ax1.set_xlabel('Epoch'); ax1.set_ylabel(metric_name)
            ax1.set_title('Training vs Validation Metric'); ax1.legend()
            ax2.plot(history.history['loss'], label='Train loss')
            ax2.plot(history.history['val_loss'], label='Val loss')
            ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
            ax2.set_title('Training vs Validation Loss'); ax2.legend()
            plt.tight_layout(); plt.show()

    if config['save_model']:
        model = save_model(model, config)
    if config['load_model']:
        model = load_model(config)

    if config.get('use_optimal_threshold', False):
        optimal_thresh = find_optimal_threshold(
            model, X_snomed_ids_val, X_med_ids_val, X_other_features_val, y_val, config)
        config['proba_threshold'] = optimal_thresh
        print(f"config['proba_threshold'] set to optimal threshold found = {optimal_thresh}")

    # ── Save artifacts (encoder + threshold + metadata) ──
    # Without these, predict-only re-runs rebuild encoder/threshold and drift.
    # Loading the .keras alone is NOT enough — the model expects specific
    # input encoding (snomed_id_to_int) and operating threshold.
    try:
        import joblib as _joblib
        _artifacts_path = config['saved_model_path'].replace('.keras', '_artifacts.joblib')
        _joblib.dump({
            'snomed_id_to_int':  data.get('snomed_id_to_int', {}),
            'med_id_to_int':     data.get('med_id_to_int', {}),
            'snomed_vocab_size': data.get('snomed_vocab_size', 0),
            'med_vocab_size':    data.get('med_vocab_size', 0),
            'max_snomed_seq':    config.get('max_snomed_seq_length', 50),
            'max_med_seq':       config.get('max_med_seq_length', 30),
            'threshold':         float(config.get('proba_threshold', 0.5)),
            'sensitivity_floor': float(config.get('sensitivity_floor', 0.95)),
            'use_med_branch':    bool(config.get('use_med_branch', False)),
            'use_snomed_branch': bool(config.get('use_snomed_branch', True)),
            'other_feature_cols': list(other_feature_cols) if 'other_feature_cols' in dir() else [],
            'patient_ids_test':  list(patient_ids_test),
            'data_version':      config.get('data_version', ''),
            'code_version':      config.get('code_version', ''),
            'min_code_frequency': config.get('min_code_frequency', 5),
            'min_snomed_seq':    config.get('min_snomed_seq', 0),
            'min_med_seq':       config.get('min_med_seq', 0),
        }, _artifacts_path)
        print(f"[artifacts] saved encoder + threshold to: {_artifacts_path}")
    except Exception as _e:
        print(f"[artifacts] WARN: failed to save artifacts ({_e})")

    evaluate_model(model, X_snomed_ids_test, X_med_ids_test, X_other_features_test, y_test, config)
    
    if config['individual_verification']:
        patient_id = patient_ids_test[config['selected_patient_id_test_index']]
        predict_cancer_for_patient(model, patient_id, other_feature_cols,
                                   max_snomed_seq_length, max_med_seq_length,
                                   df_mapped, config)
        verify_results_on_a_subset_of_test_data(
            model, patient_ids_test,
            X_snomed_ids_test, X_med_ids_test, X_other_features_test, y_test,
            other_feature_cols, max_snomed_seq_length, max_med_seq_length,
            df_mapped, config)

    if config['show_demographics']:
        show_train_test_demographics(df, patient_ids_train, patient_ids_test, config)

    if config.get('run_explainability', False):
        run_explainability(model, data, config)

    return data


## Cancer Profiles

In [ ]:
# Per-cancer configuration profiles. 
# A new cancer type can be added by adding entries to the four dicts below (`CONCEPT_CODELIST_PATHS`,
# `RELEVANCE_PATTERNS_BY_CANCER`, `SEMANTIC_REFERENCE_PHRASES`,
# `SHORTCUT_TEXT_PATTERNS`) plus `CANCER_FILE_SLUGS` for path strings.

# ═══════════════════════════════════════════════════════════════════════════
# Cancer Profiles — per-cancer configuration source-of-truth.
# Single dict keyed by cancer type.  Add a new cancer by appending one entry.
# The Config section below reads the active profile via
# `config.update(CANCER_PROFILES[TARGET_CANCER])` and derives DATA_FILE /
# saved_model_path / tuner_project_name from TARGET_CANCER + data_version +
# code_version (see Config cell).
# ═══════════════════════════════════════════════════════════════════════════

CANCER_PROFILES = {
    'breast': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            'data/codelists/breast_cancer_relevance/breast_cancer_diagnosis_filtered.csv',
            'data/codelists/breast_cancer_relevance/breast_cancer_screening.csv',
        ],
        'relevance_patterns':         [
    # Breast anatomy & disease subtypes
    r'\bbreast', r'mammar', r'nipple', r'areola',
    r'\bductal', r'\blobular', r'\bdcis\b', r'\blcis\b',
    r'paget.*breast', r"paget.?s.*breast", r'phyllodes',
    r'invasive.*(?:ductal|lobular)', r'carcinoma.*in\s*situ',
    r'inflammatory.*breast', r'triple[-\s]?negative',
    r'hormone[-\s]?(?:receptor|positive|negative)',
    r'oestrogen.*receptor', r'estrogen.*receptor', r'\ber\s*positive',
    r'progesterone.*receptor', r'\bpr\s*positive',
    r'\bher2', r'her[-\s]?2\s*positive', r'her[-\s]?2\s*negative',

    # Symptoms (NICE NG12)
    r'breast.*(?:lump|mass|swelling|pain|tenderness|asymmetr|change|deformity)',
    r'nipple.*(?:discharge|retraction|inversion|eczema|rash|bleeding)',
    r'skin.*(?:dimpling|tethering|erythema.*breast)',
    r'peau.*d.orange',
    r'axillary.*(?:lump|mass|lymphadenopathy|node)',
    r'supraclav.*(?:node|lymphadenopathy)',

    # Investigations
    r'mammogr', r'tomosynthesis',
    r'breast.*(?:ultrasound|\bus\b|sonograph)',
    r'breast.*(?:mri|magnetic resonance)',
    r'ductogram', r'galactogram',
    r'fine[-\s]?needle.*aspir', r'\bfna\b',
    r'core.*biopsy', r'biopsy.*breast', r'breast.*biopsy',
    r'vacuum[-\s]?assisted.*biopsy', r'\bvab\b',
    r'sentinel.*node', r'axillary.*(?:clearance|dissection)',
    r'breast.*clip', r'breast.*wire.*localis',
    r'histopath',

    # Endocrine and targeted therapies
    r'tamoxifen', r'anastrozole', r'letrozole', r'exemestane',
    r'aromatase.*inhibitor', r'\bai\s*therapy',
    r'fulvestrant', r'zoladex', r'goserelin', r'leuprolide', r'triptorelin',
    r'lhrh.*agonist', r'ovarian.*supp',
    r'trastuzumab', r'herceptin', r'pertuzumab', r'perjeta',
    r'\bt-dm1\b', r'trastuzumab.*emtansine', r'kadcyla',
    r'palbociclib', r'ribociclib', r'abemaciclib', r'cdk4',
    r'olaparib', r'talazoparib', r'parp.*inhibitor',
    r'neoadjuvant.*(?:chemo|therapy)', r'adjuvant.*(?:endocrine|chemo|hormonal)',

    # Family history / risk factors
    r'\bbrca\b', r'brca1', r'brca2', r'hereditary.*breast',
    r'family.*history.*(?:breast|ovarian|brca)',
    r'\bpalb2', r'\btp53\b.*breast',
    r'li[-\s]?fraumeni',
    r'hormone[-\s]?replac', r'\bhrt\b',
    r'nulliparous', r'nulliparity', r'menarche', r'menopause',
    r'oral.*contracep',

    # Screening programme
    r'breast.*screen', r'nhs.*breast.*screen', r'screening.*mammogr',
],
        'semantic_reference_phrases': [
            'breast cancer',
            'malignant neoplasm of breast',
            'ductal carcinoma',
            'invasive ductal carcinoma',
            'lobular carcinoma',
            'invasive lobular carcinoma',
            'ductal carcinoma in situ',
            'breast adenocarcinoma',
            'triple-negative breast cancer',
            'HER2-positive breast cancer',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening (still off-topic for breast)
    r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
    r'faecal.*occult', r'\bfobt\b',
    r'cervical.*smear', r'cervical.*screen',
    r'\bpsa\b', r'prostate.*screen',
    r'aaa.*screen', r'abdominal.*aortic.*screen',
    # Lung-specific (off-topic for breast)
    r'bronchoscop', r'spirom', r'peak.*flow', r'\bcxr\b',
    r'chest.*x[-\s]?ray', r'ct.*thorax', r'pulmonary.*function',
    # Generic admin / utilisation proxies
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations unlikely to be breast-signal
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },
    'lung': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            'data/codelists/lung_cancer_relevance/lung_cancer_snomed.csv',
            'data/codelists/lung_cancer_relevance/chronic_respiratory_disease.csv',
            'data/codelists/lung_cancer_relevance/smoking_clear.csv',
        ],
        'relevance_patterns':         [
    # Lung / respiratory anatomy & disease
    r'respirat', r'\bcopd\b', r'asthma', r'pulmonary', r'bronch',
    r'pneumon', r'\blung\b', r'\bthorax', r'\bchest', r'airway',
    r'tracheo', r'mediastin', r'pleura', r'\bnodule',
    r'emphysema', r'fibros(?:is|ing).*lung', r'interstitial.*lung',
    r'silicosis', r'asbestos', r'mesothelioma', r'carcinoid',

    # Smoking & tobacco (risk factor)
    r'\bsmok', r'cigarette', r'tobacco', r'nicotin', r'\bvape',
    r'pack[-\s]*year',

    # NICE NG12 symptoms
    r'dyspn', r'\bcough\b', r'wheez', r'haemoptys', r'hemoptys',
    r'short(?:ness)?\s*of\s*breath', r'breathless',
    r'weight\s*loss', r'loss of appetite', r'appetite.*(?:loss|poor)',
    r'finger.*club', r'clubbing', r'hoars', r'stridor',
    r'recurrent.*(?:chest|respiratory).*infection',
    r'supraclav.*(?:node|lymphadenopathy)',

    # Lung-targeted investigations
    r'spirom', r'peak\s*flow', r'pulmon(?:ary)?\s*function',
    r'\bpft\b', r'lung\s*function', r'pefr', r'\bfev\b', r'\bfvc\b',
    r'bronchoscop', r'\binhal',
    r'ct\s*(?:thorax|chest)', r'chest\s*x[-\s]?ray', r'\bcxr\b',
    r'chest.*radiograph', r'sputum', r'histopath',
    r'biopsy.*(?:lung|bronch)', r'lung.*biopsy',
    r'pleural.*(?:effusion|fluid|tap)', r'thoracen',
    r'thoracosc', r'mediastinosc',
    r'pet[-\s]?ct', r'\bpet\s*scan',
    r'oxygen sat', r'\bsao2\b', r'\bspo2\b',

    # Respiratory medications & smoking-cessation
    r'\bent\b', r'inhaler', r'bronchodil',
    r'\bsaba\b', r'\blaba\b', r'\blama\b', r'\bics\b',
    r'salbutamol', r'terbutaline', r'ipratropium', r'tiotropium',
    r'formoterol', r'salmeterol', r'budesonide', r'beclomet',
    r'fluticason', r'prednisol', r'oral\s*steroid',
    r'varenicline', r'bupropion', r'nicotine\s*replac',

    # Family history / risk factors
    r'family\s*history.*(?:lung|bronch|respirat|trachea|pulmon)',
    r'occupation.*(?:asbest|silica|diesel|radon)',
    r'radon', r'occupational.*carcinog',
    r'prior.*(?:thoracic|chest).*radiotherapy',
],
        'semantic_reference_phrases': [
            'lung cancer',
            'malignant neoplasm of lung',
            'bronchial carcinoma',
            'pulmonary neoplasm',
            'small cell lung cancer',
            'non-small cell lung cancer',
            'respiratory tract carcinoma',
            'lung adenocarcinoma',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening (off-topic for lung)
    r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
    r'faecal.*occult', r'\bfobt\b',
    r'cervical.*smear', r'cervical.*screen',
    r'mammogr', r'breast.*screen',
    r'\bpsa\b', r'prostate.*screen',
    # Generic admin / utilisation proxies
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations unlikely to be lung-signal
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },
    'prostate': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            'data/codelists/prostate_cancer_relevance/prostate_cancer_snomed.csv',
            'data/codelists/prostate_cancer_relevance/benign_prostatic_hyperplasia.csv',
        ],
        'relevance_patterns':         [
    # Anatomy & disease subtypes
    r'\bprostate', r'prostatic', r'\bprostatic\s+gland',
    r'transition\s*zone', r'periphera(?:l)?\s*zone',
    r'adenocarcinoma.*prostate', r'prostate.*adenocarcinoma',
    r'\bgleason', r'\bisup\b', r'castrate.*resistant', r'\bcrpc\b',
    r'metastatic.*prostate', r'androgen[-\s]?dependent',

    # NICE NG12 symptoms (LUTS + advanced)
    r'lower\s*urinary\s*tract', r'\bluts\b',
    r'urinary\s*(?:hesitanc|frequenc|urgenc|retent|incontin)',
    r'nocturia', r'weak\s*(?:stream|flow)', r'poor\s*(?:stream|flow)',
    r'dribbl', r'strangury', r'\bdysuria\b',
    r'haematuria', r'hematuria', r'blood.*urine',
    r'erectile\s*dysfunc', r'\bed\b.*sexual',
    r'bone\s*pain', r'back\s*pain.*(?:bone|metasta)',
    r'pelvic\s*pain',

    # Investigations
    r'\bpsa\b', r'prostate[-\s]?specific\s*antigen',
    r'free.*psa', r'psa.*velocity', r'psa.*density',
    r'digital\s*rectal', r'\bdre\b',
    r'prostate.*biops', r'transrectal.*ultrasound', r'\btrus\b',
    r'multiparametric.*mri', r'\bmpmri\b', r'mri.*prostate', r'prostate.*mri',
    r'\bpirads\b', r'pi[-\s]?rads',
    r'bone\s*scan', r'isotope.*bone', r'pet[-\s]?(?:psma|choline)',
    r'transperineal.*biops', r'\btemplate.*biops',
    r'histopath',

    # Treatments
    r'radical.*prostatect', r'prostatect',
    r'brachythera', r'external\s*beam.*radio',
    r'androgen.*deprivation', r'\badt\b',
    r'goserelin', r'leuprolide', r'leuprorel', r'triptorelin',
    r'bicalutamide', r'flutamide', r'cyproterone',
    r'abiraterone', r'enzalutamide', r'apalutamide', r'darolutamide',
    r'docetaxel', r'cabazitaxel',
    r'radium[-\s]?223', r'lutetium[-\s]?177',
    r'active\s*surveillance', r'watchful\s*waiting',

    # Family history / risk factors
    r'family.*history.*prostate',
    r'\bbrca2\b', r'hereditary.*prostate',
    r'\bhoxb13', r'lynch.*syndrome',
    r'african.*caribbean', r'african\s*ancestry',
],
        'semantic_reference_phrases': [
            'prostate cancer',
            'malignant neoplasm of prostate',
            'prostate adenocarcinoma',
            'castration-resistant prostate cancer',
            'metastatic prostate cancer',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening (off-topic for prostate)
    r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
    r'faecal.*occult', r'\bfobt\b', r'\bfit\s*test',
    r'cervical.*smear', r'cervical.*screen',
    r'mammogr', r'breast.*screen',
    # Lung screening codes
    r'spirom', r'peak.*flow', r'chest.*x[-\s]?ray', r'\bcxr\b',
    # Generic admin / utilisation proxies
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations unlikely to be prostate-signal
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },

    'colorectal': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            'data/codelists/colorectal_cancer_relevance/colorectal_cancer_snomed.csv',
            'data/codelists/colorectal_cancer_relevance/inflammatory_bowel_disease.csv',
            'data/codelists/colorectal_cancer_relevance/bowel_cancer_screening.csv',
            'data/codelists/colorectal_cancer_relevance/colorectal_symptoms.csv',
        ],
        'relevance_patterns':         [
    # Anatomy & disease subtypes
    r'\bcolon\b', r'\brectum\b', r'rectal', r'colorectal',
    r'sigmoid', r'caecum', r'cec(?:um|al)', r'ascending.*colon',
    r'descending.*colon', r'transverse.*colon', r'splenic.*flexure',
    r'hepatic.*flexure', r'\banus\b', r'anal\s*canal',
    r'bowel.*(?:cancer|tumour|tumor|malignan|neoplasm)',
    r'colon.*(?:cancer|adenocarcinoma|adenoma)', r'rectal.*cancer',
    r'\bpolyp', r'adenomatous.*polyp', r'tubular.*adenom',
    r'villous.*adenom', r'serrated.*adenom',

    # NICE NG12 symptoms
    r'rectal.*bleed', r'\bpr\s*bleed', r'blood.*stool', r'haematochez',
    r'melaena', r'melena', r'\boccult\s*blood',
    r'change.*bowel.*habit', r'\bbowel.*habit',
    r'persistent.*diarrhoea', r'persistent.*constipation',
    r'\btenesmus\b',
    r'iron[-\s]?deficien', r'\bida\b', r'microcytic.*anaem',
    r'abdominal.*(?:mass|pain.*lower|distension)',
    r'right.*iliac.*(?:mass|fossa)',
    r'weight\s*loss', r'unintentional.*weight',
    r'(?:rectal|abdomin).*examination',

    # Investigations
    r'colonoscop', r'flexible.*sigmoidoscop', r'\bsigmoidoscop',
    r'ct.*colonograph', r'virtual.*colonoscop', r'barium.*enema',
    r'faecal.*occult', r'\bfobt\b', r'\bfit\s*test', r'\bfaecal\s*immuno',
    r'\bcea\b', r'carcinoembryonic',
    r'colon.*biops', r'rectal.*biops', r'polyp.*biops',
    r'bowel.*screen', r'colorectal.*screen',
    r'pet[-\s]?ct', r'liver.*mri', r'mri.*rectum',
    r'histopath',

    # Treatments
    r'\bcolectom', r'hemicolectom', r'sigmoidectom',
    r'anterior.*resection', r'abdominoperin', r'\bapr\b',
    r'\bhartmann', r'low.*anterior',
    r'colostom', r'ileostom', r'stoma',
    r'5[-\s]?fluorouracil', r'\b5[-\s]?fu\b', r'capecitab',
    r'oxaliplatin', r'irinotecan', r'leucovorin',
    r'folfox', r'folfiri', r'\bcapox\b', r'\bxelox\b',
    r'cetuximab', r'panitumumab', r'bevacizumab',
    r'\btme\b', r'total.*mesorectal',

    # Family history / risk factors
    r'family.*history.*(?:bowel|colon|rectal|colorectal)',
    r'\blynch\b', r'hnpcc', r'\bfap\b', r'familial.*adenomatous',
    r'peutz[-\s]?jegher', r'mlh1', r'msh2', r'msh6',
    r'inflammatory.*bowel', r'ulcerative.*colit', r'crohn',
    r'history.*polyps',
],
        'semantic_reference_phrases': [
            'colorectal cancer',
            'colon cancer',
            'rectal cancer',
            'malignant neoplasm of colon',
            'malignant neoplasm of rectum',
            'colorectal adenocarcinoma',
            'bowel cancer',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening (off-topic for colorectal — keep bowel screen as in-topic!)
    r'cervical.*smear', r'cervical.*screen',
    r'mammogr', r'breast.*screen',
    r'\bpsa\b', r'prostate.*screen',
    # Lung screening codes
    r'spirom', r'peak.*flow', r'chest.*x[-\s]?ray', r'\bcxr\b',
    # Generic admin / utilisation proxies
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },

    'lymphoma': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            'data/codelists/lymphoma_relevance/lymphoma_snomed.csv',
            'data/codelists/lymphoma_relevance/immunosuppression.csv',
        ],
        'relevance_patterns':         [
    # Anatomy & disease subtypes
    r'\blymphoma', r'hodgkin', r'non[-\s]?hodgkin', r'\bnhl\b',
    r'\bdlbcl\b', r'diffuse.*large.*b[-\s]?cell',
    r'follicular.*lymphoma', r'mantle.*cell', r'burkitt',
    r'marginal.*zone', r'\bmalt\b.*lymphoma',
    r'cutaneous.*lymphoma', r'mycosis.*fungoides', r'sezary',
    r't[-\s]?cell.*lymphoma', r'b[-\s]?cell.*lymphoma',
    r'lymph.*node.*(?:malignan|neoplas|tumour|tumor)',
    r'lymphoid.*neoplas', r'lymphoprolif',

    # NICE NG12 symptoms
    r'lymphadenop', r'enlarged.*lymph.*node',
    r'cervical.*(?:node|lymph)', r'axillary.*(?:node|lymphadenop)',
    r'inguinal.*(?:node|lymph)', r'supraclav.*(?:node|lymphadenop)',
    r'mediastin.*(?:mass|lymphadenop|widen)',
    r'night.*sweat', r'\bb[-\s]?symptom',
    r'unexplained.*(?:fever|pyrexia)',
    r'weight\s*loss', r'unintentional.*weight',
    r'pruritus', r'persistent.*itch',
    r'hepatosplen', r'splenomeg', r'hepatomeg',

    # Investigations
    r'lymph.*node.*biops', r'excisional.*biops',
    r'core.*biops.*(?:node|lymph)',
    r'pet[-\s]?ct', r'whole[-\s]?body.*pet',
    r'\bldh\b', r'lactate.*dehydrog',
    r'\besr\b', r'erythrocyte.*sediment',
    r'beta[-\s]?2[-\s]?microglob',
    r'bone.*marrow.*(?:biops|aspir)', r'\btrephine\b',
    r'flow\s*cytometry', r'immunophenotyp',
    r'immunohistoch', r'\bihc\b',
    r'\bhiv\b', r'hepatitis.*screen', r'\bebv\b',
    r'\bhtlv', r'\bcd20\b', r'\bcd30\b',

    # Treatments
    r'\br[-\s]?chop', r'\bchop\b', r'\babvd\b',
    r'\brituximab', r'mabthera',
    r'obinutuzumab', r'brentuximab', r'\bbv\b',
    r'\bbendamust', r'lenalidomide',
    r'methotrexate.*(?:lymphoma|cns)', r'cytarabine',
    r'doxorubicin', r'vincristine', r'cyclophosph',
    r'\bcar[-\s]?t', r'tisagenlec',
    r'stem\s*cell.*transplant', r'autologous.*transplant', r'allo.*transplant',
    r'mantle.*radio', r'involved[-\s]?field.*radio',

    # Family history / risk factors
    r'family.*history.*lymphoma',
    r'immunosup', r'post[-\s]?transplant', r'\bptld\b',
    r'h\.?\s*pylori', r'helicobacter',
    r'\beatorial', r'pesticide',
],
        'semantic_reference_phrases': [
            'lymphoma',
            'Hodgkin lymphoma',
            'non-Hodgkin lymphoma',
            'malignant neoplasm of lymphatic tissue',
            'diffuse large B-cell lymphoma',
            'follicular lymphoma',
            'lymphoid neoplasm',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening (lymphoma has no screening programme)
    r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
    r'faecal.*occult', r'\bfobt\b', r'\bfit\s*test',
    r'cervical.*smear', r'cervical.*screen',
    r'mammogr', r'breast.*screen',
    r'\bpsa\b', r'prostate.*screen',
    # Lung screening
    r'spirom', r'peak.*flow', r'chest.*x[-\s]?ray', r'\bcxr\b',
    # Generic admin
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },

    'melanoma': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            'data/codelists/melanoma_relevance/melanoma_snomed.csv',
        ],
        'relevance_patterns':         [
    # Anatomy & disease subtypes
    r'melanoma', r'malignant.*melanoma', r'melanoma.*in\s*situ',
    r'lentigo.*maligna', r'nodular.*melanoma',
    r'superficial.*spreading.*melanoma', r'acral.*lentig',
    r'subungual.*melanoma', r'desmoplastic.*melanoma',
    r'mucosal.*melanoma', r'ocular.*melanoma', r'uveal.*melanoma',
    r'\bnaev', r'\bnevus\b', r'\bnevi\b',
    r'atypical.*(?:naev|nevus)', r'dysplastic.*(?:naev|nevus)',
    r'pigmented.*(?:lesion|mole|skin)',

    # NICE NG12 symptoms — ABCDE
    r'asymmetr.*(?:mole|naev|lesion)',
    r'irregular.*(?:border|edge|outline)',
    r'changing.*(?:mole|naev|lesion|colour|color)',
    r'mole.*(?:bleed|itch|scab|crust|ulcer|change)',
    r'pigmented.*(?:bleed|chang|growth)',
    r'new.*pigmented.*(?:lesion|mole)',
    r'\babcde\b.*(?:skin|mole|melan)',
    r'mole.*concerning',

    # Investigations
    r'dermatosc', r'dermosc', r'\bderm\b.*scope',
    r'skin.*biops', r'shave.*biops', r'punch.*biops',
    r'excision.*(?:biops|naev|mole|lesion)',
    r'sentinel.*node', r'sentinel.*lymph',
    r'\bldh\b', r'lactate.*dehydrog',
    r'pet[-\s]?ct', r'staging.*(?:ct|imag)',
    r'\bbreslow', r'breslow.*thickness',
    r'\bclark.*level',
    r'histopath',
    r'\bbraf\b', r'\bnras\b', r'\bckit\b', r'mutational.*analys',

    # Treatments
    r'wide.*local.*excision', r'\bwle\b.*(?:skin|melan)',
    r'\binterferon', r'adjuvant.*interferon',
    r'ipilimumab', r'nivolumab', r'pembrolizumab',
    r'\bpd[-\s]?1\b', r'\bctla[-\s]?4\b', r'immune.*checkpoint',
    r'vemurafenib', r'dabrafenib', r'encorafenib',
    r'trametinib', r'cobimetinib', r'binimetinib',
    r'\bmek\s*inhib', r'\bbraf\s*inhib',
    r'imiquimod',
    r'cryother', r'curettage',

    # Family history / risk factors
    r'family.*history.*(?:melanoma|skin.*cancer)',
    r'\bcdkn2a\b', r'\bp16\b.*(?:germ|hered)',
    r'familial.*atypical.*(?:naev|mole)',
    r'\bfamm\b', r'\bbap1\b',
    r'\buv\b.*exposure', r'sunburn', r'sun.*exposure',
    r'tanning.*(?:bed|booth|sunbed)', r'sunbed',
    r'\bfair\s*skin', r'red\s*hair', r'fitzpatrick',
    r'multiple.*(?:naev|nevi|moles)',
    r'immunosup', r'transplant.*(?:skin|melan)',
],
        'semantic_reference_phrases': [
            'melanoma',
            'malignant melanoma of skin',
            'malignant neoplasm of skin',
            'cutaneous melanoma',
            'melanoma in situ',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening
    r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
    r'faecal.*occult', r'\bfobt\b', r'\bfit\s*test',
    r'cervical.*smear', r'cervical.*screen',
    r'mammogr', r'breast.*screen',
    r'\bpsa\b', r'prostate.*screen',
    # Lung screening
    r'spirom', r'peak.*flow', r'chest.*x[-\s]?ray', r'\bcxr\b',
    # Generic admin
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },

    'bladder': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            # No public OpenCodelists SNOMED list found for this cancer.
            # Relevance scoring falls back to the regex patterns below.
        ],
        'relevance_patterns':         [
    # Anatomy & disease subtypes
    r'\bbladder\b', r'urinary.*bladder', r'urotheli',
    r'transitional.*cell.*(?:carcinoma|bladder)', r'\btcc\b',
    r'squamous.*cell.*bladder', r'adenocarcinoma.*bladder',
    r'\burinary.*tract.*(?:cancer|carcinoma|malignan|tumour|tumor)',
    r'\bcis\b.*bladder', r'carcinoma.*in\s*situ.*bladder',
    r'non[-\s]?muscle[-\s]?invasive', r'\bnmibc\b',
    r'muscle[-\s]?invasive.*bladder', r'\bmibc\b',

    # NICE NG12 symptoms
    r'haematuria', r'hematuria', r'\bvisible\s*haemat',
    r'macroscopic.*haematuria', r'painless.*haematuria',
    r'microscopic.*haematuria', r'\bnvh\b', r'\bnv.*haemat',
    r'blood.*urine',
    r'recurrent.*\buti\b', r'recurrent.*urinary.*tract.*infection',
    r'\bdysuria\b', r'urinary\s*frequenc', r'urinary\s*urgenc',
    r'lower\s*urinary\s*tract', r'\bluts\b',
    r'pelvic\s*pain.*persist',
    r'urinary.*retent',

    # Investigations
    r'\bcystoscop', r'flexible.*cystoscop', r'rigid.*cystoscop',
    r'urine.*cytology', r'\bumc\b',
    r'\bturbt\b', r'transurethral.*resect.*bladder',
    r'ct.*urogr', r'\bivu\b', r'intravenous.*urogr',
    r'renal.*tract.*(?:ultrasound|us|sonogr)',
    r'mri.*pelvis', r'mri.*bladder',
    r'urinalys', r'\budip\b', r'urine.*dipstick',
    r'\bnmp22\b', r'urine.*marker',
    r'histopath',

    # Treatments
    r'\bturbt\b', r'\bbcg\b.*(?:bladder|intravesic)',
    r'mitomycin', r'intravesic',
    r'\bcystectom', r'radical.*cystectom', r'partial.*cystectom',
    r'urostom', r'ileal.*conduit', r'neobladder',
    r'cisplatin', r'gemcitab', r'\bmvac\b',
    r'pembrolizumab', r'atezolizumab', r'nivolumab', r'durvalumab',
    r'erdafitinib', r'enfortumab',

    # Family history / risk factors
    r'family.*history.*bladder',
    r'\bsmok', r'cigarette', r'tobacco',
    r'occupation.*(?:dye|aniline|rubber|leather|chemical|paint)',
    r'aromatic.*amine',
    r'schistosom',
    r'cyclophosph.*(?:expos|histor)',
    r'pelvic.*radio.*(?:histor|prior)',
],
        'semantic_reference_phrases': [
            'bladder cancer',
            'malignant neoplasm of bladder',
            'urothelial carcinoma',
            'transitional cell carcinoma of bladder',
            'bladder tumour',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening
    r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
    r'faecal.*occult', r'\bfobt\b', r'\bfit\s*test',
    r'cervical.*smear', r'cervical.*screen',
    r'mammogr', r'breast.*screen',
    r'\bpsa\b', r'prostate.*screen',
    # Lung screening
    r'spirom', r'peak.*flow', r'chest.*x[-\s]?ray', r'\bcxr\b',
    # Generic admin
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },

    'pancreatic': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            'data/codelists/pancreatic_cancer_relevance/chronic_pancreatitis.csv',
            'data/codelists/pancreatic_cancer_relevance/diabetes_mellitus.csv',
        ],
        'relevance_patterns':         [
    # Anatomy & disease subtypes
    r'pancrea(?:s|tic)', r'pancreatic.*(?:cancer|adenocarcinoma|carcinoma)',
    r'\bpdac\b', r'ductal.*adenocarcinoma.*pancrea',
    r'head.*pancrea', r'body.*pancrea', r'tail.*pancrea',
    r'ampulla.*vater', r'periampullary',
    r'pancreatic.*neuroendocrine', r'\bpnet\b', r'islet.*cell',
    r'\bipmn\b', r'intraductal.*papillary',

    # NICE NG12 symptoms
    r'jaundic', r'painless.*jaundic',
    r'icteric', r'sclera.*yellow',
    r'epigastric.*pain', r'abdominal.*pain.*radiat.*back',
    r'back\s*pain.*(?:abdomin|radiat)',
    r'weight\s*loss', r'unintentional.*weight',
    r'anorex', r'loss.*appetite',
    r'new[-\s]?onset.*diabetes', r'diabetes.*new.*diagn',
    r'steatorrh', r'fatty.*stool',
    r'pruritus', r'itch.*(?:jaundic|cholestat)',
    r'courvoisier', r'palpable.*gallbladder',
    r'pale.*stool', r'dark.*urine',

    # Investigations
    r'\bca\s*19[-\s]?9\b', r'ca19[-\s]?9',
    r'ct.*pancrea', r'ct.*abdomen.*pancrea',
    r'mri.*pancrea', r'\bmrcp\b', r'magnetic.*resonance.*cholang',
    r'\beus\b.*(?:panc|abdomin)', r'endoscopic.*ultrasound',
    r'\bercp\b', r'endoscopic.*retrograde',
    r'pancreatic.*biops', r'\bfna\b.*pancrea',
    r'pet[-\s]?ct',
    r'\blfts\b.*deranged', r'\balt\b.*\balp\b', r'bilirubin.*raised',
    r'histopath',

    # Treatments
    r'\bwhipple', r'pancreaticoduoden',
    r'distal.*pancreatect', r'total.*pancreatect',
    r'gemcitab', r'folfirinox', r'nab[-\s]?paclitax',
    r'erlotinib', r'olaparib.*pancrea',
    r'\bstent', r'biliary.*stent', r'plastic.*stent',
    r'pancreatic.*enzyme', r'pancreatin', r'\bcreon\b',
    r'\bppi\b.*(?:pancr|gastric)',

    # Family history / risk factors
    r'family.*history.*(?:pancrea|pancre)',
    r'hereditary.*pancreatitis', r'chronic.*pancreatitis',
    r'\bbrca2\b.*pancrea', r'\bbrca\b.*pancr',
    r'peutz[-\s]?jegher', r'lynch.*syndrome',
    r'\bsmok', r'cigarette', r'tobacco',
    r'\bobes', r'\bbmi\b',
    r'diabetes.*mellitus.*(?:long|chronic)',
],
        'semantic_reference_phrases': [
            'pancreatic cancer',
            'malignant neoplasm of pancreas',
            'pancreatic adenocarcinoma',
            'ductal adenocarcinoma of pancreas',
            'pancreatic neuroendocrine tumour',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening (pancreatic has no screening programme)
    r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
    r'faecal.*occult', r'\bfobt\b', r'\bfit\s*test',
    r'cervical.*smear', r'cervical.*screen',
    r'mammogr', r'breast.*screen',
    r'\bpsa\b', r'prostate.*screen',
    # Lung screening
    r'spirom', r'peak.*flow', r'chest.*x[-\s]?ray', r'\bcxr\b',
    # Generic admin
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },

    'ovarian': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            'data/codelists/ovarian_cancer_relevance/endometriosis.csv',
        ],
        'relevance_patterns':         [
    # Anatomy & disease subtypes
    r'\bovar', r'ovarian', r'ovary', r'fallopian.*tube',
    r'tubo[-\s]?ovarian', r'peritone(?:um|al)',
    r'ovarian.*(?:cancer|carcinoma|malignan|neoplasm)',
    r'epithelial.*ovarian', r'serous.*(?:ovar|carcinoma)',
    r'endometrioid.*(?:ovar|carcinoma)',
    r'mucinous.*(?:ovar|carcinoma)',
    r'clear.*cell.*(?:ovar|carcinoma)',
    r'granulosa.*cell', r'germ.*cell.*ovar',
    r'borderline.*ovar',

    # NICE NG12 symptoms (NICE recommends ALL women with these symptoms get CA125)
    r'persistent.*(?:bloat|distension|abdominal)',
    r'abdominal.*bloat', r'persistent.*bloat',
    r'early.*satiety', r'feeling.*full.*quickly',
    r'loss.*appetite', r'anorex',
    r'pelvic.*pain', r'abdomin.*pain.*persist',
    r'urinary.*(?:frequenc|urgenc)',
    r'change.*bowel.*habit',
    r'\bibs\b.*(?:postmenopausal|new[-\s]?onset)',
    r'postmenopausal.*bleed', r'\bpmb\b',
    r'abnormal.*vagin.*bleed', r'intermenstrual.*bleed',

    # Investigations
    r'\bca\s*125\b', r'ca125',
    r'transvagin.*(?:ultrasound|us|scan)', r'\btvs\b',
    r'pelvic.*(?:ultrasound|us|scan)',
    r'ct.*(?:abdomen|pelvis)', r'mri.*pelvis',
    r'\brmi\b', r'risk.*malign.*index',
    r'\bhe4\b', r'human.*epididymis',
    r'ascites', r'ascitic.*tap',
    r'omental.*biops', r'peritoneal.*biops',
    r'histopath',

    # Treatments
    r'total.*hysterect.*(?:bso|salping)',
    r'\btah\b.*\bbso\b', r'\btah[-\s]?bso\b',
    r'salpingo[-\s]?oophorectom',
    r'debulking.*surgery', r'cytoreduct',
    r'carboplatin', r'paclitaxel', r'cisplatin.*ovar',
    r'olaparib', r'niraparib', r'rucaparib',
    r'parp.*inhibitor',
    r'bevacizumab.*ovar',

    # Family history / risk factors
    r'family.*history.*(?:ovarian|breast.*ovar|brca)',
    r'\bbrca1\b', r'\bbrca2\b', r'hereditary.*ovar',
    r'lynch.*syndrome', r'\bmlh1', r'\bmsh2',
    r'nullipar',
    r'late.*menopause', r'early.*menarche',
    r'hormone[-\s]?replac', r'\bhrt\b',
    r'endometriosis',
    r'oral.*contracep',
],
        'semantic_reference_phrases': [
            'ovarian cancer',
            'malignant neoplasm of ovary',
            'epithelial ovarian carcinoma',
            'serous ovarian carcinoma',
            'tubo-ovarian carcinoma',
            'fallopian tube cancer',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening
    r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
    r'faecal.*occult', r'\bfobt\b', r'\bfit\s*test',
    r'cervical.*smear', r'cervical.*screen',
    r'mammogr', r'breast.*screen',
    r'\bpsa\b', r'prostate.*screen',
    # Lung screening
    r'spirom', r'peak.*flow', r'chest.*x[-\s]?ray', r'\bcxr\b',
    # Generic admin
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },

    'leukaemia': {
        'MAX_CANCER_PATIENTS':        30000,
        'MAX_NO_CANCER_PATIENTS':     30000,
        'concept_codelist_paths':     [
            'data/codelists/leukaemia_relevance/leukaemia_snomed.csv',
            'data/codelists/leukaemia_relevance/myelodysplastic_syndromes.csv',
            'data/codelists/leukaemia_relevance/myeloproliferative_disorders.csv',
        ],
        'relevance_patterns':         [
    # Anatomy & disease subtypes (UK and US spellings)
    r'leuk(?:a)?em', r'leukaemia', r'leukemia',
    r'acute.*myeloid', r'\baml\b',
    r'acute.*lymphobl', r'\ball\s*leuk', r'acute.*lymphocytic',
    r'chronic.*myeloid', r'\bcml\b',
    r'chronic.*lymphocytic', r'\bcll\b',
    r'hairy.*cell.*leuk',
    r'myelodysplas', r'\bmds\b',
    r'myeloprolif', r'polycyth(?:a)?em.*vera', r'\bessential\s*thrombocyt',
    r'philadelphia.*chromosom', r'bcr[-\s]?abl',
    r'blast.*cell',

    # NICE NG12 symptoms
    r'persistent.*(?:fatigue|tiredness)',
    r'unexplained.*(?:fever|pyrexia|night.*sweat)',
    r'unexplained.*bruis', r'easy.*bruis',
    r'\bpetechi', r'purpura',
    r'recurrent.*(?:infect|chest|urinary)',
    r'unexplained.*bleed', r'epistax', r'gum.*bleed',
    r'bone\s*pain', r'(?:joint|musculoskelet).*pain.*child',
    r'lymphadenop', r'enlarged.*lymph.*node',
    r'splenomeg', r'hepatomeg', r'hepatosplen',
    r'pallor', r'\banaemi', r'\banemia',
    r'fatigue.*(?:persist|severe|unexplain)',

    # Investigations
    r'\bfbc\b', r'full\s*blood\s*count',
    r'blood\s*film', r'peripheral\s*(?:smear|blood\s*film)',
    r'\bwbc\b.*(?:high|elevat|raised)',
    r'\bwcc\b', r'white.*cell.*count',
    r'thrombocytop', r'platelet.*low',
    r'bone\s*marrow.*(?:biops|aspir|trephine)',
    r'\btrephine\b',
    r'cytogenet', r'\bfish\b.*(?:bcr|chromosom)',
    r'flow\s*cytometry', r'immunophenotyp',
    r'philadelphia.*chromosom', r'bcr[-\s]?abl',
    r'\bldh\b', r'urate.*raised', r'tumour\s*lysis',
    r'\bcd34\b', r'\bcd19\b', r'\bcd20\b',

    # Treatments
    r'cytarabine', r'ara[-\s]?c',
    r'daunorubicin', r'idarubicin',
    r'vincristine', r'methotrexate.*(?:leuk|all)',
    r'asparaginase', r'pegasparag',
    r'\bimatinib\b', r'glivec',
    r'dasatinib', r'nilotinib', r'ponatinib', r'bosutinib',
    r'\btki\b.*(?:leuk|cml)',
    r'rituximab', r'obinutuzumab',
    r'ibrutinib', r'acalabrutinib', r'venetoclax',
    r'fludarabine',
    r'\bcar[-\s]?t', r'tisagenlec',
    r'allogeneic.*(?:stem.*cell|bone.*marrow|transplant)',
    r'autologous.*(?:stem.*cell|transplant)',
    r'hydroxycarbam', r'hydroxyurea',
    r'azacitid', r'decitab', r'hypomethyl',

    # Family history / risk factors
    r'family.*history.*leuk',
    r'down.*syndrome', r'trisomy.*21',
    r'fanconi.*anaem',
    r'prior.*(?:chemo|radio|cytotox)',
    r'benzene.*(?:expos|environmental)',
],
        'semantic_reference_phrases': [
            'leukaemia',
            'leukemia',
            'acute myeloid leukaemia',
            'chronic myeloid leukaemia',
            'acute lymphoblastic leukaemia',
            'chronic lymphocytic leukaemia',
            'malignant neoplasm of haematopoietic tissue',
        ],
        'shortcut_text_patterns':     [
    # Other-cancer screening (leukaemia has no screening programme)
    r'bowel.*screen', r'flexible.*sigmoid', r'colonoscop',
    r'faecal.*occult', r'\bfobt\b', r'\bfit\s*test',
    r'cervical.*smear', r'cervical.*screen',
    r'mammogr', r'breast.*screen',
    r'\bpsa\b', r'prostate.*screen',
    # Lung screening
    r'spirom', r'peak.*flow', r'chest.*x[-\s]?ray', r'\bcxr\b',
    # Generic admin
    r'driving.*licen', r'\bfit\s*for\s*work\b',
    r'insurance.*medical', r'life assurance',
    r'\bbill.*fee\b', r'administrative procedure',
    r'did not attend', r'\bdna\b',
    # Other-system investigations
    r'retinal.*screen', r'diabetic.*eye',
    r'dexa.*scan', r'bone.*density',
],
    },
}


## Execution

### Config

In [ ]:
# config
config = get_config()

# Cancer-type selection
TARGET_CANCER = 'lung'     # <── change to 'breast' (or any profile key in CANCER_PROFILES) to switch

# Data version
config['data_version'] = 'v45'

# Code version
config['code_version'] = 'v147'

# Data source
#   'bigquery' → run config['SQL_FILE'] against BigQuery with a local parquet cache
#   'csv'      → read CSVs from config['DATA_FILE'] (v127 behaviour, glob wildcards OK)
config['DATA_SOURCE']        = 'bigquery'   

# BigQuery SQL path (used when DATA_SOURCE='bigquery')
config['SQL_FILE']           = f'sql/{TARGET_CANCER}_{config["data_version"]}.sql'
config['BQ_PROJECT']         = None     # auto-detect from VM metadata server (or set to a project string to force one)
config['BQ_LOCATION']        = 'US'     # EMIS data is in US

# CSV path (used when DATA_SOURCE='csv')
config['DATA_FILE']          = f'data/{TARGET_CANCER}_{config["data_version"]}-*.csv'

# Local parquet cache for the BigQuery result.
# Editing the SQL changes the hash -> cache auto-invalidates.
config['use_local_cache']    = True     # set False to disable caching entirely
config['cache_dir']          = 'cache'  # parquet files written here
config['force_bq_refresh']   = False    # set True to bypass cache and re-fetch (overwrites cache)

# Derived paths (independent of DATA_SOURCE).
config['saved_model_path'] = 'saved_models/lung_lstm_v147_v45.keras'
config['tuner_project_name'] = 'lung_lstm_v147_v45_tuning'

# Apply the cancer profile — codelists, regex patterns, reference phrases,
# shortcut patterns, MAX_CANCER_PATIENTS, MAX_NO_CANCER_PATIENTS.
config.update(CANCER_PROFILES[TARGET_CANCER])

# static params
config['train_model'] = True
config['save_model'] = True
config['load_model'] = True

# Branch toggles
config['use_snomed_branch'] = True
config['use_med_branch'] = False

# Separate sequence length caps per branch
config['MAX_SNOMED_SEQ_LENGTH'] = 30                  # v117: 30 (v116=50, v115=100)
config['MAX_MED_SEQ_LENGTH'] = 30

# Minimum sequence length thresholds
config['MIN_SNOMED_SEQ_LENGTH'] = 10              # v115: 30 (was 10)
config['MIN_MED_SEQ_LENGTH'] = 10

# additional static params
config['show_demographics'] = True
config['individual_verification'] = True
config['individual_verification_n_subset'] = 20

# cohorts plot by year 
config['cohort_plot_year_start'] = 2017
config['cohort_plot_year_end']   = 2025

# selected_patient_id_test_index
config['selected_patient_id_test_index'] = 0

# seeds
config['SEED'] = 42
config['SPLIT_SEED'] = 42
config['epochs'] = 25                               
config['early_stopping_patience'] = 8               
config['lr_reduction_patience'] = 4                 

# min snomed code frequency (below wich the code is just replaced with <RARE> to reduce embedding size) 
config['min_code_frequency'] = 30          

# sensitivity and specificity weights for training-time metric
config['sensitivity_weight'] = 3.0          
config['specificity_weight'] = 1.0

# training-time metric
config['optimizer_metric'] = WeightedSensSpec(
    name='weighted_sens_spec',
    sensitivity_weight=config['sensitivity_weight'],
    specificity_weight=config['specificity_weight'])
config['monitor_metric'] = 'val_weighted_sens_spec'

# Threshold selection 
config['proba_threshold'] = 0.5
config['use_optimal_threshold'] = True
config['use_sensitivity_floor'] = True
config['sensitivity_floor'] = 0.95
config['threshold_sensitivity_weight'] = 1.5  # used only if use_sensitivity_floor=False
config['threshold_specificity_weight'] = 1.0

# Model HPs
config['embedding_dim'] = 16                         
config['snomed_lstm_units'] = 48                     
config['med_embedding_dim'] = 8                      
config['med_lstm_units'] = 32                       
config['other_dense_units'] = 16                    
config['dense_1_units'] = 48                        
config['embedding_dropout'] = 0.4                    
config['dense_dropout'] = 0.5                        
config['l2_reg'] = 0.0035512477440179644             
config['learning_rate'] = 0.00012                    
config['cancer_class_weight'] = 1.5                  
config['batch_size'] = 128                           

# Anti-overfitting params
config['weight_decay'] = 0.001602482736794853        
config['label_smoothing'] = 0.15                     
config['use_focal_loss'] = True                      
config['mask_rate'] = 0.5                            
config['med_mask_rate'] = 0.1                        

# Explainability settings
config['run_explainability'] = True
config['explainability_patient_idx'] = 2
config['explainability_sample_n'] = 1047
config['explainability_ig_m_steps'] = 128
config['explainability_predict_batch_size'] = 1024
config['explainability_chunk_patients'] = 32

# baseline_type
#
# options: 'healthy' | 'shap_training' | 'shap_eg'
#
# 'healthy' — The reference baseline is the average healthy patient in the training data. 
#             This should show how a patient deviates from what a healthy record looks like, and each deviation's contribution.
#
# 'shap_training' — The reference baseline is the average of all patients in the training data (cancer + non-cancer, mixed together). 
#                   This should should show how a patient deviates from a generic patient, regardless of their eventual diagnosis.
#                   This is a more standard option for SHAP, but it's less clinically meaningful
#
# 'shap_eg' — The reference baseline isn't one point — it's 1,000 actual training patients, and the model averages the explanation across all of them. 
#             This should should show how a patient's prediction changes compared to each of 1,000 reference patients, averaged.
#             Analogy: a doctor saying "your blood pressure is high compared to everyone we've ever measured"
config['explainability_baseline_type'] = 'healthy'
config['shap_background_size'] = 1000    # v106: 1000 random training rows (near-complete distribution)
config['shap_eg_n_samples'] = 1000       # v106: 1000 MC samples per patient (beyond this is diminishing returns)
config['shap_eg_patient_chunk_size'] = 32

# Additional param for the config['explainability_baseline_type'] = 'shap_eg' baseline
# Possible values:
#   'per_position_mode' — single (1, T) mode baseline  (v105/v105s behaviour)
#   'marginal_sample'   — one random baseline patient per test patient (SHAP-interventional for sequences; actually differs from v105)
#
# Semantics
# 'per_position_mode' asks: "what if this timestep had the most-common code that patients usually have at that position?"  
# 'marginal_sample' asks: "what if this patient's timeline had been a different (randomly chosen) patient's?"  
#
# Tradeoff
# per_position_mode gives reproducible deltas but underestimates how much the ablation really changes the prediction — the model has seen the mode code 
# at that position during training and isn't surprised by it.
# marginal_sample is closer to proper SHAP for sequences but adds Monte Carlo noise (K=1 sample per patient). Faithfulness AOPC may drop 0.02-0.05 versus mode-baseline. 
config['shap_ablation_mode'] = 'marginal_sample'

# Explainability automated rating (faithfulness + clinical-relevance + local↔global consistency)
config['rate_explainability'] = True
config['explainability_rating_weights'] = {'faithfulness': 0.1, 'relevance': 0.8, 'consistency': 0.1}
config['explainability_aopc_n_patients'] = 1047
config['explainability_aopc_k_values'] = [1, 5, 10, 20]
config['explainability_relevance_top_k'] = 50
config['explainability_consistency_top_k'] = 10
config['explainability_consistency_top_global_n'] = 50

# explainability-charts
config['show_explainability_charts']     = False
config['show_relevance_filtered_charts'] = True
config['relevance_filtered_top_n']       = 50

# combined static + SNOMED chart cap (top-K of the union by magnitude). Only applies when show_explainability_charts is True.
config['combined_chart_top_k']           = 30

# drop event rows whose snomed_text matches obvious
# shortcut patterns (other-cancer screening, generic admin, insurance medicals)
# BEFORE the model sees them to reduces confounders that pollute the top-10.
config['exclude_shortcut_codes'] = False

# semantic-embedding match for concepts that auth/regex miss. Needs sentence-transformers (graceful fallback if missing).
config['use_semantic_relevance'] = False
config['semantic_relevance_model'] = 'sentence-transformers/all-MiniLM-L6-v2'
config['semantic_relevance_threshold'] = 0.5

# Hyperparameter tuning 
config['run_tuning'] = False
config['tuner_max_trials'] = 50                     
config['tuner_epochs'] = 15                         # match normal run training epochs

# rate the top-N tuning trials' explainability after tuning finishes
config['rate_top_tuning_trials'] = True
config['tuning_top_n_rate'] = 15        

### Tuning

In [ ]:
# Run hyperparameter tuning
if config.get('run_tuning', False):
    tuning_result = run_tuning(config)

### Run

In [ ]:
# Run
outputs = run(config)